In [1]:
# Imports functions into the program
import os
import csv
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

#Imported for categorization of the Sources
from scipy import stats
import astropy.stats as ast
from astropy.visualization import hist
import urllib.request
import lmfit
from importlib import reload

# Import from BB analysis github
import LC
from LC_Set import LC_Set
from fermi_catalog import select_bll, select_fsrq, select_bcu

from matplotlib.ticker import FormatStrFormatter
from scipy.optimize import curve_fit
from scipy.interpolate import SmoothBivariateSpline
import astropy.units as u

# COSI Analysis

In [2]:
# Using the new 2025 dataset. The new dataset has renamed ts2 to return_code2, so the code has been updated accordingly.
table = "new_db_Aug2025.csv"

opened = open(table,"r")
readed = pd.read_csv(table, sep=",", na_filter=True)
readed = readed.fillna(-3333)

cadence = 'weekly'
cadence_df = readed.loc[(readed['cadence'] == cadence) #& \
                        # (readed['ts2'] <= 4.) & \
                        # (readed['photon_flux_error2'] <  readed['photon_flux2'])
                        ]
display(pd.DataFrame(cadence_df))
#cadence_df.loc[cadence_df['ts2'] <= 9., 'photon_flux2'] = -3333
cadence_df.loc[cadence_df['ts2'] <= 4., ['photon_flux2', 'photon_flux_error2']] = -3333
#Jorge: Set as -3333 all points with an error larger than the flux
cadence_df.loc[cadence_df['photon_flux_error2'] > cadence_df['photon_flux2'], ['photon_flux2', 'photon_flux_error2']] = -3333
#Jorge: Set cuts to remove possible outliers due to bad convergence of a bin
cadence_df.loc[cadence_df['photon_flux2'] > 1e-4, ['photon_flux2', 'photon_flux_error2']] = -3333
cadence_df.loc[cadence_df['photon_flux2'] < 1e-10, ['photon_flux2', 'photon_flux_error2']] = -3333
cadence_df.loc[(cadence_df['ts2']<=25) & (cadence_df['photon_flux2'] > 1e-6)] = -3333
#print(cadence_df)
cadence_df = cadence_df.reset_index(drop=True)
TSTART = np.amin(cadence_df['tmin'])
TSTOP = np.amax(cadence_df['tmax'])

names = np.unique(readed['source_name']) #.drop_duplicates(subset=['source_name'], inplace=False)#[1]

df_4lacdr3 = pd.read_csv('4lac_redshifts.csv', delimiter='\t', comment='#')

,Unnamed: 0,source_name,cadence,tmin,tmax,photon_flux2,photon_flux_error2,photon_index2,return_code2,ts2
3050,3050,4FGL J0001.2-0747,weekly,239587201,240192001,-3.333000e+03,-3.333000e+03,-3333.00,0.0,0.51
3051,3051,4FGL J0001.5+2113,weekly,239587201,240192001,1.480000e-08,1.550000e-08,2.12,0.0,1.47
3052,3052,4FGL J0003.3-1928,weekly,239587201,240192001,-3.333000e+03,-3.333000e+03,-3333.00,0.0,0.25
3053,3053,4FGL J0004.3+4614,weekly,239587201,240192001,-3.333000e+03,-3.333000e+03,-3333.00,0.0,0.31
3054,3054,4FGL J0004.4-4737,weekly,239587201,240192001,5.030000e-08,3.010000e-08,3.15,0.0,5.02
...,...,...,...,...,...,...,...,...,...,...
4817470,4817470,4FGL J2358.0-4601,weekly,774835201,775440001,-3.333000e+03,-3.333000e+03,-3333.00,-3333.0,-3333.00
4817471,4817471,4FGL J2358.3-1021,weekly,774835201,775440001,-3.333000e+03,-3.333000e+03,-3333.00,-3333.0,-3333.00
4817472,4817472,4FGL J2358.3+3830,weekly,774835201,775440001,-3.333000e+03,-3.333000e+03,-3333.00,-3333.0,-3333.00
4817473,4817473,4FGL J2359.0+3922,weekly,774835201,775440001,-3.333000e+03,-3.333000e+03,-3333.00,-3333.0,-3333.00


In [ ]:
table = 'COSICSV/COSI_factors_all_updated_only_logpar_flare_softer_+0.5_updated_60deg_offaxis_last.csv'
#table = 'COSICSV/COSI_factors_all_updated_only_logpar_flare_updated_60deg_offaxis_last.csv'
COSI_LAT_Sources = pd.read_csv(table, sep=",", na_filter=True)

MJDREFI = 51910
MJDREFF = 7.428703703703703e-4
SecsInDay = 86400
import copy

def MET_to_MJD(MET,MJDREF=51910):
    return MET/86400 + MJDREF


def quiescent_background_finder(sourcelightcurve, method='forward'):
    # Determines the "quiescent background" of a given lightcurve
    qui = copy.deepcopy(sourcelightcurve)
    
    if not hasattr(sourcelightcurve, "hops"):
        sourcelightcurve.find_hop(method='baseline', lc_edges ='add', baseline = thresholdflux)


    if sourcelightcurve.hops is None:
        quiescent_background,qui_err = np.nan, np.nan
        print('No flares detected initially, quiescent background is all bins')
        return quiescent_background, qui_err

    # Mask out flaring regions
    mask = []
    for hop in sourcelightcurve.hops:
        start_idx = np.searchsorted(sourcelightcurve.time, hop.start_time)
        end_idx = np.searchsorted(sourcelightcurve.time, hop.end_time)
        if start_idx < end_idx:
            mask.extend(range(start_idx, end_idx))

    maskindices = np.array(mask)
    
    # Avoid deleting empty list.
    if maskindices.size > 0:
        qui.flux = np.delete(qui.flux, maskindices)
        qui.time = np.delete(qui.time, maskindices)
        qui.flux_error = np.delete(qui.flux_error, maskindices)

    if qui.flux.size == 0:
        print("All time bins were flaring. No quiescent background can be determined.")
        return np.nan, np.nan

    # Compute weighted average of non-flaring sections
    baseaverage = []
    weights = []
    tempavg = []

    if cadence_df['cadence'][0] == 'daily':
        tdiff = 3
    if cadence_df['cadence'][0] == 'weekly':
        tdiff = 7
    if cadence_df['cadence'][0] == 'monthly':
        tdiff = 30


    for i in range(len(qui.flux) - 1):
        tempavg.append(qui.flux[i])

        # Check for time gap or last element
        if i == len(qui.flux) - 2 or (qui.time[i + 1] - qui.time[i] != tdiff):
            if tempavg:
                baseaverage.append(np.nanmean(tempavg))
                weights.append(len(tempavg))
            tempavg = []

    # Computing Weighted Mean, putting failsafes to avoid empty averages.
    if baseaverage and weights:
        quiescent_background = np.average(baseaverage, weights=weights)
    else:
        quiescent_background = np.nan

    qui_err = np.sqrt(np.sum((np.array(qui.flux_error) ** 2) * (1 / len(qui.flux_error)) ** 2)) if len(qui.flux_error) > 0 else 0

    print(f"Quiescent Background: {quiescent_background}, Error: {qui_err}")
    return quiescent_background, qui_err



def LCTimeRange(sourcearray, timerangestart,timerangeend):
    sourcearray = sourcearray[timerangestart:timerangeend]
    return sourcearray

def quiescent_flare_plot(cadence_df,sourcename=None,sourcenum=0,percent = 0.1, MJDREFI=51910, MJDREFF=7.428703703703703e-4,bkg_err = False, factor = 1):
    # Depending on what we are doing analysis on/how our time is binned, we can change the MJDREFI to other values.
    


    if sourcename == None:
        sourcearray = cadence_df[cadence_df['source_name'] == cadence_df['source_name'][sourcenum]]
        titlestring=cadence_df['source_name'][sourcenum]
    else:
        sourcearray = cadence_df[cadence_df['source_name'] == sourcename].reset_index(drop=True)
        titlestring=sourcename


    sourcearray = sourcearray[sourcearray['photon_flux2']!=-3333].reset_index(drop=True)
    average_flux = np.mean(sourcearray['photon_flux2'])
    sourcearray = sourcearray[sourcearray['photon_flux2']<=100*average_flux].reset_index(drop=True)
    time = sourcearray['tmin']/SecsInDay + MJDREFI
    photon_flux = sourcearray['photon_flux2'] * factor
    errors = sourcearray['photon_flux_error2']

    sourcelightcurve = LC.LightCurve(time,photon_flux,errors,time_format='mjd',name=titlestring)
    maxflux = np.max(photon_flux)
    minflux =np.min(photon_flux)
    delta_flux = maxflux - minflux
    delta_flux_percent = delta_flux * percent
    thresholdflux = minflux + delta_flux_percent


    # Finding first set of flares using threshold flux.
    sourcelightcurve.get_bblocks(gamma_value=0.05)
    sourcelightcurve.find_hop(method = 'baseline', lc_edges ='add', baseline = thresholdflux)

    #if sourcelightcurve.hops == None:
    #    print("No flares detected for "+str(titlestring))
    #    return None


    # Finding quiescent background.
    quiescent_background, qui_err = quiescent_background_finder(sourcelightcurve,'forward')

        

    # Using quiescent background to find flares again.
    sourcelightcurve = LC.LightCurve(time,photon_flux,errors,time_format='mjd')


    #sourcelightcurve.flux = np.subtract(sourcelightcurve.flux,quiescent_background)
    sourcelightcurve.get_bblocks(gamma_value=0.05)
    #sourcelightcurve.get_bblocks_above(threshold = 0)
    sourcelightcurve.find_hop(method = 'baseline', lc_edges ='add',baseline = quiescent_background)


    
    # Plotting the Lightcurve itself.
    plt.figure(figsize=(16,9))
    plt.xlabel("MJD")
    plt.ylabel('Photon Flux (Photons/$cm^2\u22c5s^{-1}$) (0.1 - 100 GeV)')
    plt.title("Photon Flux vs Time" ' (Source: ' +str(titlestring)+ ')' )
    sourcelightcurve.plot_bblocks(size=2)
    sourcelightcurve.plot_hline(value = quiescent_background, color='green',label='Q BKG',lw=3,linestyle = 'dashed')

    if bkg_err == True:
        y1 = quiescent_background + qui_err
        y2 = quiescent_background - qui_err
        plt.fill_between(sourcelightcurve.time,y1,y2,alpha = 0.3,color='green',label = 'Q BKG Error')

    sourcelightcurve.plot_hop()
    plotting_anomalies()
    plt.legend()

def plotting_anomalies():
    ROCKING_pre50_START   = MET_to_MJD(239557417.000)
    ROCKING_50_START      = MET_to_MJD(273628802.000)

    GC_START              = MET_to_MJD(407898663.000)  
    GC_STOP               = MET_to_MJD(458755204.000)   

    ANOMALY_MET           = MET_to_MJD(542851205.000) 
    START_NEW_PROFILE     = MET_to_MJD(571795205.000) 

    #plt.axvspan((ROCKING_pre50_START), (ROCKING_50_START),
        #color='y', alpha=0.6, label='OLD ROCKING')
    plt.rcParams['hatch.color'] = 'black'
    plt.fill_between([ROCKING_pre50_START,ROCKING_50_START],y1=-(np.max(sourcelightcurve.flux)*0.02),y2=0,color='y',alpha=0.5,label='OLD Rocking',edgecolor='black')
    plt.fill_between([GC_START,GC_STOP],y1=-(np.max(sourcelightcurve.flux)*0.02),y2=0,color='y',alpha=0.5,label='GC POINTING', hatch = '/',edgecolor='black')
    plt.fill_between([ANOMALY_MET,START_NEW_PROFILE],y1=-(np.max(sourcelightcurve.flux)*0.02),y2=0,color='y',alpha=0.5,label='SOLAR PANEL ANOMALY', hatch = '///',edgecolor='black')

    #plt.axvspan((GC_START),
    #    (GC_STOP),
    #    color='y', alpha=0.40, label='GC POINTING')
    #plt.axvspan((ANOMALY_MET),(START_NEW_PROFILE),
    #    color='y', alpha=0.2, label='SOLAR PANEL ANOMALY')

def fluence_integrator(hops_bl,sourcelightcurve, time = 's',):
    flarestack=[0,0,0,0,0,0,0,0,0,0]
    for i in range(0,len(hops_bl)):
        temp = sourcelightcurve.flux[np.min(hops_bl[i].iis):np.max(hops_bl[i].iis)] # array of flux ph/cm2/s
        if cadence_df['cadence'][0] == 'daily':
            temp_sum = temp * 3
        if cadence_df['cadence'][0] == 'weekly':
            temp_sum = temp * 7
        if cadence_df['cadence'][0] == 'monthly':
            temp_sum = temp * 30

        if time == 's':
            if cadence_df['cadence'][0] == 'daily':
                temp_sum = temp_sum * 86400 # ph/cm2/s * days * s/day = ph/cm2
            if cadence_df['cadence'][0] == 'weekly':
                temp_sum = temp_sum * 86400
            if cadence_df['cadence'][0] == 'monthly':
                temp_sum = temp_sum * 86400
        duration = (hops_bl[i].end_time-hops_bl[i].start_time)*86400
        starttime_MJD= hops_bl[i].start_time
        starttime_s = starttime_MJD * 86400
        observation_time = (np.max(sourcelightcurve.time)-np.min(sourcelightcurve.time))*86400
        portion_of_obs = duration/observation_time
        integral = np.sum(temp_sum) # sum of flux over duration ph/cm2 for the flare
        background_counts = COSI_bkg_rate * duration
        fluxsum = np.sum(temp) # sum of flux ph/cm2/s over duration
        flare_asym = hops_bl[i].asym
        if len(fsrq_names[fsrq_names==sourcelightcurve.name])>0:
            blazartype='FSRQ'
        elif len(bll_names[bll_names==sourcelightcurve.name])>0:
            blazartype='BLL'
        elif len(bcu_names[bcu_names==sourcelightcurve.name])>0:
            blazartype='BCU'
        else:
            blazartype='None'
        
        if len(flarestack)<1:
            flarestack = [sourcelightcurve.name,fluxsum,integral,duration,background_counts,starttime_s,starttime_MJD,portion_of_obs,blazartype,flare_asym]
        else:
            flarestack = np.vstack((flarestack,[sourcelightcurve.name,fluxsum,integral,duration,background_counts,starttime_s,starttime_MJD,portion_of_obs,blazartype,flare_asym]))
    return flarestack

def hop_characterizer(hops, COSI_bkg_rate, COSI_Aeff, LAT_Aeff, ph_ratio, int_flux_ratio):
    # Given a hop_bl object (object that contains data about each flare), returns an array of characteristics for that each flare.

    # Empty arrays that will store the characteristics of each flare.
    hop_durations = []
    hop_sourcecounts = []
    hop_asymmetries = []
    hop_coverages = []
    hop_background = []
    hop_sourcecounts = []
    hop_MDP99 = []
    hop_name = []
    blazartype = []
    hop_fluences = []
    hop_avg_fluxes = []
    hop_start_times = []
    entire_avg_flux = []
    hop_peak_fluxes = []

    # Loop through each hop and extract its characteristics.
    for hop in hops:
        
        
        # Since duration is in days (MJD), convert to seconds.
        duration = hop.dur * 86400

        # add all flux values, then multiply by Effective Area of Fermi and the conversion factor. Divide by Area of COSI to get counts in COSI band.
        
        # Asymmetry of the flare. This is tacked on just in case we want to use it later.
        asymmetry = hop.asym

        # Portion of time the flare is active.
        coverage = ((hop.dur) / (sourcelightcurve.time[-1] - sourcelightcurve.time[0]))

        # Background counts during the flare duration.
        background_counts = duration * COSI_bkg_rate 

        COSI_ph_rate = np.mean(hop.flux)  * LAT_Aeff * ph_ratio

        # Total fluence during the flare.
        fluence_LAT = np.mean(hop.flux) * duration

        fluence_COSI = np.mean(hop.flux) * int_flux_ratio * duration

        # Average flux of the flare.
        avg_flux = np.mean(hop.flux) * int_flux_ratio
        lc_avg_flux = np.mean(sourcelightcurve.flux) * int_flux_ratio
        peak_flux = np.max(hop.flux) * int_flux_ratio

        # Average flux values, then multiply by Effective Area of Fermi and the conversion factor.
        # ARM CUTS: 
        # 20 ct/s: 1 factor
        # 10 ct/s: 1.27 factor
        # 1 ct/s: 3.22 factor

        if COSI_bkg_rate == 20*0.25:
            ARM_reduction = 1
        elif COSI_bkg_rate == 10*0.25:
            ARM_reduction = 1.27
        elif COSI_bkg_rate == 1*0.25:
            ARM_reduction = 3.22
        else:
            ARM_reduction = 1

        sourcecounts = COSI_ph_rate * duration / ARM_reduction

        start_time = hop.start_time

        # Append characteristics to their respective arrays.
        hop_name.append(hop.name)
        hop_durations.append(duration)
        hop_sourcecounts.append(sourcecounts)
        hop_asymmetries.append(asymmetry)
        hop_coverages.append(coverage)
        hop_background.append(background_counts)
        hop_MDP99.append(ComputeMDP99(sourcecounts, background_counts))
        hop_fluences.append(fluence_COSI)
        hop_avg_fluxes.append(avg_flux)
        hop_peak_fluxes.append(peak_flux)
        hop_start_times.append(start_time)
        entire_avg_flux.append(lc_avg_flux)

        if len(fsrq_names[fsrq_names==sourcelightcurve.name])>0:
            blazartype.append('FSRQ')
        elif len(bll_names[bll_names==sourcelightcurve.name])>0:
            blazartype.append('BLL')
        elif len(bcu_names[bcu_names==sourcelightcurve.name])>0:
            blazartype.append('BCU')
        else:
            blazartype.append('None')
            
    # Return all characteristics
    return hop_name, hop_durations, hop_sourcecounts, hop_asymmetries, hop_coverages, hop_background, hop_MDP99, blazartype, hop_fluences, hop_avg_fluxes, hop_start_times, entire_avg_flux, hop_peak_fluxes

def ComputeMDP99(src_counts, bkg_counts, average_mu=0.3):
    
    mdp99 = (4.29 / (average_mu*(src_counts))) * np.sqrt(src_counts+bkg_counts) * 100
    
    return mdp99


def flare_isolator(sourcename, initialtime,finaltime, threshold=None, COSI_LAT_Sources=COSI_LAT_Sources,thresholdpercent = 0.3):
    sourcearray = cadence_df[cadence_df['source_name'] == sourcename].reset_index(drop=True)
    titlestring = sourcename
    sourcearray = sourcearray[sourcearray['photon_flux2']!=-3333].reset_index(drop=True)
    average_flux = np.nanmean(sourcearray['photon_flux2'])
    sourcearray = sourcearray[sourcearray['photon_flux2']<=100*average_flux].reset_index(drop=True)
    time = sourcearray['tmin']/SecsInDay + MJDREFI
    photon_flux = sourcearray['photon_flux2']
    errors = sourcearray['photon_flux_error2']
    # Creates the lightcurve object of the source.
    sourcelightcurve = LC.LightCurve(time,photon_flux,errors,name=titlestring)
    i = COSI_LAT_Sources[COSI_LAT_Sources['Name']==sourcename].index[0]

    selected1 = sourcelightcurve.select_by_time(initialtime,finaltime)
    originalLC = sourcelightcurve
    sourcelightcurve.get_bblocks()
    photon_flux = selected1.flux

    # Finding the initial flux threshold for detection.
    maxflux = np.max(photon_flux)
    minflux = np.min(photon_flux)
    delta_flux = maxflux - minflux
    delta_flux_percent = delta_flux * thresholdpercent
    thresholdflux = minflux + delta_flux_percent


    # Initial flare detection.
    selected1.get_bblocks(gamma_value=0.05)
    selected1.find_hop(method='baseline', lc_edges ='add', baseline = thresholdflux)

    quiescent_background,qui_err = quiescent_background_finder(sourcelightcurve=selected1,method='forward')


    # Re-detecting flares with quiescent background.
    sourcelightcurve = originalLC.select_by_time(initialtime,finaltime)
    sourcelightcurve.get_bblocks(gamma_value=0.05)
    if threshold == None:
        sourcelightcurve.find_hop(method = 'baseline', lc_edges ='add',baseline = quiescent_background)
    else: 
        sourcelightcurve.find_hop(method = 'baseline', lc_edges ='add',baseline = threshold)

    hops_bl1 = sourcelightcurve.hops
    plotit = originalLC.select_by_time(initialtime,finaltime)
    plotit.plot_lc()
    sourcelightcurve.plot_hop()

    COSI_LAT_Sources = pd.read_csv(table, sep=",", na_filter=True)
    # Effective Area of Fermi LAT for the source.
    LAT_Aeff = COSI_LAT_Sources['Aeff_mean_LAT(cm2)'][i]

    # Effective Area of COSI for the source.
    COSI_Aeff = COSI_LAT_Sources['Aeff_mean_COSI(cm2)'][i]

    # Name of Source
    # Conversion factor of counts between Fermi and COSI Bands. (0.1-100 GeV to 0.2-5 MeV)
    ph_ratio = COSI_LAT_Sources['ph/s_ratio'][i]

    #ph_ratio = 1

    # Conversion factor of flux between Fermi and COSI Bands. (0.1-100 GeV to 0.2-5 MeV)
    flux_ratio = COSI_LAT_Sources['Int_flux_ratio'][i]


    fsrq_names=select_fsrq()
    blarray = [hops_bl1]
    COSI_BAND_ALL = np.array([0,0,0,0,0,0,0,0,0,0,0,0,0])
    for bl in blarray:
        characterized_flares = hop_characterizer(bl, COSI_bkg_rate=COSI_bkg_rate, COSI_Aeff=COSI_Aeff, LAT_Aeff=LAT_Aeff, ph_ratio=ph_ratio, int_flux_ratio=flux_ratio)
        COSI_BAND = np.array(characterized_flares).T
        COSI_BAND_ALL = np.vstack((COSI_BAND_ALL, COSI_BAND))

    COSI_BAND_BAT_weekly_df = pd.DataFrame(COSI_BAND_ALL)
    COSI_BAND_BAT_weekly_df.columns = ['Name','Duration_(s)', 'Source_Counts(ph)_(0.2-5_MeV)', 'Asymmetry', 'Coverage', 'Background_Counts', 'MDP99_(%)', 'Class', 'Photon_Fluence_(ph/cm2)_(0.2-5_MeV)', 'Average_Photon_Flux_(ph/cm2/s)_(0.2-5_MeV)','Start_Time_(MJD)','Average_Flux_of_Entire_Source_(0.2-5_MeV)', 'Peak_Flare_Flux_(0.2-5_MeV)']
    COSI_BAND_BAT_weekly_df = COSI_BAND_BAT_weekly_df[COSI_BAND_BAT_weekly_df[:]['Source_Counts(ph)_(0.2-5_MeV)']!=0].reset_index(drop=True)
    COSI_BAND_BAT_weekly_df = COSI_BAND_BAT_weekly_df[COSI_BAND_BAT_weekly_df[:]['Source_Counts(ph)_(0.2-5_MeV)']!='0.0'].reset_index(drop=True)
    COSI_BAND_BAT_weekly_df['Source_Counts(ph)_(0.2-5_MeV)'] = COSI_BAND_BAT_weekly_df['Source_Counts(ph)_(0.2-5_MeV)'].astype(float)
    COSI_BAND_BAT_weekly_df['Duration_(s)'] = COSI_BAND_BAT_weekly_df['Duration_(s)'].astype(float)
    COSI_BAND_BAT_weekly_df['MDP99_(%)'] = ComputeMDP99(COSI_BAND_BAT_weekly_df['Source_Counts(ph)_(0.2-5_MeV)'].astype(float),
                                                    COSI_BAND_BAT_weekly_df['Background_Counts'].astype(float))
    COSI_BAND_BAT_weekly_df['Coverage'] = COSI_BAND_BAT_weekly_df['Coverage'].astype(float)
    COSI_BAND_BAT_weekly_df = COSI_BAND_BAT_weekly_df[COSI_BAND_BAT_weekly_df[:]['Duration_(s)']<=2e8].reset_index(drop=True)


    neworder = ['Name','Class','Average_Photon_Flux_(ph/cm2/s)_(0.2-5_MeV)','Photon_Fluence_(ph/cm2)_(0.2-5_MeV)','Source_Counts(ph)_(0.2-5_MeV)','Background_Counts','Duration_(s)','Start_Time_(MJD)','Coverage', 'MDP99_(%)','Asymmetry','Average_Flux_of_Entire_Source_(0.2-5_MeV)', 'Peak_Flare_Flux_(0.2-5_MeV)']
    return COSI_BAND_BAT_weekly_df,sourcelightcurve
#COSI_BAND_BAT_weekly_df.to_csv(sourcename+'Analysis.csv')
# Remove blank space in sourcename

ignorelist = ['4FGL J0010.6+2043','4FGL J0222.0-1616','4FGL J0228.0-3026','4FGL J0312.8+0134',
'4FGL J0358.9+6004','4FGL J0824.7+5552','4FGL J1031.6+6019','4FGL J1209.8+1810','4FGL J1446.7+1719','4FGL J1635.6+3628',
'4FGL J1647.5+4950','4FGL J1716.1+6836','4FGL J1724.9+7654','4FGL J1808.1-5013','4FGL J2256.0-2740','4FGL J1222.5+0414',
'4FGL J0405.6-1308','4FGL J0336.4+3224','4FGL J1337.6-1257','4FGL J1924.8-2914','4FGL J0024.7+0349','4FGL J0239.7+0415',
'4FGL J0904.6+5200','4FGL J1118.2-0415','4FGL J1205.7-2635','4FGL J1324.9+4748','4FGL J1445.9-1626','4FGL J1559.9+2319',
'4FGL J1650.3-5045','4FGL J0116.0-1136','4FGL J0200.6-6637','4FGL J0224.9+1843','4FGL J0304.5+3349','4FGL J0347.7-3616',
'4FGL J0427.3-3900','4FGL J0450.3-4419','4FGL J0617.6-4028','4FGL J0746.4+2546','4FGL J0805.4+6147','4FGL J0943.7+6137',
'4FGL J1043.2+2408','4FGL J1200.7+2008','4FGL J1659.7-3131','4FGL J1747.6-5308','4FGL J1824.5+0107','4FGL J1912.1-0803',
'4FGL J2040.5-1705','4FGL J2120.6-1254','4FGL J2148.6+0652','4FGL J2149.7+1917','4FGL J2207.6+0053','4FGL J2358.0-4601',
'4FGL J0034.0-4116','4FGL J0035.8+6131','4FGL J0137.9+5814','4FGL J0152.2+3714','4FGL J0156.5+3914','4FGL J0342.2+3858', 
'4FGL J0353.0+5654','4FGL J0401.7+2112','4FGL J0407.5+0741','4FGL J0453.3+2843','4FGL J0455.7-4617','4FGL J0512.8+4041',
'4FGL J0635.6-7518','4FGL J0638.7+5658','4FGL J0647.7-6058','4FGL J0713.0+5738','4FGL J0723.5+2900','4FGL J0747.3-3310',
'4FGL J0806.5+4503','4FGL J0836.5-2026','4FGL J0911.0-5047','4FGL J0923.5+3852','4FGL J1001.1+2911','4FGL J1015.6+5553',
'4FGL J1016.0+0512','4FGL J1018.1+1905','4FGL J1045.8-2928','4FGL J1136.2+3407','4FGL J1158.5+4824','4FGL J1225.6-7313',
'4FGL J1238.5-1201','4FGL J1322.2+0842','4FGL J1440.0-1530','4FGL J1454.4-3744','4FGL J1513.4-3231','4FGL J1516.9+1934',
'4FGL J1656.0+2047','4FGL J1753.6-5014','4FGL J1808.2+3500','4FGL J1834.7-5858','4FGL J1912.0+1612','4FGL J1917.7-6930', 
'4FGL J1942.1+4011','4FGL J2130.4-4241','4FGL J2143.1-3929','4FGL J2146.4-1528','4FGL J2219.2-0342','4FGL J2359.2-3134', 
'4FGL J0059.4-5654','4FGL J0132.1-0956','4FGL J0156.3-2420','4FGL J0205.7+6449','4FGL J0240.5+6113','4FGL J0259.0+0552',
'4FGL J0308.4+0407','4FGL J0333.0-3044','4FGL J0405.4-6929','4FGL J0448.7-2116','4FGL J0521.2+1637','4FGL J0534.5+2200',
'4FGL J0537.5+0959','4FGL J0540.0-7552','4FGL J0622.9+3326','4FGL J0833.3-4342c','4FGL J0850.0+5108','4FGL J0931.9+6737',
'4FGL J0948.9+0022','4FGL J1023.7+0038','4FGL J1045.1-5940','4FGL J1047.2+6740','4FGL J1106.7+3623','4FGL J1228.0-4853',
'4FGL J1230.8+1223','4FGL J1231.1-1412','4FGL J1401.7-3217','4FGL J1418.4+3543','4FGL J1505.0+0326','4FGL J1514.8+4448',
'4FGL J1535.9+3743','4FGL J1543.6+0452','4FGL J1626.5-4406','4FGL J1644.9+2620','4FGL J1732.7-5050','4FGL J1753.9+2443',
'4FGL J1839.4-0553','4FGL J1935.2+2029','4FGL J2007.9-4432','4FGL J2021.5+4026','4FGL J2234.7+0943','4FGL J2237.6-5126',
'4FGL J0522.9-3628','4FGL J0433.0+0522','4FGL J0319.8+4130','4FGL J0324.8+3412','4FGL J1829.5+4845','4FGL J2055.0-5218',
'4FGL J1459.0+7140','4FGL J1632.8-1048','4FGL J1148.5+2629','4FGL J0319.8+4130']

## Softer Editor Analysis

In [ ]:
COSI_bkg_rate = 1*0.25

In [ ]:
#['4FGL J2253.9+1609', 54500, 57000, '8e-6'],['4FGL J2253.9+1609', 56500, 57000, '3e-6']
initialtime, finaltime = 54500, 57000
sourcename = '4FGL J2253.9+1609'
sourcearray = cadence_df[cadence_df['source_name'] == sourcename].reset_index(drop=True)
titlestring = sourcename
sourcearray = sourcearray[sourcearray['photon_flux2']!=-3333].reset_index(drop=True)
# omit the highest flux value of the sourcearray, without changing the order of the other values, to avoid outliers dominating the lightcurve.
sourcearray = sourcearray[sourcearray['photon_flux2']!=sourcearray['photon_flux2'].max()].reset_index(drop=True)
average_flux = np.nanmean(sourcearray['photon_flux2'])
sourcearray = sourcearray[sourcearray['photon_flux2']<=100*average_flux].reset_index(drop=True)
time = sourcearray['tmin']/SecsInDay + MJDREFI
photon_flux = sourcearray['photon_flux2']
errors = sourcearray['photon_flux_error2']
# Creates the lightcurve object of the source.
sourcelightcurve = LC.LightCurve(time,photon_flux,errors,name=titlestring)
i = COSI_LAT_Sources[COSI_LAT_Sources['Name']==sourcename].index[0]

selected1 = sourcelightcurve
originalLC = sourcelightcurve
photon_flux = selected1.flux

# Finding the initial flux threshold for detection.
maxflux = np.max(photon_flux)
minflux = np.min(photon_flux)
delta_flux = maxflux - minflux
delta_flux_percent = delta_flux * 0.3
thresholdflux = minflux + delta_flux_percent


# Initial flare detection.
selected1.get_bblocks(gamma_value=0.05)
selected1.find_hop(method='baseline', lc_edges ='add', baseline = thresholdflux)

quiescent_background,qui_err = quiescent_background_finder(sourcelightcurve=selected1,method='forward')


# Re-detecting flares with quiescent background.
#sourcelightcurve = originalLC.select_by_time(initialtime,finaltime)
sourcelightcurve.get_bblocks(gamma_value=0.05)
sourcelightcurve.find_hop(method = 'baseline', lc_edges ='add',baseline = quiescent_background)

hops_bl1 = sourcelightcurve.hops

COSI_LAT_Sources = pd.read_csv(table, sep=",", na_filter=True)
# Effective Area of Fermi LAT for the source.
LAT_Aeff = COSI_LAT_Sources['Aeff_mean_LAT(cm2)'][i]

# Effective Area of COSI for the source.
COSI_Aeff = COSI_LAT_Sources['Aeff_mean_COSI(cm2)'][i]

# Name of Source
# Conversion factor of counts between Fermi and COSI Bands. (0.1-100 GeV to 0.2-5 MeV)
ph_ratio = COSI_LAT_Sources['ph/s_ratio'][i]
print(ph_ratio)
#ph_ratio = 1

# Conversion factor of flux between Fermi and COSI Bands. (0.1-100 GeV to 0.2-5 MeV)
flux_ratio = COSI_LAT_Sources['Int_flux_ratio'][i]
print(flux_ratio)

fsrq_names=select_fsrq()
blarray = [hops_bl1]
COSI_BAND_ALL = np.array([0,0,0,0,0,0,0,0,0,0,0,0,0])
for bl in blarray:
    characterized_flares = hop_characterizer(bl, COSI_bkg_rate=COSI_bkg_rate, COSI_Aeff=COSI_Aeff, LAT_Aeff=LAT_Aeff, ph_ratio=ph_ratio, int_flux_ratio=flux_ratio)
    COSI_BAND = np.array(characterized_flares).T
    COSI_BAND_ALL = np.vstack((COSI_BAND_ALL, COSI_BAND))

COSI_BAND_BAT_weekly_df = pd.DataFrame(COSI_BAND_ALL)
COSI_BAND_BAT_weekly_df.columns = ['Name','Duration_(s)', 'Source_Counts(ph)_(0.2-5_MeV)', 'Asymmetry', 'Coverage', 'Background_Counts', 'MDP99_(%)', 'Class', 'Photon_Fluence_(ph/cm2)_(0.2-5_MeV)', 'Average_Photon_Flux_(ph/cm2/s)_(0.2-5_MeV)','Start_Time_(MJD)','Average_Flux_of_Entire_Source_(0.2-5_MeV)', 'Peak_Flare_Flux_(0.2-5_MeV)']
COSI_BAND_BAT_weekly_df = COSI_BAND_BAT_weekly_df[COSI_BAND_BAT_weekly_df[:]['Source_Counts(ph)_(0.2-5_MeV)']!=0].reset_index(drop=True)
COSI_BAND_BAT_weekly_df = COSI_BAND_BAT_weekly_df[COSI_BAND_BAT_weekly_df[:]['Source_Counts(ph)_(0.2-5_MeV)']!='0.0'].reset_index(drop=True)
COSI_BAND_BAT_weekly_df['Source_Counts(ph)_(0.2-5_MeV)'] = COSI_BAND_BAT_weekly_df['Source_Counts(ph)_(0.2-5_MeV)'].astype(float)
COSI_BAND_BAT_weekly_df['Duration_(s)'] = COSI_BAND_BAT_weekly_df['Duration_(s)'].astype(float)
COSI_BAND_BAT_weekly_df['MDP99_(%)'] = ComputeMDP99(COSI_BAND_BAT_weekly_df['Source_Counts(ph)_(0.2-5_MeV)'].astype(float),
                                                COSI_BAND_BAT_weekly_df['Background_Counts'].astype(float))
COSI_BAND_BAT_weekly_df['Coverage'] = COSI_BAND_BAT_weekly_df['Coverage'].astype(float)
COSI_BAND_BAT_weekly_df = COSI_BAND_BAT_weekly_df[COSI_BAND_BAT_weekly_df[:]['Duration_(s)']<=2e8].reset_index(drop=True)


neworder = ['Name','Class','Average_Photon_Flux_(ph/cm2/s)_(0.2-5_MeV)','Photon_Fluence_(ph/cm2)_(0.2-5_MeV)','Source_Counts(ph)_(0.2-5_MeV)','Background_Counts','Duration_(s)','Start_Time_(MJD)','Coverage', 'MDP99_(%)','Asymmetry','Average_Flux_of_Entire_Source_(0.2-5_MeV)', 'Peak_Flare_Flux_(0.2-5_MeV)']


plotit = originalLC.select_by_time(initialtime,finaltime)
plt.figure(figsize=(16,9))
plotit.plot_lc()
sourcelightcurve.plot_bblocks(size=2)
sourcelightcurve.plot_hop()
plt.xlim(initialtime+500,finaltime-1300)
# above each hop block, label the block with the MDP99 value of that flare.
#MDP99_value=0
#sourcelightcurve.hops=sourcelightcurve.hops[:-1]
#hopsarray=[hop1,hop2,hop3]
#characterizedflares = [characterized_flare1,characterized_flare2,characterized_flare3]
# for i in range(len(hopsarray)):
#     #characterized_flare = hop_characterizer([hop], COSI_bkg_rate=COSI_bkg_rate, COSI_Aeff=COSI_Aeff, LAT_Aeff=LAT_Aeff, ph_ratio=ph_ratio, int_flux_ratio=flux_ratio)
#     characterized_flare = characterizedflares[i]
#     hop = hopsarray[i]
#     MDP99_value = characterized_flare[6]
#     plt.text(hop.peak_time, maxflux, f'MDP99: {MDP99_value:.2f}%', fontsize=18, color='red', ha='center', va='bottom')
#     # plot the hop as a shaded block that covers the duration of the flare, filling from 0 to peak flux of the flare.
#     plt.fill_betweenx(y=[0, maxflux], x1=hop.start_time, x2=hop.end_time, color='#FDBF6F' if i%2==0 else '#A6CEE3', alpha=0.5,label= 'hop' if (i==0 or i==1) else '')
#plt.text(blarray[0].peak_time, maxflux, f'MDP99: {MDP99_value:.2f}%', fontsize=18, color='red', ha='center', va='bottom')
plt.xlabel("MJD",fontsize=16)
plt.ylabel('Photon Flux (Photons\u22c5$cm^{-2}\u22c5s^{-1}$) (0.1 - 100 GeV)',fontsize=16)
plt.title("Photon Flux vs Time" ' (Source: ' +str(titlestring)+ ')',fontsize=18 )
plt.legend(loc='upper right', fontsize=13)
plt.show()

COSI_BAND_BAT_weekly_df

In [ ]:

sourcename = '4FGL J0221.1+3556'
sourcearray = cadence_df[cadence_df['source_name'] == sourcename].reset_index(drop=True)
titlestring = sourcename
sourcearray = sourcearray[sourcearray['photon_flux2']!=-3333].reset_index(drop=True)
average_flux = np.nanmean(sourcearray['photon_flux2'])
sourcearray = sourcearray[sourcearray['photon_flux2']<=100*average_flux].reset_index(drop=True)
time = sourcearray['tmin']/SecsInDay + MJDREFI
photon_flux = sourcearray['photon_flux2']
errors = sourcearray['photon_flux_error2']
# Creates the lightcurve object of the source.
sourcelightcurve = LC.LightCurve(time,photon_flux,errors,name=titlestring)
i = COSI_LAT_Sources[COSI_LAT_Sources['Name']==sourcename].index[0]

selected1 = sourcelightcurve.select_by_time(56150,56280)
originalLC = sourcelightcurve
photon_flux = selected1.flux

# Finding the initial flux threshold for detection.
maxflux = np.max(photon_flux)
minflux = np.min(photon_flux)
delta_flux = maxflux - minflux
delta_flux_percent = delta_flux * 0.3
thresholdflux = minflux + delta_flux_percent


# Initial flare detection.
selected1.get_bblocks(gamma_value=0.05)
selected1.find_hop(method='baseline', lc_edges ='add', baseline = thresholdflux)

quiescent_background,qui_err = quiescent_background_finder(sourcelightcurve=selected1,method='forward')


# Re-detecting flares with quiescent background.
sourcelightcurve = originalLC.select_by_time(56200,56250)
sourcelightcurve.get_bblocks(gamma_value=0.05)
sourcelightcurve.find_hop(method = 'baseline', lc_edges ='add',baseline = quiescent_background)

hops_bl1 = sourcelightcurve.hops
plotit = originalLC.select_by_time(56000,56500)
plotit.plot_lc()
sourcelightcurve.plot_hop()


COSI_LAT_Sources = pd.read_csv(table, sep=",", na_filter=True)
# Effective Area of Fermi LAT for the source.
LAT_Aeff = COSI_LAT_Sources['Aeff_mean_LAT(cm2)'][i]

# Effective Area of COSI for the source.
COSI_Aeff = COSI_LAT_Sources['Aeff_mean_COSI(cm2)'][i]

# Name of Source
# Conversion factor of counts between Fermi and COSI Bands. (0.1-100 GeV to 0.2-5 MeV)
ph_ratio = COSI_LAT_Sources['ph/s_ratio'][i]
print(ph_ratio)
#ph_ratio = 1

# Conversion factor of flux between Fermi and COSI Bands. (0.1-100 GeV to 0.2-5 MeV)
flux_ratio = COSI_LAT_Sources['Int_flux_ratio'][i]
print(flux_ratio)

fsrq_names=select_fsrq()
blarray = [hops_bl1]
COSI_BAND_ALL = np.array([0,0,0,0,0,0,0,0,0,0,0,0,0])
for bl in blarray:
    characterized_flares = hop_characterizer(bl, COSI_bkg_rate=COSI_bkg_rate, COSI_Aeff=COSI_Aeff, LAT_Aeff=LAT_Aeff, ph_ratio=ph_ratio, int_flux_ratio=flux_ratio)
    COSI_BAND = np.array(characterized_flares).T
    COSI_BAND_ALL = np.vstack((COSI_BAND_ALL, COSI_BAND))

COSI_BAND_BAT_weekly_df = pd.DataFrame(COSI_BAND_ALL)
COSI_BAND_BAT_weekly_df.columns = ['Name','Duration_(s)', 'Source_Counts(ph)_(0.2-5_MeV)', 'Asymmetry', 'Coverage', 'Background_Counts', 'MDP99_(%)', 'Class', 'Photon_Fluence_(ph/cm2)_(0.2-5_MeV)', 'Average_Photon_Flux_(ph/cm2/s)_(0.2-5_MeV)','Start_Time_(MJD)','Average_Flux_of_Entire_Source_(0.2-5_MeV)', 'Peak_Flare_Flux_(0.2-5_MeV)']
COSI_BAND_BAT_weekly_df = COSI_BAND_BAT_weekly_df[COSI_BAND_BAT_weekly_df[:]['Source_Counts(ph)_(0.2-5_MeV)']!=0].reset_index(drop=True)
COSI_BAND_BAT_weekly_df = COSI_BAND_BAT_weekly_df[COSI_BAND_BAT_weekly_df[:]['Source_Counts(ph)_(0.2-5_MeV)']!='0.0'].reset_index(drop=True)
COSI_BAND_BAT_weekly_df['Source_Counts(ph)_(0.2-5_MeV)'] = COSI_BAND_BAT_weekly_df['Source_Counts(ph)_(0.2-5_MeV)'].astype(float)
COSI_BAND_BAT_weekly_df['Duration_(s)'] = COSI_BAND_BAT_weekly_df['Duration_(s)'].astype(float)
COSI_BAND_BAT_weekly_df['MDP99_(%)'] = ComputeMDP99(COSI_BAND_BAT_weekly_df['Source_Counts(ph)_(0.2-5_MeV)'].astype(float),
                                                COSI_BAND_BAT_weekly_df['Background_Counts'].astype(float))
COSI_BAND_BAT_weekly_df['Coverage'] = COSI_BAND_BAT_weekly_df['Coverage'].astype(float)
COSI_BAND_BAT_weekly_df = COSI_BAND_BAT_weekly_df[COSI_BAND_BAT_weekly_df[:]['Duration_(s)']<=2e8].reset_index(drop=True)


neworder = ['Name','Class','Average_Photon_Flux_(ph/cm2/s)_(0.2-5_MeV)','Photon_Fluence_(ph/cm2)_(0.2-5_MeV)','Source_Counts(ph)_(0.2-5_MeV)','Background_Counts','Duration_(s)','Start_Time_(MJD)','Coverage', 'MDP99_(%)','Asymmetry','Average_Flux_of_Entire_Source_(0.2-5_MeV)', 'Peak_Flare_Flux_(0.2-5_MeV)']
COSI_BAND_BAT_weekly_df
#COSI_BAND_BAT_weekly_df.to_csv(sourcename+'Analysis.csv')
# Remove blank space in sourcename

In [ ]:

initialtime,finaltime = 57700,58400
sourcename = '4FGL J0319.8+4130'
sourcearray = cadence_df[cadence_df['source_name'] == sourcename].reset_index(drop=True)
titlestring = sourcename
sourcearray = sourcearray[sourcearray['photon_flux2']!=-3333].reset_index(drop=True)
average_flux = np.nanmean(sourcearray['photon_flux2'])
sourcearray = sourcearray[sourcearray['photon_flux2']<=100*average_flux].reset_index(drop=True)
time = sourcearray['tmin']/SecsInDay + MJDREFI
photon_flux = sourcearray['photon_flux2']
errors = sourcearray['photon_flux_error2']
# Creates the lightcurve object of the source.
sourcelightcurve = LC.LightCurve(time,photon_flux,errors,name=titlestring)
i = COSI_LAT_Sources[COSI_LAT_Sources['Name']==sourcename].index[0]

selected1 = sourcelightcurve.select_by_time(initialtime,finaltime)
originalLC = sourcelightcurve
photon_flux = selected1.flux

# Finding the initial flux threshold for detection.
maxflux = np.max(photon_flux)
minflux = np.min(photon_flux)
delta_flux = maxflux - minflux
delta_flux_percent = delta_flux * 0.3
thresholdflux = minflux + delta_flux_percent


# Initial flare detection.
selected1.get_bblocks(gamma_value=0.05)
selected1.find_hop(method='baseline', lc_edges ='add', baseline = thresholdflux)

quiescent_background,qui_err = quiescent_background_finder(sourcelightcurve=selected1,method='forward')


# Re-detecting flares with quiescent background.
sourcelightcurve = originalLC.select_by_time(initialtime,finaltime)
sourcelightcurve.get_bblocks(gamma_value=0.05)
sourcelightcurve.find_hop(method = 'baseline', lc_edges ='add',baseline=0.8e-6)

hops_bl1 = sourcelightcurve.hops
plotit = originalLC.select_by_time(initialtime,finaltime)
plotit.plot_lc()
sourcelightcurve.plot_hop()

COSI_LAT_Sources = pd.read_csv(table, sep=",", na_filter=True)
# Effective Area of Fermi LAT for the source.
LAT_Aeff = COSI_LAT_Sources['Aeff_mean_LAT(cm2)'][i]

# Effective Area of COSI for the source.
COSI_Aeff = COSI_LAT_Sources['Aeff_mean_COSI(cm2)'][i]

# Name of Source
# Conversion factor of counts between Fermi and COSI Bands. (0.1-100 GeV to 0.2-5 MeV)
ph_ratio = COSI_LAT_Sources['ph/s_ratio'][i]

#ph_ratio = 1

# Conversion factor of flux between Fermi and COSI Bands. (0.1-100 GeV to 0.2-5 MeV)
flux_ratio = COSI_LAT_Sources['Int_flux_ratio'][i]


fsrq_names=select_fsrq()
blarray = [hops_bl1]
COSI_BAND_ALL = np.array([0,0,0,0,0,0,0,0,0,0,0,0,0])
for bl in blarray:
    characterized_flares = hop_characterizer(bl, COSI_bkg_rate=COSI_bkg_rate, COSI_Aeff=COSI_Aeff, LAT_Aeff=LAT_Aeff, ph_ratio=ph_ratio, int_flux_ratio=flux_ratio)
    COSI_BAND = np.array(characterized_flares).T
    COSI_BAND_ALL = np.vstack((COSI_BAND_ALL, COSI_BAND))

COSI_BAND_BAT_weekly_df = pd.DataFrame(COSI_BAND_ALL)
COSI_BAND_BAT_weekly_df.columns = ['Name','Duration_(s)', 'Source_Counts(ph)_(0.2-5_MeV)', 'Asymmetry', 'Coverage', 'Background_Counts', 'MDP99_(%)', 'Class', 'Photon_Fluence_(ph/cm2)_(0.2-5_MeV)', 'Average_Photon_Flux_(ph/cm2/s)_(0.2-5_MeV)','Start_Time_(MJD)','Average_Flux_of_Entire_Source_(0.2-5_MeV)', 'Peak_Flare_Flux_(0.2-5_MeV)']
COSI_BAND_BAT_weekly_df = COSI_BAND_BAT_weekly_df[COSI_BAND_BAT_weekly_df[:]['Source_Counts(ph)_(0.2-5_MeV)']!=0].reset_index(drop=True)
COSI_BAND_BAT_weekly_df = COSI_BAND_BAT_weekly_df[COSI_BAND_BAT_weekly_df[:]['Source_Counts(ph)_(0.2-5_MeV)']!='0.0'].reset_index(drop=True)
COSI_BAND_BAT_weekly_df['Source_Counts(ph)_(0.2-5_MeV)'] = COSI_BAND_BAT_weekly_df['Source_Counts(ph)_(0.2-5_MeV)'].astype(float)
COSI_BAND_BAT_weekly_df['Duration_(s)'] = COSI_BAND_BAT_weekly_df['Duration_(s)'].astype(float)
COSI_BAND_BAT_weekly_df['MDP99_(%)'] = ComputeMDP99(COSI_BAND_BAT_weekly_df['Source_Counts(ph)_(0.2-5_MeV)'].astype(float),
                                                COSI_BAND_BAT_weekly_df['Background_Counts'].astype(float))
COSI_BAND_BAT_weekly_df['Coverage'] = COSI_BAND_BAT_weekly_df['Coverage'].astype(float)
COSI_BAND_BAT_weekly_df = COSI_BAND_BAT_weekly_df[COSI_BAND_BAT_weekly_df[:]['Duration_(s)']<=2e8].reset_index(drop=True)


neworder = ['Name','Class','Average_Photon_Flux_(ph/cm2/s)_(0.2-5_MeV)','Photon_Fluence_(ph/cm2)_(0.2-5_MeV)','Source_Counts(ph)_(0.2-5_MeV)','Background_Counts','Duration_(s)','Start_Time_(MJD)','Coverage', 'MDP99_(%)','Asymmetry','Average_Flux_of_Entire_Source_(0.2-5_MeV)', 'Peak_Flare_Flux_(0.2-5_MeV)']
COSI_BAND_BAT_weekly_df
#COSI_BAND_BAT_weekly_df.to_csv(sourcename+'Analysis.csv')
# Remove blank space in sourcename

In [ ]:
initialtime,finaltime = 59900,60200
sourcename = '4FGL J0403.9-3605'
sourcearray = cadence_df[cadence_df['source_name'] == sourcename].reset_index(drop=True)
titlestring = sourcename
sourcearray = sourcearray[sourcearray['photon_flux2']!=-3333].reset_index(drop=True)
average_flux = np.nanmean(sourcearray['photon_flux2'])
sourcearray = sourcearray[sourcearray['photon_flux2']<=100*average_flux].reset_index(drop=True)
time = sourcearray['tmin']/SecsInDay + MJDREFI
photon_flux = sourcearray['photon_flux2']
errors = sourcearray['photon_flux_error2']
# Creates the lightcurve object of the source.
sourcelightcurve = LC.LightCurve(time,photon_flux,errors,name=titlestring)
i = COSI_LAT_Sources[COSI_LAT_Sources['Name']==sourcename].index[0]

selected1 = sourcelightcurve.select_by_time(initialtime,finaltime)
originalLC = sourcelightcurve
sourcelightcurve.get_bblocks()
photon_flux = selected1.flux

# Finding the initial flux threshold for detection.
maxflux = np.max(photon_flux)
minflux = np.min(photon_flux)
delta_flux = maxflux - minflux
delta_flux_percent = delta_flux * 0.3
thresholdflux = minflux + delta_flux_percent


# Initial flare detection.
selected1.get_bblocks(gamma_value=0.05)
selected1.find_hop(method='baseline', lc_edges ='add', baseline = thresholdflux)

quiescent_background,qui_err = quiescent_background_finder(sourcelightcurve=selected1,method='forward')


# Re-detecting flares with quiescent background.
sourcelightcurve = originalLC.select_by_time(initialtime,finaltime)
sourcelightcurve.get_bblocks(gamma_value=0.05)
sourcelightcurve.find_hop(method = 'baseline', lc_edges ='add',baseline = 1e-6)

hops_bl1 = sourcelightcurve.hops
plotit = originalLC.select_by_time(initialtime,finaltime)
plotit.plot_lc()
sourcelightcurve.plot_hop()

COSI_LAT_Sources = pd.read_csv(table, sep=",", na_filter=True)
# Effective Area of Fermi LAT for the source.
LAT_Aeff = COSI_LAT_Sources['Aeff_mean_LAT(cm2)'][i]

# Effective Area of COSI for the source.
COSI_Aeff = COSI_LAT_Sources['Aeff_mean_COSI(cm2)'][i]

# Name of Source
# Conversion factor of counts between Fermi and COSI Bands. (0.1-100 GeV to 0.2-5 MeV)
ph_ratio = COSI_LAT_Sources['ph/s_ratio'][i]

#ph_ratio = 1

# Conversion factor of flux between Fermi and COSI Bands. (0.1-100 GeV to 0.2-5 MeV)
flux_ratio = COSI_LAT_Sources['Int_flux_ratio'][i]


fsrq_names=select_fsrq()
blarray = [hops_bl1]
COSI_BAND_ALL = np.array([0,0,0,0,0,0,0,0,0,0,0,0,0])
for bl in blarray:
    characterized_flares = hop_characterizer(bl, COSI_bkg_rate=COSI_bkg_rate, COSI_Aeff=COSI_Aeff, LAT_Aeff=LAT_Aeff, ph_ratio=ph_ratio, int_flux_ratio=flux_ratio)
    COSI_BAND = np.array(characterized_flares).T
    COSI_BAND_ALL = np.vstack((COSI_BAND_ALL, COSI_BAND))

COSI_BAND_BAT_weekly_df = pd.DataFrame(COSI_BAND_ALL)
COSI_BAND_BAT_weekly_df.columns = ['Name','Duration_(s)', 'Source_Counts(ph)_(0.2-5_MeV)', 'Asymmetry', 'Coverage', 'Background_Counts', 'MDP99_(%)', 'Class', 'Photon_Fluence_(ph/cm2)_(0.2-5_MeV)', 'Average_Photon_Flux_(ph/cm2/s)_(0.2-5_MeV)','Start_Time_(MJD)','Average_Flux_of_Entire_Source_(0.2-5_MeV)', 'Peak_Flare_Flux_(0.2-5_MeV)']
COSI_BAND_BAT_weekly_df = COSI_BAND_BAT_weekly_df[COSI_BAND_BAT_weekly_df[:]['Source_Counts(ph)_(0.2-5_MeV)']!=0].reset_index(drop=True)
COSI_BAND_BAT_weekly_df = COSI_BAND_BAT_weekly_df[COSI_BAND_BAT_weekly_df[:]['Source_Counts(ph)_(0.2-5_MeV)']!='0.0'].reset_index(drop=True)
COSI_BAND_BAT_weekly_df['Source_Counts(ph)_(0.2-5_MeV)'] = COSI_BAND_BAT_weekly_df['Source_Counts(ph)_(0.2-5_MeV)'].astype(float)
COSI_BAND_BAT_weekly_df['Duration_(s)'] = COSI_BAND_BAT_weekly_df['Duration_(s)'].astype(float)
COSI_BAND_BAT_weekly_df['MDP99_(%)'] = ComputeMDP99(COSI_BAND_BAT_weekly_df['Source_Counts(ph)_(0.2-5_MeV)'].astype(float),
                                                COSI_BAND_BAT_weekly_df['Background_Counts'].astype(float))
COSI_BAND_BAT_weekly_df['Coverage'] = COSI_BAND_BAT_weekly_df['Coverage'].astype(float)
COSI_BAND_BAT_weekly_df = COSI_BAND_BAT_weekly_df[COSI_BAND_BAT_weekly_df[:]['Duration_(s)']<=2e8].reset_index(drop=True)


neworder = ['Name','Class','Average_Photon_Flux_(ph/cm2/s)_(0.2-5_MeV)','Photon_Fluence_(ph/cm2)_(0.2-5_MeV)','Source_Counts(ph)_(0.2-5_MeV)','Background_Counts','Duration_(s)','Start_Time_(MJD)','Coverage', 'MDP99_(%)','Asymmetry','Average_Flux_of_Entire_Source_(0.2-5_MeV)', 'Peak_Flare_Flux_(0.2-5_MeV)']
COSI_BAND_BAT_weekly_df
#COSI_BAND_BAT_weekly_df.to_csv(sourcename+'Analysis.csv')
# Remove blank space in sourcename

In [ ]:
initialtime,finaltime = 60300,60800
sourcename = '4FGL J0442.6-0017'
sourcearray = cadence_df[cadence_df['source_name'] == sourcename].reset_index(drop=True)
titlestring = sourcename
sourcearray = sourcearray[sourcearray['photon_flux2']!=-3333].reset_index(drop=True)
average_flux = np.nanmean(sourcearray['photon_flux2'])
sourcearray = sourcearray[sourcearray['photon_flux2']<=100*average_flux].reset_index(drop=True)
time = sourcearray['tmin']/SecsInDay + MJDREFI
photon_flux = sourcearray['photon_flux2']
errors = sourcearray['photon_flux_error2']
# Creates the lightcurve object of the source.
sourcelightcurve = LC.LightCurve(time,photon_flux,errors,name=titlestring)
i = COSI_LAT_Sources[COSI_LAT_Sources['Name']==sourcename].index[0]

selected1 = sourcelightcurve.select_by_time(initialtime,finaltime)
originalLC = sourcelightcurve
sourcelightcurve.get_bblocks()
photon_flux = selected1.flux

# Finding the initial flux threshold for detection.
maxflux = np.max(photon_flux)
minflux = np.min(photon_flux)
delta_flux = maxflux - minflux
delta_flux_percent = delta_flux * 0.3
thresholdflux = minflux + delta_flux_percent


# Initial flare detection.
selected1.get_bblocks(gamma_value=0.05)
selected1.find_hop(method='baseline', lc_edges ='add', baseline = thresholdflux)

quiescent_background,qui_err = quiescent_background_finder(sourcelightcurve=selected1,method='forward')


# Re-detecting flares with quiescent background.
sourcelightcurve = originalLC.select_by_time(initialtime,finaltime)
sourcelightcurve.get_bblocks(gamma_value=0.05)
sourcelightcurve.find_hop(method = 'baseline', lc_edges ='add',baseline = 0.6e-6)

hops_bl1 = sourcelightcurve.hops
plotit = originalLC.select_by_time(initialtime,finaltime)
plotit.plot_lc()
sourcelightcurve.plot_hop()


COSI_LAT_Sources = pd.read_csv(table, sep=",", na_filter=True)
# Effective Area of Fermi LAT for the source.
LAT_Aeff = COSI_LAT_Sources['Aeff_mean_LAT(cm2)'][i]

# Effective Area of COSI for the source.
COSI_Aeff = COSI_LAT_Sources['Aeff_mean_COSI(cm2)'][i]

# Name of Source
# Conversion factor of counts between Fermi and COSI Bands. (0.1-100 GeV to 0.2-5 MeV)
ph_ratio = COSI_LAT_Sources['ph/s_ratio'][i]

#ph_ratio = 1

# Conversion factor of flux between Fermi and COSI Bands. (0.1-100 GeV to 0.2-5 MeV)
flux_ratio = COSI_LAT_Sources['Int_flux_ratio'][i]


fsrq_names=select_fsrq()
blarray = [hops_bl1]
COSI_BAND_ALL = np.array([0,0,0,0,0,0,0,0,0,0,0,0,0])
for bl in blarray:
    characterized_flares = hop_characterizer(bl, COSI_bkg_rate=COSI_bkg_rate, COSI_Aeff=COSI_Aeff, LAT_Aeff=LAT_Aeff, ph_ratio=ph_ratio, int_flux_ratio=flux_ratio)
    COSI_BAND = np.array(characterized_flares).T
    COSI_BAND_ALL = np.vstack((COSI_BAND_ALL, COSI_BAND))

COSI_BAND_BAT_weekly_df = pd.DataFrame(COSI_BAND_ALL)
COSI_BAND_BAT_weekly_df.columns = ['Name','Duration_(s)', 'Source_Counts(ph)_(0.2-5_MeV)', 'Asymmetry', 'Coverage', 'Background_Counts', 'MDP99_(%)', 'Class', 'Photon_Fluence_(ph/cm2)_(0.2-5_MeV)', 'Average_Photon_Flux_(ph/cm2/s)_(0.2-5_MeV)','Start_Time_(MJD)','Average_Flux_of_Entire_Source_(0.2-5_MeV)', 'Peak_Flare_Flux_(0.2-5_MeV)']
COSI_BAND_BAT_weekly_df = COSI_BAND_BAT_weekly_df[COSI_BAND_BAT_weekly_df[:]['Source_Counts(ph)_(0.2-5_MeV)']!=0].reset_index(drop=True)
COSI_BAND_BAT_weekly_df = COSI_BAND_BAT_weekly_df[COSI_BAND_BAT_weekly_df[:]['Source_Counts(ph)_(0.2-5_MeV)']!='0.0'].reset_index(drop=True)
COSI_BAND_BAT_weekly_df['Source_Counts(ph)_(0.2-5_MeV)'] = COSI_BAND_BAT_weekly_df['Source_Counts(ph)_(0.2-5_MeV)'].astype(float)
COSI_BAND_BAT_weekly_df['Duration_(s)'] = COSI_BAND_BAT_weekly_df['Duration_(s)'].astype(float)
COSI_BAND_BAT_weekly_df['MDP99_(%)'] = ComputeMDP99(COSI_BAND_BAT_weekly_df['Source_Counts(ph)_(0.2-5_MeV)'].astype(float),
                                                COSI_BAND_BAT_weekly_df['Background_Counts'].astype(float))
COSI_BAND_BAT_weekly_df['Coverage'] = COSI_BAND_BAT_weekly_df['Coverage'].astype(float)
COSI_BAND_BAT_weekly_df = COSI_BAND_BAT_weekly_df[COSI_BAND_BAT_weekly_df[:]['Duration_(s)']<=2e8].reset_index(drop=True)


neworder = ['Name','Class','Average_Photon_Flux_(ph/cm2/s)_(0.2-5_MeV)','Photon_Fluence_(ph/cm2)_(0.2-5_MeV)','Source_Counts(ph)_(0.2-5_MeV)','Background_Counts','Duration_(s)','Start_Time_(MJD)','Coverage', 'MDP99_(%)','Asymmetry','Average_Flux_of_Entire_Source_(0.2-5_MeV)', 'Peak_Flare_Flux_(0.2-5_MeV)']
COSI_BAND_BAT_weekly_df
#COSI_BAND_BAT_weekly_df.to_csv(sourcename+'Analysis.csv')
# Remove blank space in sourcename

In [ ]:
initialtime,finaltime = 56450,56550
sourcename = '4FGL J0442.6-0017'
sourcearray = cadence_df[cadence_df['source_name'] == sourcename].reset_index(drop=True)
titlestring = sourcename
sourcearray = sourcearray[sourcearray['photon_flux2']!=-3333].reset_index(drop=True)
average_flux = np.nanmean(sourcearray['photon_flux2'])
sourcearray = sourcearray[sourcearray['photon_flux2']<=100*average_flux].reset_index(drop=True)
time = sourcearray['tmin']/SecsInDay + MJDREFI
photon_flux = sourcearray['photon_flux2']
errors = sourcearray['photon_flux_error2']
# Creates the lightcurve object of the source.
sourcelightcurve = LC.LightCurve(time,photon_flux,errors,name=titlestring)
i = COSI_LAT_Sources[COSI_LAT_Sources['Name']==sourcename].index[0]

selected1 = sourcelightcurve.select_by_time(initialtime,finaltime)
originalLC = sourcelightcurve
sourcelightcurve.get_bblocks()
photon_flux = selected1.flux

# Finding the initial flux threshold for detection.
maxflux = np.max(photon_flux)
minflux = np.min(photon_flux)
delta_flux = maxflux - minflux
delta_flux_percent = delta_flux * 0.3
thresholdflux = minflux + delta_flux_percent


# Initial flare detection.
selected1.get_bblocks(gamma_value=0.05)
selected1.find_hop(method='baseline', lc_edges ='add', baseline = thresholdflux)

quiescent_background,qui_err = quiescent_background_finder(sourcelightcurve=selected1,method='forward')


# Re-detecting flares with quiescent background.
sourcelightcurve = originalLC.select_by_time(initialtime,finaltime)
sourcelightcurve.get_bblocks(gamma_value=0.05)
sourcelightcurve.find_hop(method = 'baseline', lc_edges ='add',baseline = 0.6e-6)

hops_bl1 = sourcelightcurve.hops
plotit = originalLC.select_by_time(initialtime,finaltime)
plotit.plot_lc()
sourcelightcurve.plot_hop()

COSI_LAT_Sources = pd.read_csv(table, sep=",", na_filter=True)
# Effective Area of Fermi LAT for the source.
LAT_Aeff = COSI_LAT_Sources['Aeff_mean_LAT(cm2)'][i]

# Effective Area of COSI for the source.
COSI_Aeff = COSI_LAT_Sources['Aeff_mean_COSI(cm2)'][i]

# Name of Source
# Conversion factor of counts between Fermi and COSI Bands. (0.1-100 GeV to 0.2-5 MeV)
ph_ratio = COSI_LAT_Sources['ph/s_ratio'][i]

#ph_ratio = 1

# Conversion factor of flux between Fermi and COSI Bands. (0.1-100 GeV to 0.2-5 MeV)
flux_ratio = COSI_LAT_Sources['Int_flux_ratio'][i]


fsrq_names=select_fsrq()
blarray = [hops_bl1]
COSI_BAND_ALL = np.array([0,0,0,0,0,0,0,0,0,0,0,0,0])
for bl in blarray:
    characterized_flares = hop_characterizer(bl, COSI_bkg_rate=COSI_bkg_rate, COSI_Aeff=COSI_Aeff, LAT_Aeff=LAT_Aeff, ph_ratio=ph_ratio, int_flux_ratio=flux_ratio)
    COSI_BAND = np.array(characterized_flares).T
    COSI_BAND_ALL = np.vstack((COSI_BAND_ALL, COSI_BAND))

COSI_BAND_BAT_weekly_df = pd.DataFrame(COSI_BAND_ALL)
COSI_BAND_BAT_weekly_df.columns = ['Name','Duration_(s)', 'Source_Counts(ph)_(0.2-5_MeV)', 'Asymmetry', 'Coverage', 'Background_Counts', 'MDP99_(%)', 'Class', 'Photon_Fluence_(ph/cm2)_(0.2-5_MeV)', 'Average_Photon_Flux_(ph/cm2/s)_(0.2-5_MeV)','Start_Time_(MJD)','Average_Flux_of_Entire_Source_(0.2-5_MeV)', 'Peak_Flare_Flux_(0.2-5_MeV)']
COSI_BAND_BAT_weekly_df = COSI_BAND_BAT_weekly_df[COSI_BAND_BAT_weekly_df[:]['Source_Counts(ph)_(0.2-5_MeV)']!=0].reset_index(drop=True)
COSI_BAND_BAT_weekly_df = COSI_BAND_BAT_weekly_df[COSI_BAND_BAT_weekly_df[:]['Source_Counts(ph)_(0.2-5_MeV)']!='0.0'].reset_index(drop=True)
COSI_BAND_BAT_weekly_df['Source_Counts(ph)_(0.2-5_MeV)'] = COSI_BAND_BAT_weekly_df['Source_Counts(ph)_(0.2-5_MeV)'].astype(float)
COSI_BAND_BAT_weekly_df['Duration_(s)'] = COSI_BAND_BAT_weekly_df['Duration_(s)'].astype(float)
COSI_BAND_BAT_weekly_df['MDP99_(%)'] = ComputeMDP99(COSI_BAND_BAT_weekly_df['Source_Counts(ph)_(0.2-5_MeV)'].astype(float),
                                                COSI_BAND_BAT_weekly_df['Background_Counts'].astype(float))
COSI_BAND_BAT_weekly_df['Coverage'] = COSI_BAND_BAT_weekly_df['Coverage'].astype(float)
COSI_BAND_BAT_weekly_df = COSI_BAND_BAT_weekly_df[COSI_BAND_BAT_weekly_df[:]['Duration_(s)']<=2e8].reset_index(drop=True)


neworder = ['Name','Class','Average_Photon_Flux_(ph/cm2/s)_(0.2-5_MeV)','Photon_Fluence_(ph/cm2)_(0.2-5_MeV)','Source_Counts(ph)_(0.2-5_MeV)','Background_Counts','Duration_(s)','Start_Time_(MJD)','Coverage', 'MDP99_(%)','Asymmetry','Average_Flux_of_Entire_Source_(0.2-5_MeV)', 'Peak_Flare_Flux_(0.2-5_MeV)']
COSI_BAND_BAT_weekly_df
#COSI_BAND_BAT_weekly_df.to_csv(sourcename+'Analysis.csv')
# Remove blank space in sourcename

In [ ]:
initialtime,finaltime = 55200,56100
sourcename = '4FGL J0538.8-4405'
sourcearray = cadence_df[cadence_df['source_name'] == sourcename].reset_index(drop=True)
titlestring = sourcename
sourcearray = sourcearray[sourcearray['photon_flux2']!=-3333].reset_index(drop=True)
average_flux = np.nanmean(sourcearray['photon_flux2'])
sourcearray = sourcearray[sourcearray['photon_flux2']<=100*average_flux].reset_index(drop=True)
time = sourcearray['tmin']/SecsInDay + MJDREFI
photon_flux = sourcearray['photon_flux2']
errors = sourcearray['photon_flux_error2']
# Creates the lightcurve object of the source.
sourcelightcurve = LC.LightCurve(time,photon_flux,errors,name=titlestring)
i = COSI_LAT_Sources[COSI_LAT_Sources['Name']==sourcename].index[0]

selected1 = sourcelightcurve.select_by_time(initialtime,finaltime)
originalLC = sourcelightcurve
sourcelightcurve.get_bblocks()
photon_flux = selected1.flux

# Finding the initial flux threshold for detection.
maxflux = np.max(photon_flux)
minflux = np.min(photon_flux)
delta_flux = maxflux - minflux
delta_flux_percent = delta_flux * 0.3
thresholdflux = minflux + delta_flux_percent


# Initial flare detection.
selected1.get_bblocks(gamma_value=0.05)
selected1.find_hop(method='baseline', lc_edges ='add', baseline = thresholdflux)

quiescent_background,qui_err = quiescent_background_finder(sourcelightcurve=selected1,method='forward')


# Re-detecting flares with quiescent background.
sourcelightcurve = originalLC.select_by_time(initialtime,finaltime)
sourcelightcurve.get_bblocks(gamma_value=0.05)
sourcelightcurve.find_hop(method = 'baseline', lc_edges ='add',baseline = 0.7e-6)

hops_bl1 = sourcelightcurve.hops
plotit = originalLC.select_by_time(initialtime,finaltime)
plotit.plot_lc()
sourcelightcurve.plot_hop()

COSI_LAT_Sources = pd.read_csv(table, sep=",", na_filter=True)
# Effective Area of Fermi LAT for the source.
LAT_Aeff = COSI_LAT_Sources['Aeff_mean_LAT(cm2)'][i]

# Effective Area of COSI for the source.
COSI_Aeff = COSI_LAT_Sources['Aeff_mean_COSI(cm2)'][i]

# Name of Source
# Conversion factor of counts between Fermi and COSI Bands. (0.1-100 GeV to 0.2-5 MeV)
ph_ratio = COSI_LAT_Sources['ph/s_ratio'][i]
print(ph_ratio)

#ph_ratio = 1

# Conversion factor of flux between Fermi and COSI Bands. (0.1-100 GeV to 0.2-5 MeV)
flux_ratio = COSI_LAT_Sources['Int_flux_ratio'][i]


fsrq_names=select_fsrq()
blarray = [hops_bl1]
COSI_BAND_ALL = np.array([0,0,0,0,0,0,0,0,0,0,0,0,0])
for bl in blarray:
    characterized_flares = hop_characterizer(bl, COSI_bkg_rate=COSI_bkg_rate, COSI_Aeff=COSI_Aeff, LAT_Aeff=LAT_Aeff, ph_ratio=ph_ratio, int_flux_ratio=flux_ratio)
    COSI_BAND = np.array(characterized_flares).T
    COSI_BAND_ALL = np.vstack((COSI_BAND_ALL, COSI_BAND))

COSI_BAND_BAT_weekly_df = pd.DataFrame(COSI_BAND_ALL)
COSI_BAND_BAT_weekly_df.columns = ['Name','Duration_(s)', 'Source_Counts(ph)_(0.2-5_MeV)', 'Asymmetry', 'Coverage', 'Background_Counts', 'MDP99_(%)', 'Class', 'Photon_Fluence_(ph/cm2)_(0.2-5_MeV)', 'Average_Photon_Flux_(ph/cm2/s)_(0.2-5_MeV)','Start_Time_(MJD)','Average_Flux_of_Entire_Source_(0.2-5_MeV)', 'Peak_Flare_Flux_(0.2-5_MeV)']
COSI_BAND_BAT_weekly_df = COSI_BAND_BAT_weekly_df[COSI_BAND_BAT_weekly_df[:]['Source_Counts(ph)_(0.2-5_MeV)']!=0].reset_index(drop=True)
COSI_BAND_BAT_weekly_df = COSI_BAND_BAT_weekly_df[COSI_BAND_BAT_weekly_df[:]['Source_Counts(ph)_(0.2-5_MeV)']!='0.0'].reset_index(drop=True)
COSI_BAND_BAT_weekly_df['Source_Counts(ph)_(0.2-5_MeV)'] = COSI_BAND_BAT_weekly_df['Source_Counts(ph)_(0.2-5_MeV)'].astype(float)
COSI_BAND_BAT_weekly_df['Duration_(s)'] = COSI_BAND_BAT_weekly_df['Duration_(s)'].astype(float)
COSI_BAND_BAT_weekly_df['MDP99_(%)'] = ComputeMDP99(COSI_BAND_BAT_weekly_df['Source_Counts(ph)_(0.2-5_MeV)'].astype(float),
                                                COSI_BAND_BAT_weekly_df['Background_Counts'].astype(float))
COSI_BAND_BAT_weekly_df['Coverage'] = COSI_BAND_BAT_weekly_df['Coverage'].astype(float)
COSI_BAND_BAT_weekly_df = COSI_BAND_BAT_weekly_df[COSI_BAND_BAT_weekly_df[:]['Duration_(s)']<=2e8].reset_index(drop=True)


neworder = ['Name','Class','Average_Photon_Flux_(ph/cm2/s)_(0.2-5_MeV)','Photon_Fluence_(ph/cm2)_(0.2-5_MeV)','Source_Counts(ph)_(0.2-5_MeV)','Background_Counts','Duration_(s)','Start_Time_(MJD)','Coverage', 'MDP99_(%)','Asymmetry','Average_Flux_of_Entire_Source_(0.2-5_MeV)', 'Peak_Flare_Flux_(0.2-5_MeV)']
COSI_BAND_BAT_weekly_df
#COSI_BAND_BAT_weekly_df.to_csv(sourcename+'Analysis.csv')
# Remove blank space in sourcename

In [ ]:
initialtime,finaltime = 58800,59100
sourcename = '4FGL J0904.9-5734'
sourcearray = cadence_df[cadence_df['source_name'] == sourcename].reset_index(drop=True)
titlestring = sourcename
sourcearray = sourcearray[sourcearray['photon_flux2']!=-3333].reset_index(drop=True)
average_flux = np.nanmean(sourcearray['photon_flux2'])
sourcearray = sourcearray[sourcearray['photon_flux2']<=100*average_flux].reset_index(drop=True)
time = sourcearray['tmin']/SecsInDay + MJDREFI
photon_flux = sourcearray['photon_flux2']
errors = sourcearray['photon_flux_error2']
# Creates the lightcurve object of the source.
sourcelightcurve = LC.LightCurve(time,photon_flux,errors,name=titlestring)
i = COSI_LAT_Sources[COSI_LAT_Sources['Name']==sourcename].index[0]

selected1 = sourcelightcurve.select_by_time(initialtime,finaltime)
originalLC = sourcelightcurve
sourcelightcurve.get_bblocks()
photon_flux = selected1.flux

# Finding the initial flux threshold for detection.
maxflux = np.max(photon_flux)
minflux = np.min(photon_flux)
delta_flux = maxflux - minflux
delta_flux_percent = delta_flux * 0.3
thresholdflux = minflux + delta_flux_percent


# Initial flare detection.
selected1.get_bblocks(gamma_value=0.05)
selected1.find_hop(method='baseline', lc_edges ='add', baseline = thresholdflux)

quiescent_background,qui_err = quiescent_background_finder(sourcelightcurve=selected1,method='forward')


# Re-detecting flares with quiescent background.
sourcelightcurve = originalLC.select_by_time(initialtime,finaltime)
sourcelightcurve.get_bblocks(gamma_value=0.05)
sourcelightcurve.find_hop(method = 'baseline', lc_edges ='add',baseline = 2e-6)

hops_bl1 = sourcelightcurve.hops
plotit = originalLC.select_by_time(initialtime,finaltime)
plotit.plot_lc()
sourcelightcurve.plot_hop()

COSI_LAT_Sources = pd.read_csv(table, sep=",", na_filter=True)
# Effective Area of Fermi LAT for the source.
LAT_Aeff = COSI_LAT_Sources['Aeff_mean_LAT(cm2)'][i]

# Effective Area of COSI for the source.
COSI_Aeff = COSI_LAT_Sources['Aeff_mean_COSI(cm2)'][i]

# Name of Source
# Conversion factor of counts between Fermi and COSI Bands. (0.1-100 GeV to 0.2-5 MeV)
ph_ratio = COSI_LAT_Sources['ph/s_ratio'][i]

#ph_ratio = 1

# Conversion factor of flux between Fermi and COSI Bands. (0.1-100 GeV to 0.2-5 MeV)
flux_ratio = COSI_LAT_Sources['Int_flux_ratio'][i]


fsrq_names=select_fsrq()
blarray = [hops_bl1]
COSI_BAND_ALL = np.array([0,0,0,0,0,0,0,0,0,0,0,0,0])
for bl in blarray:
    characterized_flares = hop_characterizer(bl, COSI_bkg_rate=COSI_bkg_rate, COSI_Aeff=COSI_Aeff, LAT_Aeff=LAT_Aeff, ph_ratio=ph_ratio, int_flux_ratio=flux_ratio)
    COSI_BAND = np.array(characterized_flares).T
    COSI_BAND_ALL = np.vstack((COSI_BAND_ALL, COSI_BAND))

COSI_BAND_BAT_weekly_df = pd.DataFrame(COSI_BAND_ALL)
COSI_BAND_BAT_weekly_df.columns = ['Name','Duration_(s)', 'Source_Counts(ph)_(0.2-5_MeV)', 'Asymmetry', 'Coverage', 'Background_Counts', 'MDP99_(%)', 'Class', 'Photon_Fluence_(ph/cm2)_(0.2-5_MeV)', 'Average_Photon_Flux_(ph/cm2/s)_(0.2-5_MeV)','Start_Time_(MJD)','Average_Flux_of_Entire_Source_(0.2-5_MeV)', 'Peak_Flare_Flux_(0.2-5_MeV)']
COSI_BAND_BAT_weekly_df = COSI_BAND_BAT_weekly_df[COSI_BAND_BAT_weekly_df[:]['Source_Counts(ph)_(0.2-5_MeV)']!=0].reset_index(drop=True)
COSI_BAND_BAT_weekly_df = COSI_BAND_BAT_weekly_df[COSI_BAND_BAT_weekly_df[:]['Source_Counts(ph)_(0.2-5_MeV)']!='0.0'].reset_index(drop=True)
COSI_BAND_BAT_weekly_df['Source_Counts(ph)_(0.2-5_MeV)'] = COSI_BAND_BAT_weekly_df['Source_Counts(ph)_(0.2-5_MeV)'].astype(float)
COSI_BAND_BAT_weekly_df['Duration_(s)'] = COSI_BAND_BAT_weekly_df['Duration_(s)'].astype(float)
COSI_BAND_BAT_weekly_df['MDP99_(%)'] = ComputeMDP99(COSI_BAND_BAT_weekly_df['Source_Counts(ph)_(0.2-5_MeV)'].astype(float),
                                                COSI_BAND_BAT_weekly_df['Background_Counts'].astype(float))
COSI_BAND_BAT_weekly_df['Coverage'] = COSI_BAND_BAT_weekly_df['Coverage'].astype(float)
COSI_BAND_BAT_weekly_df = COSI_BAND_BAT_weekly_df[COSI_BAND_BAT_weekly_df[:]['Duration_(s)']<=2e8].reset_index(drop=True)


neworder = ['Name','Class','Average_Photon_Flux_(ph/cm2/s)_(0.2-5_MeV)','Photon_Fluence_(ph/cm2)_(0.2-5_MeV)','Source_Counts(ph)_(0.2-5_MeV)','Background_Counts','Duration_(s)','Start_Time_(MJD)','Coverage', 'MDP99_(%)','Asymmetry','Average_Flux_of_Entire_Source_(0.2-5_MeV)', 'Peak_Flare_Flux_(0.2-5_MeV)']
COSI_BAND_BAT_weekly_df
#COSI_BAND_BAT_weekly_df.to_csv(sourcename+'Analysis.csv')
# Remove blank space in sourcename

In [ ]:
initialtime,finaltime = 55500,55900
sourcename = '4FGL J1153.4+4931'
sourcearray = cadence_df[cadence_df['source_name'] == sourcename].reset_index(drop=True)
titlestring = sourcename
sourcearray = sourcearray[sourcearray['photon_flux2']!=-3333].reset_index(drop=True)
average_flux = np.nanmean(sourcearray['photon_flux2'])
sourcearray = sourcearray[sourcearray['photon_flux2']<=100*average_flux].reset_index(drop=True)
time = sourcearray['tmin']/SecsInDay + MJDREFI
photon_flux = sourcearray['photon_flux2']
errors = sourcearray['photon_flux_error2']
# Creates the lightcurve object of the source.
sourcelightcurve = LC.LightCurve(time,photon_flux,errors,name=titlestring)
i = COSI_LAT_Sources[COSI_LAT_Sources['Name']==sourcename].index[0]

selected1 = sourcelightcurve.select_by_time(initialtime,finaltime)
originalLC = sourcelightcurve
sourcelightcurve.get_bblocks()
photon_flux = selected1.flux

# Finding the initial flux threshold for detection.
maxflux = np.max(photon_flux)
minflux = np.min(photon_flux)
delta_flux = maxflux - minflux
delta_flux_percent = delta_flux * 0.3
thresholdflux = minflux + delta_flux_percent


# Initial flare detection.
selected1.get_bblocks(gamma_value=0.05)
selected1.find_hop(method='baseline', lc_edges ='add', baseline = thresholdflux)

quiescent_background,qui_err = quiescent_background_finder(sourcelightcurve=selected1,method='forward')


# Re-detecting flares with quiescent background.
sourcelightcurve = originalLC.select_by_time(initialtime,finaltime)
sourcelightcurve.get_bblocks(gamma_value=0.05)
sourcelightcurve.find_hop(method = 'baseline', lc_edges ='add',baseline = 0.2e-6)

hops_bl1 = sourcelightcurve.hops
plotit = originalLC.select_by_time(initialtime,finaltime)
plotit.plot_lc()
sourcelightcurve.plot_hop()

COSI_LAT_Sources = pd.read_csv(table, sep=",", na_filter=True)
# Effective Area of Fermi LAT for the source.
LAT_Aeff = COSI_LAT_Sources['Aeff_mean_LAT(cm2)'][i]

# Effective Area of COSI for the source.
COSI_Aeff = COSI_LAT_Sources['Aeff_mean_COSI(cm2)'][i]

# Name of Source
# Conversion factor of counts between Fermi and COSI Bands. (0.1-100 GeV to 0.2-5 MeV)
ph_ratio = COSI_LAT_Sources['ph/s_ratio'][i]

#ph_ratio = 1

# Conversion factor of flux between Fermi and COSI Bands. (0.1-100 GeV to 0.2-5 MeV)
flux_ratio = COSI_LAT_Sources['Int_flux_ratio'][i]


fsrq_names=select_fsrq()
blarray = [hops_bl1]
COSI_BAND_ALL = np.array([0,0,0,0,0,0,0,0,0,0,0,0,0])
for bl in blarray:
    characterized_flares = hop_characterizer(bl, COSI_bkg_rate=COSI_bkg_rate, COSI_Aeff=COSI_Aeff, LAT_Aeff=LAT_Aeff, ph_ratio=ph_ratio, int_flux_ratio=flux_ratio)
    COSI_BAND = np.array(characterized_flares).T
    COSI_BAND_ALL = np.vstack((COSI_BAND_ALL, COSI_BAND))

COSI_BAND_BAT_weekly_df = pd.DataFrame(COSI_BAND_ALL)
COSI_BAND_BAT_weekly_df.columns = ['Name','Duration_(s)', 'Source_Counts(ph)_(0.2-5_MeV)', 'Asymmetry', 'Coverage', 'Background_Counts', 'MDP99_(%)', 'Class', 'Photon_Fluence_(ph/cm2)_(0.2-5_MeV)', 'Average_Photon_Flux_(ph/cm2/s)_(0.2-5_MeV)','Start_Time_(MJD)','Average_Flux_of_Entire_Source_(0.2-5_MeV)', 'Peak_Flare_Flux_(0.2-5_MeV)']
COSI_BAND_BAT_weekly_df = COSI_BAND_BAT_weekly_df[COSI_BAND_BAT_weekly_df[:]['Source_Counts(ph)_(0.2-5_MeV)']!=0].reset_index(drop=True)
COSI_BAND_BAT_weekly_df = COSI_BAND_BAT_weekly_df[COSI_BAND_BAT_weekly_df[:]['Source_Counts(ph)_(0.2-5_MeV)']!='0.0'].reset_index(drop=True)
COSI_BAND_BAT_weekly_df['Source_Counts(ph)_(0.2-5_MeV)'] = COSI_BAND_BAT_weekly_df['Source_Counts(ph)_(0.2-5_MeV)'].astype(float)
COSI_BAND_BAT_weekly_df['Duration_(s)'] = COSI_BAND_BAT_weekly_df['Duration_(s)'].astype(float)
COSI_BAND_BAT_weekly_df['MDP99_(%)'] = ComputeMDP99(COSI_BAND_BAT_weekly_df['Source_Counts(ph)_(0.2-5_MeV)'].astype(float),
                                                COSI_BAND_BAT_weekly_df['Background_Counts'].astype(float))
COSI_BAND_BAT_weekly_df['Coverage'] = COSI_BAND_BAT_weekly_df['Coverage'].astype(float)
COSI_BAND_BAT_weekly_df = COSI_BAND_BAT_weekly_df[COSI_BAND_BAT_weekly_df[:]['Duration_(s)']<=2e8].reset_index(drop=True)


neworder = ['Name','Class','Average_Photon_Flux_(ph/cm2/s)_(0.2-5_MeV)','Photon_Fluence_(ph/cm2)_(0.2-5_MeV)','Source_Counts(ph)_(0.2-5_MeV)','Background_Counts','Duration_(s)','Start_Time_(MJD)','Coverage', 'MDP99_(%)','Asymmetry','Average_Flux_of_Entire_Source_(0.2-5_MeV)', 'Peak_Flare_Flux_(0.2-5_MeV)']
COSI_BAND_BAT_weekly_df
#COSI_BAND_BAT_weekly_df.to_csv(sourcename+'Analysis.csv')
# Remove blank space in sourcename

In [ ]:
initialtime,finaltime = 59700,60200
sourcename = '4FGL J1159.5+2914'
sourcearray = cadence_df[cadence_df['source_name'] == sourcename].reset_index(drop=True)
titlestring = sourcename
sourcearray = sourcearray[sourcearray['photon_flux2']!=-3333].reset_index(drop=True)
average_flux = np.nanmean(sourcearray['photon_flux2'])
sourcearray = sourcearray[sourcearray['photon_flux2']<=100*average_flux].reset_index(drop=True)
time = sourcearray['tmin']/SecsInDay + MJDREFI
photon_flux = sourcearray['photon_flux2']
errors = sourcearray['photon_flux_error2']
# Creates the lightcurve object of the source.
sourcelightcurve = LC.LightCurve(time,photon_flux,errors,name=titlestring)
i = COSI_LAT_Sources[COSI_LAT_Sources['Name']==sourcename].index[0]

selected1 = sourcelightcurve.select_by_time(initialtime,finaltime)
originalLC = sourcelightcurve
sourcelightcurve.get_bblocks()
photon_flux = selected1.flux

# Finding the initial flux threshold for detection.
maxflux = np.max(photon_flux)
minflux = np.min(photon_flux)
delta_flux = maxflux - minflux
delta_flux_percent = delta_flux * 0.3
thresholdflux = minflux + delta_flux_percent


# Initial flare detection.
selected1.get_bblocks(gamma_value=0.05)
selected1.find_hop(method='baseline', lc_edges ='add', baseline = thresholdflux)

quiescent_background,qui_err = quiescent_background_finder(sourcelightcurve=selected1,method='forward')


# Re-detecting flares with quiescent background.
sourcelightcurve = originalLC.select_by_time(initialtime,finaltime)
sourcelightcurve.get_bblocks(gamma_value=0.05)
sourcelightcurve.find_hop(method = 'baseline', lc_edges ='add',baseline = 1.3e-6)

hops_bl1 = sourcelightcurve.hops
plotit = originalLC.select_by_time(initialtime,finaltime)
plotit.plot_lc()
sourcelightcurve.plot_hop()

COSI_LAT_Sources = pd.read_csv(table, sep=",", na_filter=True)
# Effective Area of Fermi LAT for the source.
LAT_Aeff = COSI_LAT_Sources['Aeff_mean_LAT(cm2)'][i]

# Effective Area of COSI for the source.
COSI_Aeff = COSI_LAT_Sources['Aeff_mean_COSI(cm2)'][i]

# Name of Source
# Conversion factor of counts between Fermi and COSI Bands. (0.1-100 GeV to 0.2-5 MeV)
ph_ratio = COSI_LAT_Sources['ph/s_ratio'][i]

#ph_ratio = 1

# Conversion factor of flux between Fermi and COSI Bands. (0.1-100 GeV to 0.2-5 MeV)
flux_ratio = COSI_LAT_Sources['Int_flux_ratio'][i]


fsrq_names=select_fsrq()
blarray = [hops_bl1]
COSI_BAND_ALL = np.array([0,0,0,0,0,0,0,0,0,0,0,0,0])
for bl in blarray:
    characterized_flares = hop_characterizer(bl, COSI_bkg_rate=COSI_bkg_rate, COSI_Aeff=COSI_Aeff, LAT_Aeff=LAT_Aeff, ph_ratio=ph_ratio, int_flux_ratio=flux_ratio)
    COSI_BAND = np.array(characterized_flares).T
    COSI_BAND_ALL = np.vstack((COSI_BAND_ALL, COSI_BAND))

COSI_BAND_BAT_weekly_df = pd.DataFrame(COSI_BAND_ALL)
COSI_BAND_BAT_weekly_df.columns = ['Name','Duration_(s)', 'Source_Counts(ph)_(0.2-5_MeV)', 'Asymmetry', 'Coverage', 'Background_Counts', 'MDP99_(%)', 'Class', 'Photon_Fluence_(ph/cm2)_(0.2-5_MeV)', 'Average_Photon_Flux_(ph/cm2/s)_(0.2-5_MeV)','Start_Time_(MJD)','Average_Flux_of_Entire_Source_(0.2-5_MeV)', 'Peak_Flare_Flux_(0.2-5_MeV)']
COSI_BAND_BAT_weekly_df = COSI_BAND_BAT_weekly_df[COSI_BAND_BAT_weekly_df[:]['Source_Counts(ph)_(0.2-5_MeV)']!=0].reset_index(drop=True)
COSI_BAND_BAT_weekly_df = COSI_BAND_BAT_weekly_df[COSI_BAND_BAT_weekly_df[:]['Source_Counts(ph)_(0.2-5_MeV)']!='0.0'].reset_index(drop=True)
COSI_BAND_BAT_weekly_df['Source_Counts(ph)_(0.2-5_MeV)'] = COSI_BAND_BAT_weekly_df['Source_Counts(ph)_(0.2-5_MeV)'].astype(float)
COSI_BAND_BAT_weekly_df['Duration_(s)'] = COSI_BAND_BAT_weekly_df['Duration_(s)'].astype(float)
COSI_BAND_BAT_weekly_df['MDP99_(%)'] = ComputeMDP99(COSI_BAND_BAT_weekly_df['Source_Counts(ph)_(0.2-5_MeV)'].astype(float),
                                                COSI_BAND_BAT_weekly_df['Background_Counts'].astype(float))
COSI_BAND_BAT_weekly_df['Coverage'] = COSI_BAND_BAT_weekly_df['Coverage'].astype(float)
COSI_BAND_BAT_weekly_df = COSI_BAND_BAT_weekly_df[COSI_BAND_BAT_weekly_df[:]['Duration_(s)']<=2e8].reset_index(drop=True)


neworder = ['Name','Class','Average_Photon_Flux_(ph/cm2/s)_(0.2-5_MeV)','Photon_Fluence_(ph/cm2)_(0.2-5_MeV)','Source_Counts(ph)_(0.2-5_MeV)','Background_Counts','Duration_(s)','Start_Time_(MJD)','Coverage', 'MDP99_(%)','Asymmetry','Average_Flux_of_Entire_Source_(0.2-5_MeV)', 'Peak_Flare_Flux_(0.2-5_MeV)']
COSI_BAND_BAT_weekly_df
#COSI_BAND_BAT_weekly_df.to_csv(sourcename+'Analysis.csv')
# Remove blank space in sourcename

In [ ]:
initialtime,finaltime = 55000,55700
sourcename = '4FGL J1224.9+2122'
sourcearray = cadence_df[cadence_df['source_name'] == sourcename].reset_index(drop=True)
titlestring = sourcename
sourcearray = sourcearray[sourcearray['photon_flux2']!=-3333].reset_index(drop=True)
average_flux = np.nanmean(sourcearray['photon_flux2'])
sourcearray = sourcearray[sourcearray['photon_flux2']<=100*average_flux].reset_index(drop=True)
time = sourcearray['tmin']/SecsInDay + MJDREFI
photon_flux = sourcearray['photon_flux2']
errors = sourcearray['photon_flux_error2']
# Creates the lightcurve object of the source.
sourcelightcurve = LC.LightCurve(time,photon_flux,errors,name=titlestring)
i = COSI_LAT_Sources[COSI_LAT_Sources['Name']==sourcename].index[0]

selected1 = sourcelightcurve.select_by_time(initialtime,finaltime)
originalLC = sourcelightcurve
sourcelightcurve.get_bblocks()
photon_flux = selected1.flux

# Finding the initial flux threshold for detection.
maxflux = np.max(photon_flux)
minflux = np.min(photon_flux)
delta_flux = maxflux - minflux
delta_flux_percent = delta_flux * 0.3
thresholdflux = minflux + delta_flux_percent


# Initial flare detection.
selected1.get_bblocks(gamma_value=0.05)
selected1.find_hop(method='baseline', lc_edges ='add', baseline = thresholdflux)

quiescent_background,qui_err = quiescent_background_finder(sourcelightcurve=selected1,method='forward')


# Re-detecting flares with quiescent background.
sourcelightcurve = originalLC.select_by_time(initialtime,finaltime)
sourcelightcurve.get_bblocks(gamma_value=0.05)
sourcelightcurve.find_hop(method = 'baseline', lc_edges ='add',baseline = 1.7e-6)

hops_bl1 = sourcelightcurve.hops
plotit = originalLC.select_by_time(initialtime,finaltime)
plotit.plot_lc()
sourcelightcurve.plot_hop()

COSI_LAT_Sources = pd.read_csv(table, sep=",", na_filter=True)
# Effective Area of Fermi LAT for the source.
LAT_Aeff = COSI_LAT_Sources['Aeff_mean_LAT(cm2)'][i]

# Effective Area of COSI for the source.
COSI_Aeff = COSI_LAT_Sources['Aeff_mean_COSI(cm2)'][i]

# Name of Source
# Conversion factor of counts between Fermi and COSI Bands. (0.1-100 GeV to 0.2-5 MeV)
ph_ratio = COSI_LAT_Sources['ph/s_ratio'][i]

#ph_ratio = 1

# Conversion factor of flux between Fermi and COSI Bands. (0.1-100 GeV to 0.2-5 MeV)
flux_ratio = COSI_LAT_Sources['Int_flux_ratio'][i]


fsrq_names=select_fsrq()
blarray = [hops_bl1]
COSI_BAND_ALL = np.array([0,0,0,0,0,0,0,0,0,0,0,0,0])
for bl in blarray:
    characterized_flares = hop_characterizer(bl, COSI_bkg_rate=COSI_bkg_rate, COSI_Aeff=COSI_Aeff, LAT_Aeff=LAT_Aeff, ph_ratio=ph_ratio, int_flux_ratio=flux_ratio)
    COSI_BAND = np.array(characterized_flares).T
    COSI_BAND_ALL = np.vstack((COSI_BAND_ALL, COSI_BAND))

COSI_BAND_BAT_weekly_df = pd.DataFrame(COSI_BAND_ALL)
COSI_BAND_BAT_weekly_df.columns = ['Name','Duration_(s)', 'Source_Counts(ph)_(0.2-5_MeV)', 'Asymmetry', 'Coverage', 'Background_Counts', 'MDP99_(%)', 'Class', 'Photon_Fluence_(ph/cm2)_(0.2-5_MeV)', 'Average_Photon_Flux_(ph/cm2/s)_(0.2-5_MeV)','Start_Time_(MJD)','Average_Flux_of_Entire_Source_(0.2-5_MeV)', 'Peak_Flare_Flux_(0.2-5_MeV)']
COSI_BAND_BAT_weekly_df = COSI_BAND_BAT_weekly_df[COSI_BAND_BAT_weekly_df[:]['Source_Counts(ph)_(0.2-5_MeV)']!=0].reset_index(drop=True)
COSI_BAND_BAT_weekly_df = COSI_BAND_BAT_weekly_df[COSI_BAND_BAT_weekly_df[:]['Source_Counts(ph)_(0.2-5_MeV)']!='0.0'].reset_index(drop=True)
COSI_BAND_BAT_weekly_df['Source_Counts(ph)_(0.2-5_MeV)'] = COSI_BAND_BAT_weekly_df['Source_Counts(ph)_(0.2-5_MeV)'].astype(float)
COSI_BAND_BAT_weekly_df['Duration_(s)'] = COSI_BAND_BAT_weekly_df['Duration_(s)'].astype(float)
COSI_BAND_BAT_weekly_df['MDP99_(%)'] = ComputeMDP99(COSI_BAND_BAT_weekly_df['Source_Counts(ph)_(0.2-5_MeV)'].astype(float),
                                                COSI_BAND_BAT_weekly_df['Background_Counts'].astype(float))
COSI_BAND_BAT_weekly_df['Coverage'] = COSI_BAND_BAT_weekly_df['Coverage'].astype(float)
COSI_BAND_BAT_weekly_df = COSI_BAND_BAT_weekly_df[COSI_BAND_BAT_weekly_df[:]['Duration_(s)']<=2e8].reset_index(drop=True)


neworder = ['Name','Class','Average_Photon_Flux_(ph/cm2/s)_(0.2-5_MeV)','Photon_Fluence_(ph/cm2)_(0.2-5_MeV)','Source_Counts(ph)_(0.2-5_MeV)','Background_Counts','Duration_(s)','Start_Time_(MJD)','Coverage', 'MDP99_(%)','Asymmetry','Average_Flux_of_Entire_Source_(0.2-5_MeV)', 'Peak_Flare_Flux_(0.2-5_MeV)']
COSI_BAND_BAT_weekly_df
#COSI_BAND_BAT_weekly_df.to_csv(sourcename+'Analysis.csv')
# Remove blank space in sourcename

In [ ]:
initialtime,finaltime = 56400,57000
sourcename = '4FGL J1224.9+2122'
sourcearray = cadence_df[cadence_df['source_name'] == sourcename].reset_index(drop=True)
titlestring = sourcename
sourcearray = sourcearray[sourcearray['photon_flux2']!=-3333].reset_index(drop=True)
average_flux = np.nanmean(sourcearray['photon_flux2'])
sourcearray = sourcearray[sourcearray['photon_flux2']<=100*average_flux].reset_index(drop=True)
time = sourcearray['tmin']/SecsInDay + MJDREFI
photon_flux = sourcearray['photon_flux2']
errors = sourcearray['photon_flux_error2']
# Creates the lightcurve object of the source.
sourcelightcurve = LC.LightCurve(time,photon_flux,errors,name=titlestring)
i = COSI_LAT_Sources[COSI_LAT_Sources['Name']==sourcename].index[0]

selected1 = sourcelightcurve.select_by_time(initialtime,finaltime)
originalLC = sourcelightcurve
sourcelightcurve.get_bblocks()
photon_flux = selected1.flux

# Finding the initial flux threshold for detection.
maxflux = np.max(photon_flux)
minflux = np.min(photon_flux)
delta_flux = maxflux - minflux
delta_flux_percent = delta_flux * 0.3
thresholdflux = minflux + delta_flux_percent


# Initial flare detection.
selected1.get_bblocks(gamma_value=0.05)
selected1.find_hop(method='baseline', lc_edges ='add', baseline = thresholdflux)

quiescent_background,qui_err = quiescent_background_finder(sourcelightcurve=selected1,method='forward')


# Re-detecting flares with quiescent background.
sourcelightcurve = originalLC.select_by_time(initialtime,finaltime)
sourcelightcurve.get_bblocks(gamma_value=0.05)
sourcelightcurve.find_hop(method = 'baseline', lc_edges ='add',baseline = 0.8e-6)

hops_bl1 = sourcelightcurve.hops
plotit = originalLC.select_by_time(initialtime,finaltime)
plotit.plot_lc()
sourcelightcurve.plot_hop()

COSI_LAT_Sources = pd.read_csv(table, sep=",", na_filter=True)
# Effective Area of Fermi LAT for the source.
LAT_Aeff = COSI_LAT_Sources['Aeff_mean_LAT(cm2)'][i]

# Effective Area of COSI for the source.
COSI_Aeff = COSI_LAT_Sources['Aeff_mean_COSI(cm2)'][i]

# Name of Source
# Conversion factor of counts between Fermi and COSI Bands. (0.1-100 GeV to 0.2-5 MeV)
ph_ratio = COSI_LAT_Sources['ph/s_ratio'][i]

#ph_ratio = 1

# Conversion factor of flux between Fermi and COSI Bands. (0.1-100 GeV to 0.2-5 MeV)
flux_ratio = COSI_LAT_Sources['Int_flux_ratio'][i]


fsrq_names=select_fsrq()
blarray = [hops_bl1]
COSI_BAND_ALL = np.array([0,0,0,0,0,0,0,0,0,0,0,0,0])
for bl in blarray:
    characterized_flares = hop_characterizer(bl, COSI_bkg_rate=COSI_bkg_rate, COSI_Aeff=COSI_Aeff, LAT_Aeff=LAT_Aeff, ph_ratio=ph_ratio, int_flux_ratio=flux_ratio)
    COSI_BAND = np.array(characterized_flares).T
    COSI_BAND_ALL = np.vstack((COSI_BAND_ALL, COSI_BAND))

COSI_BAND_BAT_weekly_df = pd.DataFrame(COSI_BAND_ALL)
COSI_BAND_BAT_weekly_df.columns = ['Name','Duration_(s)', 'Source_Counts(ph)_(0.2-5_MeV)', 'Asymmetry', 'Coverage', 'Background_Counts', 'MDP99_(%)', 'Class', 'Photon_Fluence_(ph/cm2)_(0.2-5_MeV)', 'Average_Photon_Flux_(ph/cm2/s)_(0.2-5_MeV)','Start_Time_(MJD)','Average_Flux_of_Entire_Source_(0.2-5_MeV)', 'Peak_Flare_Flux_(0.2-5_MeV)']
COSI_BAND_BAT_weekly_df = COSI_BAND_BAT_weekly_df[COSI_BAND_BAT_weekly_df[:]['Source_Counts(ph)_(0.2-5_MeV)']!=0].reset_index(drop=True)
COSI_BAND_BAT_weekly_df = COSI_BAND_BAT_weekly_df[COSI_BAND_BAT_weekly_df[:]['Source_Counts(ph)_(0.2-5_MeV)']!='0.0'].reset_index(drop=True)
COSI_BAND_BAT_weekly_df['Source_Counts(ph)_(0.2-5_MeV)'] = COSI_BAND_BAT_weekly_df['Source_Counts(ph)_(0.2-5_MeV)'].astype(float)
COSI_BAND_BAT_weekly_df['Duration_(s)'] = COSI_BAND_BAT_weekly_df['Duration_(s)'].astype(float)
COSI_BAND_BAT_weekly_df['MDP99_(%)'] = ComputeMDP99(COSI_BAND_BAT_weekly_df['Source_Counts(ph)_(0.2-5_MeV)'].astype(float),
                                                COSI_BAND_BAT_weekly_df['Background_Counts'].astype(float))
COSI_BAND_BAT_weekly_df['Coverage'] = COSI_BAND_BAT_weekly_df['Coverage'].astype(float)
COSI_BAND_BAT_weekly_df = COSI_BAND_BAT_weekly_df[COSI_BAND_BAT_weekly_df[:]['Duration_(s)']<=2e8].reset_index(drop=True)


neworder = ['Name','Class','Average_Photon_Flux_(ph/cm2/s)_(0.2-5_MeV)','Photon_Fluence_(ph/cm2)_(0.2-5_MeV)','Source_Counts(ph)_(0.2-5_MeV)','Background_Counts','Duration_(s)','Start_Time_(MJD)','Coverage', 'MDP99_(%)','Asymmetry','Average_Flux_of_Entire_Source_(0.2-5_MeV)', 'Peak_Flare_Flux_(0.2-5_MeV)']
COSI_BAND_BAT_weekly_df
#COSI_BAND_BAT_weekly_df.to_csv(sourcename+'Analysis.csv')
# Remove blank space in sourcename

In [ ]:
initialtime,finaltime = 55000,55600
sourcename = '4FGL J1332.0-0509'
sourcearray = cadence_df[cadence_df['source_name'] == sourcename].reset_index(drop=True)
titlestring = sourcename
sourcearray = sourcearray[sourcearray['photon_flux2']!=-3333].reset_index(drop=True)
average_flux = np.nanmean(sourcearray['photon_flux2'])
sourcearray = sourcearray[sourcearray['photon_flux2']<=100*average_flux].reset_index(drop=True)
time = sourcearray['tmin']/SecsInDay + MJDREFI
photon_flux = sourcearray['photon_flux2']
errors = sourcearray['photon_flux_error2']
# Creates the lightcurve object of the source.
sourcelightcurve = LC.LightCurve(time,photon_flux,errors,name=titlestring)
i = COSI_LAT_Sources[COSI_LAT_Sources['Name']==sourcename].index[0]

selected1 = sourcelightcurve.select_by_time(initialtime,finaltime)
originalLC = sourcelightcurve
sourcelightcurve.get_bblocks()
photon_flux = selected1.flux

# Finding the initial flux threshold for detection.
maxflux = np.max(photon_flux)
minflux = np.min(photon_flux)
delta_flux = maxflux - minflux
delta_flux_percent = delta_flux * 0.3
thresholdflux = minflux + delta_flux_percent


# Initial flare detection.
selected1.get_bblocks(gamma_value=0.05)
selected1.find_hop(method='baseline', lc_edges ='add', baseline = thresholdflux)

quiescent_background,qui_err = quiescent_background_finder(sourcelightcurve=selected1,method='forward')


# Re-detecting flares with quiescent background.
sourcelightcurve = originalLC.select_by_time(initialtime,finaltime)
sourcelightcurve.get_bblocks(gamma_value=0.05)
sourcelightcurve.find_hop(method = 'baseline', lc_edges ='add',baseline = 0.7e-6)

hops_bl1 = sourcelightcurve.hops
plotit = originalLC.select_by_time(initialtime,finaltime)
plotit.plot_lc()
sourcelightcurve.plot_hop()

COSI_LAT_Sources = pd.read_csv(table, sep=",", na_filter=True)
# Effective Area of Fermi LAT for the source.
LAT_Aeff = COSI_LAT_Sources['Aeff_mean_LAT(cm2)'][i]

# Effective Area of COSI for the source.
COSI_Aeff = COSI_LAT_Sources['Aeff_mean_COSI(cm2)'][i]

# Name of Source
# Conversion factor of counts between Fermi and COSI Bands. (0.1-100 GeV to 0.2-5 MeV)
ph_ratio = COSI_LAT_Sources['ph/s_ratio'][i]

#ph_ratio = 1

# Conversion factor of flux between Fermi and COSI Bands. (0.1-100 GeV to 0.2-5 MeV)
flux_ratio = COSI_LAT_Sources['Int_flux_ratio'][i]


fsrq_names=select_fsrq()
blarray = [hops_bl1]
COSI_BAND_ALL = np.array([0,0,0,0,0,0,0,0,0,0,0,0,0])
for bl in blarray:
    characterized_flares = hop_characterizer(bl, COSI_bkg_rate=COSI_bkg_rate, COSI_Aeff=COSI_Aeff, LAT_Aeff=LAT_Aeff, ph_ratio=ph_ratio, int_flux_ratio=flux_ratio)
    COSI_BAND = np.array(characterized_flares).T
    COSI_BAND_ALL = np.vstack((COSI_BAND_ALL, COSI_BAND))

COSI_BAND_BAT_weekly_df = pd.DataFrame(COSI_BAND_ALL)
COSI_BAND_BAT_weekly_df.columns = ['Name','Duration_(s)', 'Source_Counts(ph)_(0.2-5_MeV)', 'Asymmetry', 'Coverage', 'Background_Counts', 'MDP99_(%)', 'Class', 'Photon_Fluence_(ph/cm2)_(0.2-5_MeV)', 'Average_Photon_Flux_(ph/cm2/s)_(0.2-5_MeV)','Start_Time_(MJD)','Average_Flux_of_Entire_Source_(0.2-5_MeV)', 'Peak_Flare_Flux_(0.2-5_MeV)']
COSI_BAND_BAT_weekly_df = COSI_BAND_BAT_weekly_df[COSI_BAND_BAT_weekly_df[:]['Source_Counts(ph)_(0.2-5_MeV)']!=0].reset_index(drop=True)
COSI_BAND_BAT_weekly_df = COSI_BAND_BAT_weekly_df[COSI_BAND_BAT_weekly_df[:]['Source_Counts(ph)_(0.2-5_MeV)']!='0.0'].reset_index(drop=True)
COSI_BAND_BAT_weekly_df['Source_Counts(ph)_(0.2-5_MeV)'] = COSI_BAND_BAT_weekly_df['Source_Counts(ph)_(0.2-5_MeV)'].astype(float)
COSI_BAND_BAT_weekly_df['Duration_(s)'] = COSI_BAND_BAT_weekly_df['Duration_(s)'].astype(float)
COSI_BAND_BAT_weekly_df['MDP99_(%)'] = ComputeMDP99(COSI_BAND_BAT_weekly_df['Source_Counts(ph)_(0.2-5_MeV)'].astype(float),
                                                COSI_BAND_BAT_weekly_df['Background_Counts'].astype(float))
COSI_BAND_BAT_weekly_df['Coverage'] = COSI_BAND_BAT_weekly_df['Coverage'].astype(float)
COSI_BAND_BAT_weekly_df = COSI_BAND_BAT_weekly_df[COSI_BAND_BAT_weekly_df[:]['Duration_(s)']<=2e8].reset_index(drop=True)


neworder = ['Name','Class','Average_Photon_Flux_(ph/cm2/s)_(0.2-5_MeV)','Photon_Fluence_(ph/cm2)_(0.2-5_MeV)','Source_Counts(ph)_(0.2-5_MeV)','Background_Counts','Duration_(s)','Start_Time_(MJD)','Coverage', 'MDP99_(%)','Asymmetry','Average_Flux_of_Entire_Source_(0.2-5_MeV)', 'Peak_Flare_Flux_(0.2-5_MeV)']
COSI_BAND_BAT_weekly_df
#COSI_BAND_BAT_weekly_df.to_csv(sourcename+'Analysis.csv')
# Remove blank space in sourcename

In [ ]:
initialtime,finaltime = 55000,59000
sourcename = '4FGL J1512.8-0906'
sourcearray = cadence_df[cadence_df['source_name'] == sourcename].reset_index(drop=True)
titlestring = sourcename
sourcearray = sourcearray[sourcearray['photon_flux2']!=-3333].reset_index(drop=True)
average_flux = np.nanmean(sourcearray['photon_flux2'])
sourcearray = sourcearray[sourcearray['photon_flux2']<=100*average_flux].reset_index(drop=True)
time = sourcearray['tmin']/SecsInDay + MJDREFI
photon_flux = sourcearray['photon_flux2']
errors = sourcearray['photon_flux_error2']
# Creates the lightcurve object of the source.
sourcelightcurve = LC.LightCurve(time,photon_flux,errors,name=titlestring)
i = COSI_LAT_Sources[COSI_LAT_Sources['Name']==sourcename].index[0]

selected1 = sourcelightcurve.select_by_time(initialtime,finaltime)
originalLC = sourcelightcurve
sourcelightcurve.get_bblocks()
photon_flux = selected1.flux

# Finding the initial flux threshold for detection.
maxflux = np.max(photon_flux)
minflux = np.min(photon_flux)
delta_flux = maxflux - minflux
delta_flux_percent = delta_flux * 0.3
thresholdflux = minflux + delta_flux_percent


# Initial flare detection.
selected1.get_bblocks(gamma_value=0.05)
selected1.find_hop(method='baseline', lc_edges ='add', baseline = thresholdflux)

quiescent_background,qui_err = quiescent_background_finder(sourcelightcurve=selected1,method='forward')


# Re-detecting flares with quiescent background.
sourcelightcurve = originalLC.select_by_time(initialtime,finaltime)
sourcelightcurve.get_bblocks(gamma_value=0.05)
sourcelightcurve.find_hop(method = 'baseline', lc_edges ='add',baseline = 1.5e-6)

hops_bl1 = sourcelightcurve.hops
plotit = originalLC.select_by_time(initialtime,finaltime)
plotit.plot_lc()
sourcelightcurve.plot_hop()

COSI_LAT_Sources = pd.read_csv(table, sep=",", na_filter=True)
# Effective Area of Fermi LAT for the source.
LAT_Aeff = COSI_LAT_Sources['Aeff_mean_LAT(cm2)'][i]

# Effective Area of COSI for the source.
COSI_Aeff = COSI_LAT_Sources['Aeff_mean_COSI(cm2)'][i]

# Name of Source
# Conversion factor of counts between Fermi and COSI Bands. (0.1-100 GeV to 0.2-5 MeV)
ph_ratio = COSI_LAT_Sources['ph/s_ratio'][i]

#ph_ratio = 1

# Conversion factor of flux between Fermi and COSI Bands. (0.1-100 GeV to 0.2-5 MeV)
flux_ratio = COSI_LAT_Sources['Int_flux_ratio'][i]


fsrq_names=select_fsrq()
blarray = [hops_bl1]
COSI_BAND_ALL = np.array([0,0,0,0,0,0,0,0,0,0,0,0,0])
for bl in blarray:
    characterized_flares = hop_characterizer(bl, COSI_bkg_rate=COSI_bkg_rate, COSI_Aeff=COSI_Aeff, LAT_Aeff=LAT_Aeff, ph_ratio=ph_ratio, int_flux_ratio=flux_ratio)
    COSI_BAND = np.array(characterized_flares).T
    COSI_BAND_ALL = np.vstack((COSI_BAND_ALL, COSI_BAND))

COSI_BAND_BAT_weekly_df = pd.DataFrame(COSI_BAND_ALL)
COSI_BAND_BAT_weekly_df.columns = ['Name','Duration_(s)', 'Source_Counts(ph)_(0.2-5_MeV)', 'Asymmetry', 'Coverage', 'Background_Counts', 'MDP99_(%)', 'Class', 'Photon_Fluence_(ph/cm2)_(0.2-5_MeV)', 'Average_Photon_Flux_(ph/cm2/s)_(0.2-5_MeV)','Start_Time_(MJD)','Average_Flux_of_Entire_Source_(0.2-5_MeV)', 'Peak_Flare_Flux_(0.2-5_MeV)']
COSI_BAND_BAT_weekly_df = COSI_BAND_BAT_weekly_df[COSI_BAND_BAT_weekly_df[:]['Source_Counts(ph)_(0.2-5_MeV)']!=0].reset_index(drop=True)
COSI_BAND_BAT_weekly_df = COSI_BAND_BAT_weekly_df[COSI_BAND_BAT_weekly_df[:]['Source_Counts(ph)_(0.2-5_MeV)']!='0.0'].reset_index(drop=True)
COSI_BAND_BAT_weekly_df['Source_Counts(ph)_(0.2-5_MeV)'] = COSI_BAND_BAT_weekly_df['Source_Counts(ph)_(0.2-5_MeV)'].astype(float)
COSI_BAND_BAT_weekly_df['Duration_(s)'] = COSI_BAND_BAT_weekly_df['Duration_(s)'].astype(float)
COSI_BAND_BAT_weekly_df['MDP99_(%)'] = ComputeMDP99(COSI_BAND_BAT_weekly_df['Source_Counts(ph)_(0.2-5_MeV)'].astype(float),
                                                COSI_BAND_BAT_weekly_df['Background_Counts'].astype(float))
COSI_BAND_BAT_weekly_df['Coverage'] = COSI_BAND_BAT_weekly_df['Coverage'].astype(float)
COSI_BAND_BAT_weekly_df = COSI_BAND_BAT_weekly_df[COSI_BAND_BAT_weekly_df[:]['Duration_(s)']<=2e8].reset_index(drop=True)


neworder = ['Name','Class','Average_Photon_Flux_(ph/cm2/s)_(0.2-5_MeV)','Photon_Fluence_(ph/cm2)_(0.2-5_MeV)','Source_Counts(ph)_(0.2-5_MeV)','Background_Counts','Duration_(s)','Start_Time_(MJD)','Coverage', 'MDP99_(%)','Asymmetry','Average_Flux_of_Entire_Source_(0.2-5_MeV)', 'Peak_Flare_Flux_(0.2-5_MeV)']
COSI_BAND_BAT_weekly_df
#COSI_BAND_BAT_weekly_df.to_csv(sourcename+'Analysis.csv')
# Remove blank space in sourcename

In [ ]:

initialtime,finaltime = 55500,56000
sourcename = '4FGL J1522.1+3144'
sourcearray = cadence_df[cadence_df['source_name'] == sourcename].reset_index(drop=True)
titlestring = sourcename
sourcearray = sourcearray[sourcearray['photon_flux2']!=-3333].reset_index(drop=True)
average_flux = np.nanmean(sourcearray['photon_flux2'])
sourcearray = sourcearray[sourcearray['photon_flux2']<=100*average_flux].reset_index(drop=True)
time = sourcearray['tmin']/SecsInDay + MJDREFI
photon_flux = sourcearray['photon_flux2']
errors = sourcearray['photon_flux_error2']
# Creates the lightcurve object of the source.
sourcelightcurve = LC.LightCurve(time,photon_flux,errors,name=titlestring)
i = COSI_LAT_Sources[COSI_LAT_Sources['Name']==sourcename].index[0]

selected1 = sourcelightcurve.select_by_time(initialtime,finaltime)
originalLC = sourcelightcurve
sourcelightcurve.get_bblocks()
photon_flux = selected1.flux

# Finding the initial flux threshold for detection.
maxflux = np.max(photon_flux)
minflux = np.min(photon_flux)
delta_flux = maxflux - minflux
delta_flux_percent = delta_flux * 0.3
thresholdflux = minflux + delta_flux_percent


# Initial flare detection.
selected1.get_bblocks(gamma_value=0.05)
selected1.find_hop(method='baseline', lc_edges ='add', baseline = thresholdflux)

quiescent_background,qui_err = quiescent_background_finder(sourcelightcurve=selected1,method='forward')


# Re-detecting flares with quiescent background.
sourcelightcurve = originalLC.select_by_time(initialtime,finaltime)
sourcelightcurve.get_bblocks(gamma_value=0.05)
sourcelightcurve.find_hop(method = 'baseline', lc_edges ='add',baseline = 0.5e-6)

hops_bl1 = sourcelightcurve.hops
plotit = originalLC.select_by_time(initialtime,finaltime)
plotit.plot_lc()
sourcelightcurve.plot_hop()

COSI_LAT_Sources = pd.read_csv(table, sep=",", na_filter=True)
# Effective Area of Fermi LAT for the source.
LAT_Aeff = COSI_LAT_Sources['Aeff_mean_LAT(cm2)'][i]

# Effective Area of COSI for the source.
COSI_Aeff = COSI_LAT_Sources['Aeff_mean_COSI(cm2)'][i]

# Name of Source
# Conversion factor of counts between Fermi and COSI Bands. (0.1-100 GeV to 0.2-5 MeV)
ph_ratio = COSI_LAT_Sources['ph/s_ratio'][i]

#ph_ratio = 1

# Conversion factor of flux between Fermi and COSI Bands. (0.1-100 GeV to 0.2-5 MeV)
flux_ratio = COSI_LAT_Sources['Int_flux_ratio'][i]


fsrq_names=select_fsrq()
blarray = [hops_bl1]
COSI_BAND_ALL = np.array([0,0,0,0,0,0,0,0,0,0,0,0,0])
for bl in blarray:
    characterized_flares = hop_characterizer(bl, COSI_bkg_rate=COSI_bkg_rate, COSI_Aeff=COSI_Aeff, LAT_Aeff=LAT_Aeff, ph_ratio=ph_ratio, int_flux_ratio=flux_ratio)
    COSI_BAND = np.array(characterized_flares).T
    COSI_BAND_ALL = np.vstack((COSI_BAND_ALL, COSI_BAND))

COSI_BAND_BAT_weekly_df = pd.DataFrame(COSI_BAND_ALL)
COSI_BAND_BAT_weekly_df.columns = ['Name','Duration_(s)', 'Source_Counts(ph)_(0.2-5_MeV)', 'Asymmetry', 'Coverage', 'Background_Counts', 'MDP99_(%)', 'Class', 'Photon_Fluence_(ph/cm2)_(0.2-5_MeV)', 'Average_Photon_Flux_(ph/cm2/s)_(0.2-5_MeV)','Start_Time_(MJD)','Average_Flux_of_Entire_Source_(0.2-5_MeV)', 'Peak_Flare_Flux_(0.2-5_MeV)']
COSI_BAND_BAT_weekly_df = COSI_BAND_BAT_weekly_df[COSI_BAND_BAT_weekly_df[:]['Source_Counts(ph)_(0.2-5_MeV)']!=0].reset_index(drop=True)
COSI_BAND_BAT_weekly_df = COSI_BAND_BAT_weekly_df[COSI_BAND_BAT_weekly_df[:]['Source_Counts(ph)_(0.2-5_MeV)']!='0.0'].reset_index(drop=True)
COSI_BAND_BAT_weekly_df['Source_Counts(ph)_(0.2-5_MeV)'] = COSI_BAND_BAT_weekly_df['Source_Counts(ph)_(0.2-5_MeV)'].astype(float)
COSI_BAND_BAT_weekly_df['Duration_(s)'] = COSI_BAND_BAT_weekly_df['Duration_(s)'].astype(float)
COSI_BAND_BAT_weekly_df['MDP99_(%)'] = ComputeMDP99(COSI_BAND_BAT_weekly_df['Source_Counts(ph)_(0.2-5_MeV)'].astype(float),
                                                COSI_BAND_BAT_weekly_df['Background_Counts'].astype(float))
COSI_BAND_BAT_weekly_df['Coverage'] = COSI_BAND_BAT_weekly_df['Coverage'].astype(float)
COSI_BAND_BAT_weekly_df = COSI_BAND_BAT_weekly_df[COSI_BAND_BAT_weekly_df[:]['Duration_(s)']<=2e8].reset_index(drop=True)


neworder = ['Name','Class','Average_Photon_Flux_(ph/cm2/s)_(0.2-5_MeV)','Photon_Fluence_(ph/cm2)_(0.2-5_MeV)','Source_Counts(ph)_(0.2-5_MeV)','Background_Counts','Duration_(s)','Start_Time_(MJD)','Coverage', 'MDP99_(%)','Asymmetry','Average_Flux_of_Entire_Source_(0.2-5_MeV)', 'Peak_Flare_Flux_(0.2-5_MeV)']
COSI_BAND_BAT_weekly_df
#COSI_BAND_BAT_weekly_df.to_csv(sourcename+'Analysis.csv')
# Remove blank space in sourcename

In [ ]:
initialtime,finaltime = 59300,60500
sourcename = '4FGL J1642.9+3948'
sourcearray = cadence_df[cadence_df['source_name'] == sourcename].reset_index(drop=True)
titlestring = sourcename
sourcearray = sourcearray[sourcearray['photon_flux2']!=-3333].reset_index(drop=True)
average_flux = np.nanmean(sourcearray['photon_flux2'])
sourcearray = sourcearray[sourcearray['photon_flux2']<=100*average_flux].reset_index(drop=True)
time = sourcearray['tmin']/SecsInDay + MJDREFI
photon_flux = sourcearray['photon_flux2']
errors = sourcearray['photon_flux_error2']
# Creates the lightcurve object of the source.
sourcelightcurve = LC.LightCurve(time,photon_flux,errors,name=titlestring)
i = COSI_LAT_Sources[COSI_LAT_Sources['Name']==sourcename].index[0]

selected1 = sourcelightcurve.select_by_time(initialtime,finaltime)
originalLC = sourcelightcurve
sourcelightcurve.get_bblocks()
photon_flux = selected1.flux

# Finding the initial flux threshold for detection.
maxflux = np.max(photon_flux)
minflux = np.min(photon_flux)
delta_flux = maxflux - minflux
delta_flux_percent = delta_flux * 0.3
thresholdflux = minflux + delta_flux_percent


# Initial flare detection.
selected1.get_bblocks(gamma_value=0.05)
selected1.find_hop(method='baseline', lc_edges ='add', baseline = thresholdflux)

quiescent_background,qui_err = quiescent_background_finder(sourcelightcurve=selected1,method='forward')


# Re-detecting flares with quiescent background.
sourcelightcurve = originalLC.select_by_time(initialtime,finaltime)
sourcelightcurve.get_bblocks(gamma_value=0.05)
sourcelightcurve.find_hop(method = 'baseline', lc_edges ='add',baseline = 0.5e-6)

hops_bl1 = sourcelightcurve.hops
plotit = originalLC.select_by_time(initialtime,finaltime)
plotit.plot_lc()
sourcelightcurve.plot_hop()

COSI_LAT_Sources = pd.read_csv(table, sep=",", na_filter=True)
# Effective Area of Fermi LAT for the source.
LAT_Aeff = COSI_LAT_Sources['Aeff_mean_LAT(cm2)'][i]

# Effective Area of COSI for the source.
COSI_Aeff = COSI_LAT_Sources['Aeff_mean_COSI(cm2)'][i]

# Name of Source
# Conversion factor of counts between Fermi and COSI Bands. (0.1-100 GeV to 0.2-5 MeV)
ph_ratio = COSI_LAT_Sources['ph/s_ratio'][i]

#ph_ratio = 1

# Conversion factor of flux between Fermi and COSI Bands. (0.1-100 GeV to 0.2-5 MeV)
flux_ratio = COSI_LAT_Sources['Int_flux_ratio'][i]


fsrq_names=select_fsrq()
blarray = [hops_bl1]
COSI_BAND_ALL = np.array([0,0,0,0,0,0,0,0,0,0,0,0,0])
for bl in blarray:
    characterized_flares = hop_characterizer(bl, COSI_bkg_rate=COSI_bkg_rate, COSI_Aeff=COSI_Aeff, LAT_Aeff=LAT_Aeff, ph_ratio=ph_ratio, int_flux_ratio=flux_ratio)
    COSI_BAND = np.array(characterized_flares).T
    COSI_BAND_ALL = np.vstack((COSI_BAND_ALL, COSI_BAND))

COSI_BAND_BAT_weekly_df = pd.DataFrame(COSI_BAND_ALL)
COSI_BAND_BAT_weekly_df.columns = ['Name','Duration_(s)', 'Source_Counts(ph)_(0.2-5_MeV)', 'Asymmetry', 'Coverage', 'Background_Counts', 'MDP99_(%)', 'Class', 'Photon_Fluence_(ph/cm2)_(0.2-5_MeV)', 'Average_Photon_Flux_(ph/cm2/s)_(0.2-5_MeV)','Start_Time_(MJD)','Average_Flux_of_Entire_Source_(0.2-5_MeV)', 'Peak_Flare_Flux_(0.2-5_MeV)']
COSI_BAND_BAT_weekly_df = COSI_BAND_BAT_weekly_df[COSI_BAND_BAT_weekly_df[:]['Source_Counts(ph)_(0.2-5_MeV)']!=0].reset_index(drop=True)
COSI_BAND_BAT_weekly_df = COSI_BAND_BAT_weekly_df[COSI_BAND_BAT_weekly_df[:]['Source_Counts(ph)_(0.2-5_MeV)']!='0.0'].reset_index(drop=True)
COSI_BAND_BAT_weekly_df['Source_Counts(ph)_(0.2-5_MeV)'] = COSI_BAND_BAT_weekly_df['Source_Counts(ph)_(0.2-5_MeV)'].astype(float)
COSI_BAND_BAT_weekly_df['Duration_(s)'] = COSI_BAND_BAT_weekly_df['Duration_(s)'].astype(float)
COSI_BAND_BAT_weekly_df['MDP99_(%)'] = ComputeMDP99(COSI_BAND_BAT_weekly_df['Source_Counts(ph)_(0.2-5_MeV)'].astype(float),
                                                COSI_BAND_BAT_weekly_df['Background_Counts'].astype(float))
COSI_BAND_BAT_weekly_df['Coverage'] = COSI_BAND_BAT_weekly_df['Coverage'].astype(float)
COSI_BAND_BAT_weekly_df = COSI_BAND_BAT_weekly_df[COSI_BAND_BAT_weekly_df[:]['Duration_(s)']<=2e8].reset_index(drop=True)


neworder = ['Name','Class','Average_Photon_Flux_(ph/cm2/s)_(0.2-5_MeV)','Photon_Fluence_(ph/cm2)_(0.2-5_MeV)','Source_Counts(ph)_(0.2-5_MeV)','Background_Counts','Duration_(s)','Start_Time_(MJD)','Coverage', 'MDP99_(%)','Asymmetry','Average_Flux_of_Entire_Source_(0.2-5_MeV)', 'Peak_Flare_Flux_(0.2-5_MeV)']
COSI_BAND_BAT_weekly_df
#COSI_BAND_BAT_weekly_df.to_csv(sourcename+'Analysis.csv')
# Remove blank space in sourcename

In [ ]:
initialtime,finaltime = 54500,55000
sourcename = '4FGL J1740.5+5211'
sourcearray = cadence_df[cadence_df['source_name'] == sourcename].reset_index(drop=True)
titlestring = sourcename
sourcearray = sourcearray[sourcearray['photon_flux2']!=-3333].reset_index(drop=True)
average_flux = np.nanmean(sourcearray['photon_flux2'])
sourcearray = sourcearray[sourcearray['photon_flux2']<=100*average_flux].reset_index(drop=True)
time = sourcearray['tmin']/SecsInDay + MJDREFI
photon_flux = sourcearray['photon_flux2']
errors = sourcearray['photon_flux_error2']
# Creates the lightcurve object of the source.
sourcelightcurve = LC.LightCurve(time,photon_flux,errors,name=titlestring)
i = COSI_LAT_Sources[COSI_LAT_Sources['Name']==sourcename].index[0]

selected1 = sourcelightcurve.select_by_time(initialtime,finaltime)
originalLC = sourcelightcurve
sourcelightcurve.get_bblocks()
photon_flux = selected1.flux

# Finding the initial flux threshold for detection.
maxflux = np.max(photon_flux)
minflux = np.min(photon_flux)
delta_flux = maxflux - minflux
delta_flux_percent = delta_flux * 0.3
thresholdflux = minflux + delta_flux_percent


# Initial flare detection.
selected1.get_bblocks(gamma_value=0.05)
selected1.find_hop(method='baseline', lc_edges ='add', baseline = thresholdflux)

quiescent_background,qui_err = quiescent_background_finder(sourcelightcurve=selected1,method='forward')


# Re-detecting flares with quiescent background.
sourcelightcurve = originalLC.select_by_time(initialtime,finaltime)
sourcelightcurve.get_bblocks(gamma_value=0.05)
sourcelightcurve.find_hop(method = 'baseline', lc_edges ='add',baseline = 0.1e-6)

hops_bl1 = sourcelightcurve.hops
plotit = originalLC.select_by_time(initialtime,finaltime)
plotit.plot_lc()
sourcelightcurve.plot_hop()

COSI_LAT_Sources = pd.read_csv(table, sep=",", na_filter=True)
# Effective Area of Fermi LAT for the source.
LAT_Aeff = COSI_LAT_Sources['Aeff_mean_LAT(cm2)'][i]

# Effective Area of COSI for the source.
COSI_Aeff = COSI_LAT_Sources['Aeff_mean_COSI(cm2)'][i]

# Name of Source
# Conversion factor of counts between Fermi and COSI Bands. (0.1-100 GeV to 0.2-5 MeV)
ph_ratio = COSI_LAT_Sources['ph/s_ratio'][i]

#ph_ratio = 1

# Conversion factor of flux between Fermi and COSI Bands. (0.1-100 GeV to 0.2-5 MeV)
flux_ratio = COSI_LAT_Sources['Int_flux_ratio'][i]


fsrq_names=select_fsrq()
blarray = [hops_bl1]
COSI_BAND_ALL = np.array([0,0,0,0,0,0,0,0,0,0,0,0,0])
for bl in blarray:
    characterized_flares = hop_characterizer(bl, COSI_bkg_rate=COSI_bkg_rate, COSI_Aeff=COSI_Aeff, LAT_Aeff=LAT_Aeff, ph_ratio=ph_ratio, int_flux_ratio=flux_ratio)
    COSI_BAND = np.array(characterized_flares).T
    COSI_BAND_ALL = np.vstack((COSI_BAND_ALL, COSI_BAND))

COSI_BAND_BAT_weekly_df = pd.DataFrame(COSI_BAND_ALL)
COSI_BAND_BAT_weekly_df.columns = ['Name','Duration_(s)', 'Source_Counts(ph)_(0.2-5_MeV)', 'Asymmetry', 'Coverage', 'Background_Counts', 'MDP99_(%)', 'Class', 'Photon_Fluence_(ph/cm2)_(0.2-5_MeV)', 'Average_Photon_Flux_(ph/cm2/s)_(0.2-5_MeV)','Start_Time_(MJD)','Average_Flux_of_Entire_Source_(0.2-5_MeV)', 'Peak_Flare_Flux_(0.2-5_MeV)']
COSI_BAND_BAT_weekly_df = COSI_BAND_BAT_weekly_df[COSI_BAND_BAT_weekly_df[:]['Source_Counts(ph)_(0.2-5_MeV)']!=0].reset_index(drop=True)
COSI_BAND_BAT_weekly_df = COSI_BAND_BAT_weekly_df[COSI_BAND_BAT_weekly_df[:]['Source_Counts(ph)_(0.2-5_MeV)']!='0.0'].reset_index(drop=True)
COSI_BAND_BAT_weekly_df['Source_Counts(ph)_(0.2-5_MeV)'] = COSI_BAND_BAT_weekly_df['Source_Counts(ph)_(0.2-5_MeV)'].astype(float)
COSI_BAND_BAT_weekly_df['Duration_(s)'] = COSI_BAND_BAT_weekly_df['Duration_(s)'].astype(float)
COSI_BAND_BAT_weekly_df['MDP99_(%)'] = ComputeMDP99(COSI_BAND_BAT_weekly_df['Source_Counts(ph)_(0.2-5_MeV)'].astype(float),
                                                COSI_BAND_BAT_weekly_df['Background_Counts'].astype(float))
COSI_BAND_BAT_weekly_df['Coverage'] = COSI_BAND_BAT_weekly_df['Coverage'].astype(float)
COSI_BAND_BAT_weekly_df = COSI_BAND_BAT_weekly_df[COSI_BAND_BAT_weekly_df[:]['Duration_(s)']<=2e8].reset_index(drop=True)


neworder = ['Name','Class','Average_Photon_Flux_(ph/cm2/s)_(0.2-5_MeV)','Photon_Fluence_(ph/cm2)_(0.2-5_MeV)','Source_Counts(ph)_(0.2-5_MeV)','Background_Counts','Duration_(s)','Start_Time_(MJD)','Coverage', 'MDP99_(%)','Asymmetry','Average_Flux_of_Entire_Source_(0.2-5_MeV)', 'Peak_Flare_Flux_(0.2-5_MeV)']
COSI_BAND_BAT_weekly_df
#COSI_BAND_BAT_weekly_df.to_csv(sourcename+'Analysis.csv')
# Remove blank space in sourcename

In [ ]:
initialtime,finaltime = 57000,57500
sourcename = '4FGL J1740.5+5211'
sourcearray = cadence_df[cadence_df['source_name'] == sourcename].reset_index(drop=True)
titlestring = sourcename
sourcearray = sourcearray[sourcearray['photon_flux2']!=-3333].reset_index(drop=True)
average_flux = np.nanmean(sourcearray['photon_flux2'])
sourcearray = sourcearray[sourcearray['photon_flux2']<=100*average_flux].reset_index(drop=True)
time = sourcearray['tmin']/SecsInDay + MJDREFI
photon_flux = sourcearray['photon_flux2']
errors = sourcearray['photon_flux_error2']
# Creates the lightcurve object of the source.
sourcelightcurve = LC.LightCurve(time,photon_flux,errors,name=titlestring)
i = COSI_LAT_Sources[COSI_LAT_Sources['Name']==sourcename].index[0]

selected1 = sourcelightcurve.select_by_time(initialtime,finaltime)
originalLC = sourcelightcurve
sourcelightcurve.get_bblocks()
photon_flux = selected1.flux

# Finding the initial flux threshold for detection.
maxflux = np.max(photon_flux)
minflux = np.min(photon_flux)
delta_flux = maxflux - minflux
delta_flux_percent = delta_flux * 0.3
thresholdflux = minflux + delta_flux_percent


# Initial flare detection.
selected1.get_bblocks(gamma_value=0.05)
selected1.find_hop(method='baseline', lc_edges ='add', baseline = thresholdflux)

quiescent_background,qui_err = quiescent_background_finder(sourcelightcurve=selected1,method='forward')


# Re-detecting flares with quiescent background.
sourcelightcurve = originalLC.select_by_time(initialtime,finaltime)
sourcelightcurve.get_bblocks(gamma_value=0.05)
sourcelightcurve.find_hop(method = 'baseline', lc_edges ='add',baseline = 0.3e-6)

hops_bl1 = sourcelightcurve.hops
plotit = originalLC.select_by_time(initialtime,finaltime)
plotit.plot_lc()
sourcelightcurve.plot_hop()

COSI_LAT_Sources = pd.read_csv(table, sep=",", na_filter=True)
# Effective Area of Fermi LAT for the source.
LAT_Aeff = COSI_LAT_Sources['Aeff_mean_LAT(cm2)'][i]

# Effective Area of COSI for the source.
COSI_Aeff = COSI_LAT_Sources['Aeff_mean_COSI(cm2)'][i]

# Name of Source
# Conversion factor of counts between Fermi and COSI Bands. (0.1-100 GeV to 0.2-5 MeV)
ph_ratio = COSI_LAT_Sources['ph/s_ratio'][i]

#ph_ratio = 1

# Conversion factor of flux between Fermi and COSI Bands. (0.1-100 GeV to 0.2-5 MeV)
flux_ratio = COSI_LAT_Sources['Int_flux_ratio'][i]


fsrq_names=select_fsrq()
blarray = [hops_bl1]
COSI_BAND_ALL = np.array([0,0,0,0,0,0,0,0,0,0,0,0,0])
for bl in blarray:
    characterized_flares = hop_characterizer(bl, COSI_bkg_rate=COSI_bkg_rate, COSI_Aeff=COSI_Aeff, LAT_Aeff=LAT_Aeff, ph_ratio=ph_ratio, int_flux_ratio=flux_ratio)
    COSI_BAND = np.array(characterized_flares).T
    COSI_BAND_ALL = np.vstack((COSI_BAND_ALL, COSI_BAND))

COSI_BAND_BAT_weekly_df = pd.DataFrame(COSI_BAND_ALL)
COSI_BAND_BAT_weekly_df.columns = ['Name','Duration_(s)', 'Source_Counts(ph)_(0.2-5_MeV)', 'Asymmetry', 'Coverage', 'Background_Counts', 'MDP99_(%)', 'Class', 'Photon_Fluence_(ph/cm2)_(0.2-5_MeV)', 'Average_Photon_Flux_(ph/cm2/s)_(0.2-5_MeV)','Start_Time_(MJD)','Average_Flux_of_Entire_Source_(0.2-5_MeV)', 'Peak_Flare_Flux_(0.2-5_MeV)']
COSI_BAND_BAT_weekly_df = COSI_BAND_BAT_weekly_df[COSI_BAND_BAT_weekly_df[:]['Source_Counts(ph)_(0.2-5_MeV)']!=0].reset_index(drop=True)
COSI_BAND_BAT_weekly_df = COSI_BAND_BAT_weekly_df[COSI_BAND_BAT_weekly_df[:]['Source_Counts(ph)_(0.2-5_MeV)']!='0.0'].reset_index(drop=True)
COSI_BAND_BAT_weekly_df['Source_Counts(ph)_(0.2-5_MeV)'] = COSI_BAND_BAT_weekly_df['Source_Counts(ph)_(0.2-5_MeV)'].astype(float)
COSI_BAND_BAT_weekly_df['Duration_(s)'] = COSI_BAND_BAT_weekly_df['Duration_(s)'].astype(float)
COSI_BAND_BAT_weekly_df['MDP99_(%)'] = ComputeMDP99(COSI_BAND_BAT_weekly_df['Source_Counts(ph)_(0.2-5_MeV)'].astype(float),
                                                COSI_BAND_BAT_weekly_df['Background_Counts'].astype(float))
COSI_BAND_BAT_weekly_df['Coverage'] = COSI_BAND_BAT_weekly_df['Coverage'].astype(float)
COSI_BAND_BAT_weekly_df = COSI_BAND_BAT_weekly_df[COSI_BAND_BAT_weekly_df[:]['Duration_(s)']<=2e8].reset_index(drop=True)


neworder = ['Name','Class','Average_Photon_Flux_(ph/cm2/s)_(0.2-5_MeV)','Photon_Fluence_(ph/cm2)_(0.2-5_MeV)','Source_Counts(ph)_(0.2-5_MeV)','Background_Counts','Duration_(s)','Start_Time_(MJD)','Coverage', 'MDP99_(%)','Asymmetry','Average_Flux_of_Entire_Source_(0.2-5_MeV)', 'Peak_Flare_Flux_(0.2-5_MeV)']
COSI_BAND_BAT_weekly_df
#COSI_BAND_BAT_weekly_df.to_csv(sourcename+'Analysis.csv')
# Remove blank space in sourcename

In [ ]:
initialtime,finaltime = 58900,60000
sourcename = '4FGL J1800.6+7828'
sourcearray = cadence_df[cadence_df['source_name'] == sourcename].reset_index(drop=True)
titlestring = sourcename
sourcearray = sourcearray[sourcearray['photon_flux2']!=-3333].reset_index(drop=True)
average_flux = np.nanmean(sourcearray['photon_flux2'])
sourcearray = sourcearray[sourcearray['photon_flux2']<=100*average_flux].reset_index(drop=True)
time = sourcearray['tmin']/SecsInDay + MJDREFI
photon_flux = sourcearray['photon_flux2']
errors = sourcearray['photon_flux_error2']
# Creates the lightcurve object of the source.
sourcelightcurve = LC.LightCurve(time,photon_flux,errors,name=titlestring)
i = COSI_LAT_Sources[COSI_LAT_Sources['Name']==sourcename].index[0]

selected1 = sourcelightcurve.select_by_time(initialtime,finaltime)
originalLC = sourcelightcurve
sourcelightcurve.get_bblocks()
photon_flux = selected1.flux

# Finding the initial flux threshold for detection.
maxflux = np.max(photon_flux)
minflux = np.min(photon_flux)
delta_flux = maxflux - minflux
delta_flux_percent = delta_flux * 0.3
thresholdflux = minflux + delta_flux_percent


# Initial flare detection.
selected1.get_bblocks(gamma_value=0.05)
selected1.find_hop(method='baseline', lc_edges ='add', baseline = thresholdflux)

quiescent_background,qui_err = quiescent_background_finder(sourcelightcurve=selected1,method='forward')


# Re-detecting flares with quiescent background.
sourcelightcurve = originalLC.select_by_time(initialtime,finaltime)
sourcelightcurve.get_bblocks(gamma_value=0.05)
sourcelightcurve.find_hop(method = 'baseline', lc_edges ='add',baseline = 0.5e-6)

hops_bl1 = sourcelightcurve.hops
plotit = originalLC.select_by_time(initialtime,finaltime)
plotit.plot_lc()
sourcelightcurve.plot_hop()

COSI_LAT_Sources = pd.read_csv(table, sep=",", na_filter=True)
# Effective Area of Fermi LAT for the source.
LAT_Aeff = COSI_LAT_Sources['Aeff_mean_LAT(cm2)'][i]

# Effective Area of COSI for the source.
COSI_Aeff = COSI_LAT_Sources['Aeff_mean_COSI(cm2)'][i]

# Name of Source
# Conversion factor of counts between Fermi and COSI Bands. (0.1-100 GeV to 0.2-5 MeV)
ph_ratio = COSI_LAT_Sources['ph/s_ratio'][i]

#ph_ratio = 1

# Conversion factor of flux between Fermi and COSI Bands. (0.1-100 GeV to 0.2-5 MeV)
flux_ratio = COSI_LAT_Sources['Int_flux_ratio'][i]


fsrq_names=select_fsrq()
blarray = [hops_bl1]
COSI_BAND_ALL = np.array([0,0,0,0,0,0,0,0,0,0,0,0,0])
for bl in blarray:
    characterized_flares = hop_characterizer(bl, COSI_bkg_rate=COSI_bkg_rate, COSI_Aeff=COSI_Aeff, LAT_Aeff=LAT_Aeff, ph_ratio=ph_ratio, int_flux_ratio=flux_ratio)
    COSI_BAND = np.array(characterized_flares).T
    COSI_BAND_ALL = np.vstack((COSI_BAND_ALL, COSI_BAND))

COSI_BAND_BAT_weekly_df = pd.DataFrame(COSI_BAND_ALL)
COSI_BAND_BAT_weekly_df.columns = ['Name','Duration_(s)', 'Source_Counts(ph)_(0.2-5_MeV)', 'Asymmetry', 'Coverage', 'Background_Counts', 'MDP99_(%)', 'Class', 'Photon_Fluence_(ph/cm2)_(0.2-5_MeV)', 'Average_Photon_Flux_(ph/cm2/s)_(0.2-5_MeV)','Start_Time_(MJD)','Average_Flux_of_Entire_Source_(0.2-5_MeV)', 'Peak_Flare_Flux_(0.2-5_MeV)']
COSI_BAND_BAT_weekly_df = COSI_BAND_BAT_weekly_df[COSI_BAND_BAT_weekly_df[:]['Source_Counts(ph)_(0.2-5_MeV)']!=0].reset_index(drop=True)
COSI_BAND_BAT_weekly_df = COSI_BAND_BAT_weekly_df[COSI_BAND_BAT_weekly_df[:]['Source_Counts(ph)_(0.2-5_MeV)']!='0.0'].reset_index(drop=True)
COSI_BAND_BAT_weekly_df['Source_Counts(ph)_(0.2-5_MeV)'] = COSI_BAND_BAT_weekly_df['Source_Counts(ph)_(0.2-5_MeV)'].astype(float)
COSI_BAND_BAT_weekly_df['Duration_(s)'] = COSI_BAND_BAT_weekly_df['Duration_(s)'].astype(float)
COSI_BAND_BAT_weekly_df['MDP99_(%)'] = ComputeMDP99(COSI_BAND_BAT_weekly_df['Source_Counts(ph)_(0.2-5_MeV)'].astype(float),
                                                COSI_BAND_BAT_weekly_df['Background_Counts'].astype(float))
COSI_BAND_BAT_weekly_df['Coverage'] = COSI_BAND_BAT_weekly_df['Coverage'].astype(float)
COSI_BAND_BAT_weekly_df = COSI_BAND_BAT_weekly_df[COSI_BAND_BAT_weekly_df[:]['Duration_(s)']<=2e8].reset_index(drop=True)


neworder = ['Name','Class','Average_Photon_Flux_(ph/cm2/s)_(0.2-5_MeV)','Photon_Fluence_(ph/cm2)_(0.2-5_MeV)','Source_Counts(ph)_(0.2-5_MeV)','Background_Counts','Duration_(s)','Start_Time_(MJD)','Coverage', 'MDP99_(%)','Asymmetry','Average_Flux_of_Entire_Source_(0.2-5_MeV)', 'Peak_Flare_Flux_(0.2-5_MeV)']
COSI_BAND_BAT_weekly_df
#COSI_BAND_BAT_weekly_df.to_csv(sourcename+'Analysis.csv')
# Remove blank space in sourcename

In [ ]:
initialtime,finaltime = 58300,59000
sourcename = '4FGL J1833.6-2103'
sourcearray = cadence_df[cadence_df['source_name'] == sourcename].reset_index(drop=True)
titlestring = sourcename
sourcearray = sourcearray[sourcearray['photon_flux2']!=-3333].reset_index(drop=True)
average_flux = np.nanmean(sourcearray['photon_flux2'])
sourcearray = sourcearray[sourcearray['photon_flux2']<=100*average_flux].reset_index(drop=True)
time = sourcearray['tmin']/SecsInDay + MJDREFI
photon_flux = sourcearray['photon_flux2']
errors = sourcearray['photon_flux_error2']
# Creates the lightcurve object of the source.
sourcelightcurve = LC.LightCurve(time,photon_flux,errors,name=titlestring)
i = COSI_LAT_Sources[COSI_LAT_Sources['Name']==sourcename].index[0]

selected1 = sourcelightcurve.select_by_time(initialtime,finaltime)
originalLC = sourcelightcurve
sourcelightcurve.get_bblocks()
photon_flux = selected1.flux

# Finding the initial flux threshold for detection.
maxflux = np.max(photon_flux)
minflux = np.min(photon_flux)
delta_flux = maxflux - minflux
delta_flux_percent = delta_flux * 0.3
thresholdflux = minflux + delta_flux_percent


# Initial flare detection.
selected1.get_bblocks(gamma_value=0.05)
selected1.find_hop(method='baseline', lc_edges ='add', baseline = thresholdflux)

quiescent_background,qui_err = quiescent_background_finder(sourcelightcurve=selected1,method='forward')


# Re-detecting flares with quiescent background.
sourcelightcurve = originalLC.select_by_time(initialtime,finaltime)
sourcelightcurve.get_bblocks(gamma_value=0.05)
sourcelightcurve.find_hop(method = 'baseline', lc_edges ='add',baseline = 0.7e-5)

hops_bl1 = sourcelightcurve.hops
plotit = originalLC.select_by_time(initialtime,finaltime)
plotit.plot_lc()
sourcelightcurve.plot_hop()

COSI_LAT_Sources = pd.read_csv(table, sep=",", na_filter=True)
# Effective Area of Fermi LAT for the source.
LAT_Aeff = COSI_LAT_Sources['Aeff_mean_LAT(cm2)'][i]

# Effective Area of COSI for the source.
COSI_Aeff = COSI_LAT_Sources['Aeff_mean_COSI(cm2)'][i]

# Name of Source
# Conversion factor of counts between Fermi and COSI Bands. (0.1-100 GeV to 0.2-5 MeV)
ph_ratio = COSI_LAT_Sources['ph/s_ratio'][i]

#ph_ratio = 1

# Conversion factor of flux between Fermi and COSI Bands. (0.1-100 GeV to 0.2-5 MeV)
flux_ratio = COSI_LAT_Sources['Int_flux_ratio'][i]


fsrq_names=select_fsrq()
blarray = [hops_bl1]
COSI_BAND_ALL = np.array([0,0,0,0,0,0,0,0,0,0,0,0,0])
for bl in blarray:
    characterized_flares = hop_characterizer(bl, COSI_bkg_rate=COSI_bkg_rate, COSI_Aeff=COSI_Aeff, LAT_Aeff=LAT_Aeff, ph_ratio=ph_ratio, int_flux_ratio=flux_ratio)
    COSI_BAND = np.array(characterized_flares).T
    COSI_BAND_ALL = np.vstack((COSI_BAND_ALL, COSI_BAND))

COSI_BAND_BAT_weekly_df = pd.DataFrame(COSI_BAND_ALL)
COSI_BAND_BAT_weekly_df.columns = ['Name','Duration_(s)', 'Source_Counts(ph)_(0.2-5_MeV)', 'Asymmetry', 'Coverage', 'Background_Counts', 'MDP99_(%)', 'Class', 'Photon_Fluence_(ph/cm2)_(0.2-5_MeV)', 'Average_Photon_Flux_(ph/cm2/s)_(0.2-5_MeV)','Start_Time_(MJD)','Average_Flux_of_Entire_Source_(0.2-5_MeV)', 'Peak_Flare_Flux_(0.2-5_MeV)']
COSI_BAND_BAT_weekly_df = COSI_BAND_BAT_weekly_df[COSI_BAND_BAT_weekly_df[:]['Source_Counts(ph)_(0.2-5_MeV)']!=0].reset_index(drop=True)
COSI_BAND_BAT_weekly_df = COSI_BAND_BAT_weekly_df[COSI_BAND_BAT_weekly_df[:]['Source_Counts(ph)_(0.2-5_MeV)']!='0.0'].reset_index(drop=True)
COSI_BAND_BAT_weekly_df['Source_Counts(ph)_(0.2-5_MeV)'] = COSI_BAND_BAT_weekly_df['Source_Counts(ph)_(0.2-5_MeV)'].astype(float)
COSI_BAND_BAT_weekly_df['Duration_(s)'] = COSI_BAND_BAT_weekly_df['Duration_(s)'].astype(float)
COSI_BAND_BAT_weekly_df['MDP99_(%)'] = ComputeMDP99(COSI_BAND_BAT_weekly_df['Source_Counts(ph)_(0.2-5_MeV)'].astype(float),
                                                COSI_BAND_BAT_weekly_df['Background_Counts'].astype(float))
COSI_BAND_BAT_weekly_df['Coverage'] = COSI_BAND_BAT_weekly_df['Coverage'].astype(float)
COSI_BAND_BAT_weekly_df = COSI_BAND_BAT_weekly_df[COSI_BAND_BAT_weekly_df[:]['Duration_(s)']<=2e8].reset_index(drop=True)


neworder = ['Name','Class','Average_Photon_Flux_(ph/cm2/s)_(0.2-5_MeV)','Photon_Fluence_(ph/cm2)_(0.2-5_MeV)','Source_Counts(ph)_(0.2-5_MeV)','Background_Counts','Duration_(s)','Start_Time_(MJD)','Coverage', 'MDP99_(%)','Asymmetry','Average_Flux_of_Entire_Source_(0.2-5_MeV)', 'Peak_Flare_Flux_(0.2-5_MeV)']
COSI_BAND_BAT_weekly_df
#COSI_BAND_BAT_weekly_df.to_csv(sourcename+'Analysis.csv')
# Remove blank space in sourcename

In [ ]:
initialtime,finaltime = 58500,59000
sourcename = '4FGL J2056.2-4714'
sourcearray = cadence_df[cadence_df['source_name'] == sourcename].reset_index(drop=True)
titlestring = sourcename
sourcearray = sourcearray[sourcearray['photon_flux2']!=-3333].reset_index(drop=True)
average_flux = np.nanmean(sourcearray['photon_flux2'])
sourcearray = sourcearray[sourcearray['photon_flux2']<=100*average_flux].reset_index(drop=True)
time = sourcearray['tmin']/SecsInDay + MJDREFI
photon_flux = sourcearray['photon_flux2']
errors = sourcearray['photon_flux_error2']
# Creates the lightcurve object of the source.
sourcelightcurve = LC.LightCurve(time,photon_flux,errors,name=titlestring)
i = COSI_LAT_Sources[COSI_LAT_Sources['Name']==sourcename].index[0]

selected1 = sourcelightcurve.select_by_time(initialtime,finaltime)
originalLC = sourcelightcurve
sourcelightcurve.get_bblocks()
photon_flux = selected1.flux

# Finding the initial flux threshold for detection.
maxflux = np.max(photon_flux)
minflux = np.min(photon_flux)
delta_flux = maxflux - minflux
delta_flux_percent = delta_flux * 0.3
thresholdflux = minflux + delta_flux_percent


# Initial flare detection.
selected1.get_bblocks(gamma_value=0.05)
selected1.find_hop(method='baseline', lc_edges ='add', baseline = thresholdflux)

quiescent_background,qui_err = quiescent_background_finder(sourcelightcurve=selected1,method='forward')


# Re-detecting flares with quiescent background.
sourcelightcurve = originalLC.select_by_time(initialtime,finaltime)
sourcelightcurve.get_bblocks(gamma_value=0.05)
sourcelightcurve.find_hop(method = 'baseline', lc_edges ='add',baseline = 0.5e-6)

hops_bl1 = sourcelightcurve.hops
plotit = originalLC.select_by_time(initialtime,finaltime)
plotit.plot_lc()
sourcelightcurve.plot_hop()

COSI_LAT_Sources = pd.read_csv(table, sep=",", na_filter=True)
# Effective Area of Fermi LAT for the source.
LAT_Aeff = COSI_LAT_Sources['Aeff_mean_LAT(cm2)'][i]

# Effective Area of COSI for the source.
COSI_Aeff = COSI_LAT_Sources['Aeff_mean_COSI(cm2)'][i]

# Name of Source
# Conversion factor of counts between Fermi and COSI Bands. (0.1-100 GeV to 0.2-5 MeV)
ph_ratio = COSI_LAT_Sources['ph/s_ratio'][i]

#ph_ratio = 1

# Conversion factor of flux between Fermi and COSI Bands. (0.1-100 GeV to 0.2-5 MeV)
flux_ratio = COSI_LAT_Sources['Int_flux_ratio'][i]


fsrq_names=select_fsrq()
blarray = [hops_bl1]
COSI_BAND_ALL = np.array([0,0,0,0,0,0,0,0,0,0,0,0,0])
for bl in blarray:
    characterized_flares = hop_characterizer(bl, COSI_bkg_rate=COSI_bkg_rate, COSI_Aeff=COSI_Aeff, LAT_Aeff=LAT_Aeff, ph_ratio=ph_ratio, int_flux_ratio=flux_ratio)
    COSI_BAND = np.array(characterized_flares).T
    COSI_BAND_ALL = np.vstack((COSI_BAND_ALL, COSI_BAND))

COSI_BAND_BAT_weekly_df = pd.DataFrame(COSI_BAND_ALL)
COSI_BAND_BAT_weekly_df.columns = ['Name','Duration_(s)', 'Source_Counts(ph)_(0.2-5_MeV)', 'Asymmetry', 'Coverage', 'Background_Counts', 'MDP99_(%)', 'Class', 'Photon_Fluence_(ph/cm2)_(0.2-5_MeV)', 'Average_Photon_Flux_(ph/cm2/s)_(0.2-5_MeV)','Start_Time_(MJD)','Average_Flux_of_Entire_Source_(0.2-5_MeV)', 'Peak_Flare_Flux_(0.2-5_MeV)']
COSI_BAND_BAT_weekly_df = COSI_BAND_BAT_weekly_df[COSI_BAND_BAT_weekly_df[:]['Source_Counts(ph)_(0.2-5_MeV)']!=0].reset_index(drop=True)
COSI_BAND_BAT_weekly_df = COSI_BAND_BAT_weekly_df[COSI_BAND_BAT_weekly_df[:]['Source_Counts(ph)_(0.2-5_MeV)']!='0.0'].reset_index(drop=True)
COSI_BAND_BAT_weekly_df['Source_Counts(ph)_(0.2-5_MeV)'] = COSI_BAND_BAT_weekly_df['Source_Counts(ph)_(0.2-5_MeV)'].astype(float)
COSI_BAND_BAT_weekly_df['Duration_(s)'] = COSI_BAND_BAT_weekly_df['Duration_(s)'].astype(float)
COSI_BAND_BAT_weekly_df['MDP99_(%)'] = ComputeMDP99(COSI_BAND_BAT_weekly_df['Source_Counts(ph)_(0.2-5_MeV)'].astype(float),
                                                COSI_BAND_BAT_weekly_df['Background_Counts'].astype(float))
COSI_BAND_BAT_weekly_df['Coverage'] = COSI_BAND_BAT_weekly_df['Coverage'].astype(float)
COSI_BAND_BAT_weekly_df = COSI_BAND_BAT_weekly_df[COSI_BAND_BAT_weekly_df[:]['Duration_(s)']<=2e8].reset_index(drop=True)


neworder = ['Name','Class','Average_Photon_Flux_(ph/cm2/s)_(0.2-5_MeV)','Photon_Fluence_(ph/cm2)_(0.2-5_MeV)','Source_Counts(ph)_(0.2-5_MeV)','Background_Counts','Duration_(s)','Start_Time_(MJD)','Coverage', 'MDP99_(%)','Asymmetry','Average_Flux_of_Entire_Source_(0.2-5_MeV)', 'Peak_Flare_Flux_(0.2-5_MeV)']
COSI_BAND_BAT_weekly_df
#COSI_BAND_BAT_weekly_df.to_csv(sourcename+'Analysis.csv')
# Remove blank space in sourcename

In [ ]:
initialtime,finaltime = 59000,60700
sourcename = '4FGL J2202.7+4216'
sourcearray = cadence_df[cadence_df['source_name'] == sourcename].reset_index(drop=True)
titlestring = sourcename
sourcearray = sourcearray[sourcearray['photon_flux2']!=-3333].reset_index(drop=True)
average_flux = np.nanmean(sourcearray['photon_flux2'])
sourcearray = sourcearray[sourcearray['photon_flux2']<=100*average_flux].reset_index(drop=True)
time = sourcearray['tmin']/SecsInDay + MJDREFI
photon_flux = sourcearray['photon_flux2']
errors = sourcearray['photon_flux_error2']
# Creates the lightcurve object of the source.
sourcelightcurve = LC.LightCurve(time,photon_flux,errors,name=titlestring)
i = COSI_LAT_Sources[COSI_LAT_Sources['Name']==sourcename].index[0]

selected1 = sourcelightcurve.select_by_time(initialtime,finaltime)
originalLC = sourcelightcurve
sourcelightcurve.get_bblocks()
photon_flux = selected1.flux

# Finding the initial flux threshold for detection.
maxflux = np.max(photon_flux)
minflux = np.min(photon_flux)
delta_flux = maxflux - minflux
delta_flux_percent = delta_flux * 0.3
thresholdflux = minflux + delta_flux_percent


# Initial flare detection.
selected1.get_bblocks(gamma_value=0.05)
selected1.find_hop(method='baseline', lc_edges ='add', baseline = thresholdflux)

quiescent_background,qui_err = quiescent_background_finder(sourcelightcurve=selected1,method='forward')


# Re-detecting flares with quiescent background.
sourcelightcurve = originalLC.select_by_time(initialtime,finaltime)
sourcelightcurve.get_bblocks(gamma_value=0.05)
sourcelightcurve.find_hop(method = 'baseline', lc_edges ='add',baseline = 1.5e-6)

hops_bl1 = sourcelightcurve.hops
plotit = originalLC.select_by_time(initialtime,finaltime)
plotit.plot_lc()
sourcelightcurve.plot_hop()

COSI_LAT_Sources = pd.read_csv(table, sep=",", na_filter=True)
# Effective Area of Fermi LAT for the source.
LAT_Aeff = COSI_LAT_Sources['Aeff_mean_LAT(cm2)'][i]

# Effective Area of COSI for the source.
COSI_Aeff = COSI_LAT_Sources['Aeff_mean_COSI(cm2)'][i]

# Name of Source
# Conversion factor of counts between Fermi and COSI Bands. (0.1-100 GeV to 0.2-5 MeV)
ph_ratio = COSI_LAT_Sources['ph/s_ratio'][i]

#ph_ratio = 1

# Conversion factor of flux between Fermi and COSI Bands. (0.1-100 GeV to 0.2-5 MeV)
flux_ratio = COSI_LAT_Sources['Int_flux_ratio'][i]


fsrq_names=select_fsrq()
blarray = [hops_bl1]
COSI_BAND_ALL = np.array([0,0,0,0,0,0,0,0,0,0,0,0,0])
for bl in blarray:
    characterized_flares = hop_characterizer(bl, COSI_bkg_rate=COSI_bkg_rate, COSI_Aeff=COSI_Aeff, LAT_Aeff=LAT_Aeff, ph_ratio=ph_ratio, int_flux_ratio=flux_ratio)
    COSI_BAND = np.array(characterized_flares).T
    COSI_BAND_ALL = np.vstack((COSI_BAND_ALL, COSI_BAND))

COSI_BAND_BAT_weekly_df = pd.DataFrame(COSI_BAND_ALL)
COSI_BAND_BAT_weekly_df.columns = ['Name','Duration_(s)', 'Source_Counts(ph)_(0.2-5_MeV)', 'Asymmetry', 'Coverage', 'Background_Counts', 'MDP99_(%)', 'Class', 'Photon_Fluence_(ph/cm2)_(0.2-5_MeV)', 'Average_Photon_Flux_(ph/cm2/s)_(0.2-5_MeV)','Start_Time_(MJD)','Average_Flux_of_Entire_Source_(0.2-5_MeV)', 'Peak_Flare_Flux_(0.2-5_MeV)']
COSI_BAND_BAT_weekly_df = COSI_BAND_BAT_weekly_df[COSI_BAND_BAT_weekly_df[:]['Source_Counts(ph)_(0.2-5_MeV)']!=0].reset_index(drop=True)
COSI_BAND_BAT_weekly_df = COSI_BAND_BAT_weekly_df[COSI_BAND_BAT_weekly_df[:]['Source_Counts(ph)_(0.2-5_MeV)']!='0.0'].reset_index(drop=True)
COSI_BAND_BAT_weekly_df['Source_Counts(ph)_(0.2-5_MeV)'] = COSI_BAND_BAT_weekly_df['Source_Counts(ph)_(0.2-5_MeV)'].astype(float)
COSI_BAND_BAT_weekly_df['Duration_(s)'] = COSI_BAND_BAT_weekly_df['Duration_(s)'].astype(float)
COSI_BAND_BAT_weekly_df['MDP99_(%)'] = ComputeMDP99(COSI_BAND_BAT_weekly_df['Source_Counts(ph)_(0.2-5_MeV)'].astype(float),
                                                COSI_BAND_BAT_weekly_df['Background_Counts'].astype(float))
COSI_BAND_BAT_weekly_df['Coverage'] = COSI_BAND_BAT_weekly_df['Coverage'].astype(float)
COSI_BAND_BAT_weekly_df = COSI_BAND_BAT_weekly_df[COSI_BAND_BAT_weekly_df[:]['Duration_(s)']<=2e8].reset_index(drop=True)


neworder = ['Name','Class','Average_Photon_Flux_(ph/cm2/s)_(0.2-5_MeV)','Photon_Fluence_(ph/cm2)_(0.2-5_MeV)','Source_Counts(ph)_(0.2-5_MeV)','Background_Counts','Duration_(s)','Start_Time_(MJD)','Coverage', 'MDP99_(%)','Asymmetry','Average_Flux_of_Entire_Source_(0.2-5_MeV)', 'Peak_Flare_Flux_(0.2-5_MeV)']
COSI_BAND_BAT_weekly_df
#COSI_BAND_BAT_weekly_df.to_csv(sourcename+'Analysis.csv')
# Remove blank space in sourcename

In [ ]:
initialtime,finaltime = 56800,59000
sourcename = '4FGL J2232.6+1143'
sourcearray = cadence_df[cadence_df['source_name'] == sourcename].reset_index(drop=True)
titlestring = sourcename
sourcearray = sourcearray[sourcearray['photon_flux2']!=-3333].reset_index(drop=True)
average_flux = np.nanmean(sourcearray['photon_flux2'])
sourcearray = sourcearray[sourcearray['photon_flux2']<=100*average_flux].reset_index(drop=True)
time = sourcearray['tmin']/SecsInDay + MJDREFI
photon_flux = sourcearray['photon_flux2']
errors = sourcearray['photon_flux_error2']
# Creates the lightcurve object of the source.
sourcelightcurve = LC.LightCurve(time,photon_flux,errors,name=titlestring)
i = COSI_LAT_Sources[COSI_LAT_Sources['Name']==sourcename].index[0]

selected1 = sourcelightcurve.select_by_time(initialtime,finaltime)
originalLC = sourcelightcurve
sourcelightcurve.get_bblocks()
photon_flux = selected1.flux

# Finding the initial flux threshold for detection.
maxflux = np.max(photon_flux)
minflux = np.min(photon_flux)
delta_flux = maxflux - minflux
delta_flux_percent = delta_flux * 0.3
thresholdflux = minflux + delta_flux_percent


# Initial flare detection.
selected1.get_bblocks(gamma_value=0.05)
selected1.find_hop(method='baseline', lc_edges ='add', baseline = thresholdflux)

quiescent_background,qui_err = quiescent_background_finder(sourcelightcurve=selected1,method='forward')


# Re-detecting flares with quiescent background.
sourcelightcurve = originalLC.select_by_time(initialtime,finaltime)
sourcelightcurve.get_bblocks(gamma_value=0.05)
sourcelightcurve.find_hop(method = 'baseline', lc_edges ='add',baseline = 3e-6)

hops_bl1 = sourcelightcurve.hops
plotit = originalLC.select_by_time(initialtime,finaltime)
plotit.plot_lc()
sourcelightcurve.plot_hop()

COSI_LAT_Sources = pd.read_csv(table, sep=",", na_filter=True)
# Effective Area of Fermi LAT for the source.
LAT_Aeff = COSI_LAT_Sources['Aeff_mean_LAT(cm2)'][i]

# Effective Area of COSI for the source.
COSI_Aeff = COSI_LAT_Sources['Aeff_mean_COSI(cm2)'][i]

# Name of Source
# Conversion factor of counts between Fermi and COSI Bands. (0.1-100 GeV to 0.2-5 MeV)
ph_ratio = COSI_LAT_Sources['ph/s_ratio'][i]

#ph_ratio = 1

# Conversion factor of flux between Fermi and COSI Bands. (0.1-100 GeV to 0.2-5 MeV)
flux_ratio = COSI_LAT_Sources['Int_flux_ratio'][i]


fsrq_names=select_fsrq()
blarray = [hops_bl1]
COSI_BAND_ALL = np.array([0,0,0,0,0,0,0,0,0,0,0,0,0])
for bl in blarray:
    characterized_flares = hop_characterizer(bl, COSI_bkg_rate=COSI_bkg_rate, COSI_Aeff=COSI_Aeff, LAT_Aeff=LAT_Aeff, ph_ratio=ph_ratio, int_flux_ratio=flux_ratio)
    COSI_BAND = np.array(characterized_flares).T
    COSI_BAND_ALL = np.vstack((COSI_BAND_ALL, COSI_BAND))

COSI_BAND_BAT_weekly_df = pd.DataFrame(COSI_BAND_ALL)
COSI_BAND_BAT_weekly_df.columns = ['Name','Duration_(s)', 'Source_Counts(ph)_(0.2-5_MeV)', 'Asymmetry', 'Coverage', 'Background_Counts', 'MDP99_(%)', 'Class', 'Photon_Fluence_(ph/cm2)_(0.2-5_MeV)', 'Average_Photon_Flux_(ph/cm2/s)_(0.2-5_MeV)','Start_Time_(MJD)','Average_Flux_of_Entire_Source_(0.2-5_MeV)', 'Peak_Flare_Flux_(0.2-5_MeV)']
COSI_BAND_BAT_weekly_df = COSI_BAND_BAT_weekly_df[COSI_BAND_BAT_weekly_df[:]['Source_Counts(ph)_(0.2-5_MeV)']!=0].reset_index(drop=True)
COSI_BAND_BAT_weekly_df = COSI_BAND_BAT_weekly_df[COSI_BAND_BAT_weekly_df[:]['Source_Counts(ph)_(0.2-5_MeV)']!='0.0'].reset_index(drop=True)
COSI_BAND_BAT_weekly_df['Source_Counts(ph)_(0.2-5_MeV)'] = COSI_BAND_BAT_weekly_df['Source_Counts(ph)_(0.2-5_MeV)'].astype(float)
COSI_BAND_BAT_weekly_df['Duration_(s)'] = COSI_BAND_BAT_weekly_df['Duration_(s)'].astype(float)
COSI_BAND_BAT_weekly_df['MDP99_(%)'] = ComputeMDP99(COSI_BAND_BAT_weekly_df['Source_Counts(ph)_(0.2-5_MeV)'].astype(float),
                                                COSI_BAND_BAT_weekly_df['Background_Counts'].astype(float))
COSI_BAND_BAT_weekly_df['Coverage'] = COSI_BAND_BAT_weekly_df['Coverage'].astype(float)
COSI_BAND_BAT_weekly_df = COSI_BAND_BAT_weekly_df[COSI_BAND_BAT_weekly_df[:]['Duration_(s)']<=2e8].reset_index(drop=True)


neworder = ['Name','Class','Average_Photon_Flux_(ph/cm2/s)_(0.2-5_MeV)','Photon_Fluence_(ph/cm2)_(0.2-5_MeV)','Source_Counts(ph)_(0.2-5_MeV)','Background_Counts','Duration_(s)','Start_Time_(MJD)','Coverage', 'MDP99_(%)','Asymmetry','Average_Flux_of_Entire_Source_(0.2-5_MeV)', 'Peak_Flare_Flux_(0.2-5_MeV)']
COSI_BAND_BAT_weekly_df
#COSI_BAND_BAT_weekly_df.to_csv(sourcename+'Analysis.csv')
# Remove blank space in sourcename

In [ ]:
initialtime,finaltime = 56800,59000
sourcename = '4FGL J2232.6+1143'
sourcearray = cadence_df[cadence_df['source_name'] == sourcename].reset_index(drop=True)
titlestring = sourcename
sourcearray = sourcearray[sourcearray['photon_flux2']!=-3333].reset_index(drop=True)
average_flux = np.nanmean(sourcearray['photon_flux2'])
sourcearray = sourcearray[sourcearray['photon_flux2']<=100*average_flux].reset_index(drop=True)
time = sourcearray['tmin']/SecsInDay + MJDREFI
photon_flux = sourcearray['photon_flux2']
errors = sourcearray['photon_flux_error2']
# Creates the lightcurve object of the source.
sourcelightcurve = LC.LightCurve(time,photon_flux,errors,name=titlestring)
i = COSI_LAT_Sources[COSI_LAT_Sources['Name']==sourcename].index[0]

selected1 = sourcelightcurve.select_by_time(initialtime,finaltime)
originalLC = sourcelightcurve
sourcelightcurve.get_bblocks()
photon_flux = selected1.flux

# Finding the initial flux threshold for detection.
maxflux = np.max(photon_flux)
minflux = np.min(photon_flux)
delta_flux = maxflux - minflux
delta_flux_percent = delta_flux * 0.3
thresholdflux = minflux + delta_flux_percent


# Initial flare detection.
selected1.get_bblocks(gamma_value=0.05)
selected1.find_hop(method='baseline', lc_edges ='add', baseline = thresholdflux)

quiescent_background,qui_err = quiescent_background_finder(sourcelightcurve=selected1,method='forward')


# Re-detecting flares with quiescent background.
sourcelightcurve = originalLC.select_by_time(initialtime,finaltime)
sourcelightcurve.get_bblocks(gamma_value=0.05)
sourcelightcurve.find_hop(method = 'baseline', lc_edges ='add',baseline = 6e-6)

hops_bl1 = sourcelightcurve.hops
plotit = originalLC.select_by_time(initialtime,finaltime)
plotit.plot_lc()
sourcelightcurve.plot_hop()

COSI_LAT_Sources = pd.read_csv(table, sep=",", na_filter=True)
# Effective Area of Fermi LAT for the source.
LAT_Aeff = COSI_LAT_Sources['Aeff_mean_LAT(cm2)'][i]

# Effective Area of COSI for the source.
COSI_Aeff = COSI_LAT_Sources['Aeff_mean_COSI(cm2)'][i]

# Name of Source
# Conversion factor of counts between Fermi and COSI Bands. (0.1-100 GeV to 0.2-5 MeV)
ph_ratio = COSI_LAT_Sources['ph/s_ratio'][i]

#ph_ratio = 1

# Conversion factor of flux between Fermi and COSI Bands. (0.1-100 GeV to 0.2-5 MeV)
flux_ratio = COSI_LAT_Sources['Int_flux_ratio'][i]


fsrq_names=select_fsrq()
blarray = [hops_bl1]
COSI_BAND_ALL = np.array([0,0,0,0,0,0,0,0,0,0,0,0,0])
for bl in blarray:
    characterized_flares = hop_characterizer(bl, COSI_bkg_rate=COSI_bkg_rate, COSI_Aeff=COSI_Aeff, LAT_Aeff=LAT_Aeff, ph_ratio=ph_ratio, int_flux_ratio=flux_ratio)
    COSI_BAND = np.array(characterized_flares).T
    COSI_BAND_ALL = np.vstack((COSI_BAND_ALL, COSI_BAND))

COSI_BAND_BAT_weekly_df = pd.DataFrame(COSI_BAND_ALL)
COSI_BAND_BAT_weekly_df.columns = ['Name','Duration_(s)', 'Source_Counts(ph)_(0.2-5_MeV)', 'Asymmetry', 'Coverage', 'Background_Counts', 'MDP99_(%)', 'Class', 'Photon_Fluence_(ph/cm2)_(0.2-5_MeV)', 'Average_Photon_Flux_(ph/cm2/s)_(0.2-5_MeV)','Start_Time_(MJD)','Average_Flux_of_Entire_Source_(0.2-5_MeV)', 'Peak_Flare_Flux_(0.2-5_MeV)']
COSI_BAND_BAT_weekly_df = COSI_BAND_BAT_weekly_df[COSI_BAND_BAT_weekly_df[:]['Source_Counts(ph)_(0.2-5_MeV)']!=0].reset_index(drop=True)
COSI_BAND_BAT_weekly_df = COSI_BAND_BAT_weekly_df[COSI_BAND_BAT_weekly_df[:]['Source_Counts(ph)_(0.2-5_MeV)']!='0.0'].reset_index(drop=True)
COSI_BAND_BAT_weekly_df['Source_Counts(ph)_(0.2-5_MeV)'] = COSI_BAND_BAT_weekly_df['Source_Counts(ph)_(0.2-5_MeV)'].astype(float)
COSI_BAND_BAT_weekly_df['Duration_(s)'] = COSI_BAND_BAT_weekly_df['Duration_(s)'].astype(float)
COSI_BAND_BAT_weekly_df['MDP99_(%)'] = ComputeMDP99(COSI_BAND_BAT_weekly_df['Source_Counts(ph)_(0.2-5_MeV)'].astype(float),
                                                COSI_BAND_BAT_weekly_df['Background_Counts'].astype(float))
COSI_BAND_BAT_weekly_df['Coverage'] = COSI_BAND_BAT_weekly_df['Coverage'].astype(float)
COSI_BAND_BAT_weekly_df = COSI_BAND_BAT_weekly_df[COSI_BAND_BAT_weekly_df[:]['Duration_(s)']<=2e8].reset_index(drop=True)


neworder = ['Name','Class','Average_Photon_Flux_(ph/cm2/s)_(0.2-5_MeV)','Photon_Fluence_(ph/cm2)_(0.2-5_MeV)','Source_Counts(ph)_(0.2-5_MeV)','Background_Counts','Duration_(s)','Start_Time_(MJD)','Coverage', 'MDP99_(%)','Asymmetry','Average_Flux_of_Entire_Source_(0.2-5_MeV)', 'Peak_Flare_Flux_(0.2-5_MeV)']
COSI_BAND_BAT_weekly_df
#COSI_BAND_BAT_weekly_df.to_csv(sourcename+'Analysis.csv')
# Remove blank space in sourcename

In [ ]:
initialtime,finaltime = 59100,59600
sourcename = '4FGL J0539.9-2839'
sourcearray = cadence_df[cadence_df['source_name'] == sourcename].reset_index(drop=True)
titlestring = sourcename
sourcearray = sourcearray[sourcearray['photon_flux2']!=-3333].reset_index(drop=True)
average_flux = np.nanmean(sourcearray['photon_flux2'])
sourcearray = sourcearray[sourcearray['photon_flux2']<=100*average_flux].reset_index(drop=True)
time = sourcearray['tmin']/SecsInDay + MJDREFI
photon_flux = sourcearray['photon_flux2']
errors = sourcearray['photon_flux_error2']
# Creates the lightcurve object of the source.
sourcelightcurve = LC.LightCurve(time,photon_flux,errors,name=titlestring)
i = COSI_LAT_Sources[COSI_LAT_Sources['Name']==sourcename].index[0]

selected1 = sourcelightcurve.select_by_time(initialtime,finaltime)
originalLC = sourcelightcurve
sourcelightcurve.get_bblocks()
photon_flux = selected1.flux

# Finding the initial flux threshold for detection.
maxflux = np.max(photon_flux)
minflux = np.min(photon_flux)
delta_flux = maxflux - minflux
delta_flux_percent = delta_flux * 0.3
thresholdflux = minflux + delta_flux_percent


# Initial flare detection.
selected1.get_bblocks(gamma_value=0.05)
selected1.find_hop(method='baseline', lc_edges ='add', baseline = thresholdflux)

quiescent_background,qui_err = quiescent_background_finder(sourcelightcurve=selected1,method='forward')


# Re-detecting flares with quiescent background.
sourcelightcurve = originalLC.select_by_time(initialtime,finaltime)
sourcelightcurve.get_bblocks(gamma_value=0.05)
sourcelightcurve.find_hop(method = 'baseline', lc_edges ='add',baseline = 0.2e-6)

hops_bl1 = sourcelightcurve.hops
plotit = originalLC.select_by_time(initialtime,finaltime)
plotit.plot_lc()
sourcelightcurve.plot_hop()

COSI_LAT_Sources = pd.read_csv(table, sep=",", na_filter=True)
# Effective Area of Fermi LAT for the source.
LAT_Aeff = COSI_LAT_Sources['Aeff_mean_LAT(cm2)'][i]

# Effective Area of COSI for the source.
COSI_Aeff = COSI_LAT_Sources['Aeff_mean_COSI(cm2)'][i]

# Name of Source
# Conversion factor of counts between Fermi and COSI Bands. (0.1-100 GeV to 0.2-5 MeV)
ph_ratio = COSI_LAT_Sources['ph/s_ratio'][i]

#ph_ratio = 1

# Conversion factor of flux between Fermi and COSI Bands. (0.1-100 GeV to 0.2-5 MeV)
flux_ratio = COSI_LAT_Sources['Int_flux_ratio'][i]


fsrq_names=select_fsrq()
blarray = [hops_bl1]
COSI_BAND_ALL = np.array([0,0,0,0,0,0,0,0,0,0,0,0,0])
for bl in blarray:
    characterized_flares = hop_characterizer(bl, COSI_bkg_rate=COSI_bkg_rate, COSI_Aeff=COSI_Aeff, LAT_Aeff=LAT_Aeff, ph_ratio=ph_ratio, int_flux_ratio=flux_ratio)
    COSI_BAND = np.array(characterized_flares).T
    COSI_BAND_ALL = np.vstack((COSI_BAND_ALL, COSI_BAND))

COSI_BAND_BAT_weekly_df = pd.DataFrame(COSI_BAND_ALL)
COSI_BAND_BAT_weekly_df.columns = ['Name','Duration_(s)', 'Source_Counts(ph)_(0.2-5_MeV)', 'Asymmetry', 'Coverage', 'Background_Counts', 'MDP99_(%)', 'Class', 'Photon_Fluence_(ph/cm2)_(0.2-5_MeV)', 'Average_Photon_Flux_(ph/cm2/s)_(0.2-5_MeV)','Start_Time_(MJD)','Average_Flux_of_Entire_Source_(0.2-5_MeV)', 'Peak_Flare_Flux_(0.2-5_MeV)']
COSI_BAND_BAT_weekly_df = COSI_BAND_BAT_weekly_df[COSI_BAND_BAT_weekly_df[:]['Source_Counts(ph)_(0.2-5_MeV)']!=0].reset_index(drop=True)
COSI_BAND_BAT_weekly_df = COSI_BAND_BAT_weekly_df[COSI_BAND_BAT_weekly_df[:]['Source_Counts(ph)_(0.2-5_MeV)']!='0.0'].reset_index(drop=True)
COSI_BAND_BAT_weekly_df['Source_Counts(ph)_(0.2-5_MeV)'] = COSI_BAND_BAT_weekly_df['Source_Counts(ph)_(0.2-5_MeV)'].astype(float)
COSI_BAND_BAT_weekly_df['Duration_(s)'] = COSI_BAND_BAT_weekly_df['Duration_(s)'].astype(float)
COSI_BAND_BAT_weekly_df['MDP99_(%)'] = ComputeMDP99(COSI_BAND_BAT_weekly_df['Source_Counts(ph)_(0.2-5_MeV)'].astype(float),
                                                COSI_BAND_BAT_weekly_df['Background_Counts'].astype(float))
COSI_BAND_BAT_weekly_df['Coverage'] = COSI_BAND_BAT_weekly_df['Coverage'].astype(float)
COSI_BAND_BAT_weekly_df = COSI_BAND_BAT_weekly_df[COSI_BAND_BAT_weekly_df[:]['Duration_(s)']<=2e8].reset_index(drop=True)


neworder = ['Name','Class','Average_Photon_Flux_(ph/cm2/s)_(0.2-5_MeV)','Photon_Fluence_(ph/cm2)_(0.2-5_MeV)','Source_Counts(ph)_(0.2-5_MeV)','Background_Counts','Duration_(s)','Start_Time_(MJD)','Coverage', 'MDP99_(%)','Asymmetry','Average_Flux_of_Entire_Source_(0.2-5_MeV)', 'Peak_Flare_Flux_(0.2-5_MeV)']
COSI_BAND_BAT_weekly_df
#COSI_BAND_BAT_weekly_df.to_csv(sourcename+'Analysis.csv')
# Remove blank space in sourcename

In [ ]:
initialtime,finaltime = 57000,58000
sourcename = '4FGL J0841.3+7053'
sourcearray = cadence_df[cadence_df['source_name'] == sourcename].reset_index(drop=True)
titlestring = sourcename
sourcearray = sourcearray[sourcearray['photon_flux2']!=-3333].reset_index(drop=True)
average_flux = np.nanmean(sourcearray['photon_flux2'])
sourcearray = sourcearray[sourcearray['photon_flux2']<=100*average_flux].reset_index(drop=True)
time = sourcearray['tmin']/SecsInDay + MJDREFI
photon_flux = sourcearray['photon_flux2']
errors = sourcearray['photon_flux_error2']
# Creates the lightcurve object of the source.
sourcelightcurve = LC.LightCurve(time,photon_flux,errors,name=titlestring)
i = COSI_LAT_Sources[COSI_LAT_Sources['Name']==sourcename].index[0]

selected1 = sourcelightcurve.select_by_time(initialtime,finaltime)
originalLC = sourcelightcurve
sourcelightcurve.get_bblocks()
photon_flux = selected1.flux

# Finding the initial flux threshold for detection.
maxflux = np.max(photon_flux)
minflux = np.min(photon_flux)
delta_flux = maxflux - minflux
delta_flux_percent = delta_flux * 0.3
thresholdflux = minflux + delta_flux_percent


# Initial flare detection.
selected1.get_bblocks(gamma_value=0.05)
selected1.find_hop(method='baseline', lc_edges ='add', baseline = thresholdflux)

quiescent_background,qui_err = quiescent_background_finder(sourcelightcurve=selected1,method='forward')


# Re-detecting flares with quiescent background.
sourcelightcurve = originalLC.select_by_time(initialtime,finaltime)
sourcelightcurve.get_bblocks(gamma_value=0.05)
sourcelightcurve.find_hop(method = 'baseline', lc_edges ='add',baseline = 0.4e-6)

hops_bl1 = sourcelightcurve.hops
plotit = originalLC.select_by_time(initialtime,finaltime)
plotit.plot_lc()
sourcelightcurve.plot_hop()

COSI_LAT_Sources = pd.read_csv(table, sep=",", na_filter=True)
# Effective Area of Fermi LAT for the source.
LAT_Aeff = COSI_LAT_Sources['Aeff_mean_LAT(cm2)'][i]

# Effective Area of COSI for the source.
COSI_Aeff = COSI_LAT_Sources['Aeff_mean_COSI(cm2)'][i]

# Name of Source
# Conversion factor of counts between Fermi and COSI Bands. (0.1-100 GeV to 0.2-5 MeV)
ph_ratio = COSI_LAT_Sources['ph/s_ratio'][i]

#ph_ratio = 1

# Conversion factor of flux between Fermi and COSI Bands. (0.1-100 GeV to 0.2-5 MeV)
flux_ratio = COSI_LAT_Sources['Int_flux_ratio'][i]


fsrq_names=select_fsrq()
blarray = [hops_bl1]
COSI_BAND_ALL = np.array([0,0,0,0,0,0,0,0,0,0,0,0,0])
for bl in blarray:
    characterized_flares = hop_characterizer(bl, COSI_bkg_rate=COSI_bkg_rate, COSI_Aeff=COSI_Aeff, LAT_Aeff=LAT_Aeff, ph_ratio=ph_ratio, int_flux_ratio=flux_ratio)
    COSI_BAND = np.array(characterized_flares).T
    COSI_BAND_ALL = np.vstack((COSI_BAND_ALL, COSI_BAND))

COSI_BAND_BAT_weekly_df = pd.DataFrame(COSI_BAND_ALL)
COSI_BAND_BAT_weekly_df.columns = ['Name','Duration_(s)', 'Source_Counts(ph)_(0.2-5_MeV)', 'Asymmetry', 'Coverage', 'Background_Counts', 'MDP99_(%)', 'Class', 'Photon_Fluence_(ph/cm2)_(0.2-5_MeV)', 'Average_Photon_Flux_(ph/cm2/s)_(0.2-5_MeV)','Start_Time_(MJD)','Average_Flux_of_Entire_Source_(0.2-5_MeV)', 'Peak_Flare_Flux_(0.2-5_MeV)']
COSI_BAND_BAT_weekly_df = COSI_BAND_BAT_weekly_df[COSI_BAND_BAT_weekly_df[:]['Source_Counts(ph)_(0.2-5_MeV)']!=0].reset_index(drop=True)
COSI_BAND_BAT_weekly_df = COSI_BAND_BAT_weekly_df[COSI_BAND_BAT_weekly_df[:]['Source_Counts(ph)_(0.2-5_MeV)']!='0.0'].reset_index(drop=True)
COSI_BAND_BAT_weekly_df['Source_Counts(ph)_(0.2-5_MeV)'] = COSI_BAND_BAT_weekly_df['Source_Counts(ph)_(0.2-5_MeV)'].astype(float)
COSI_BAND_BAT_weekly_df['Duration_(s)'] = COSI_BAND_BAT_weekly_df['Duration_(s)'].astype(float)
COSI_BAND_BAT_weekly_df['MDP99_(%)'] = ComputeMDP99(COSI_BAND_BAT_weekly_df['Source_Counts(ph)_(0.2-5_MeV)'].astype(float),
                                                COSI_BAND_BAT_weekly_df['Background_Counts'].astype(float))
COSI_BAND_BAT_weekly_df['Coverage'] = COSI_BAND_BAT_weekly_df['Coverage'].astype(float)
COSI_BAND_BAT_weekly_df = COSI_BAND_BAT_weekly_df[COSI_BAND_BAT_weekly_df[:]['Duration_(s)']<=2e8].reset_index(drop=True)


neworder = ['Name','Class','Average_Photon_Flux_(ph/cm2/s)_(0.2-5_MeV)','Photon_Fluence_(ph/cm2)_(0.2-5_MeV)','Source_Counts(ph)_(0.2-5_MeV)','Background_Counts','Duration_(s)','Start_Time_(MJD)','Coverage', 'MDP99_(%)','Asymmetry','Average_Flux_of_Entire_Source_(0.2-5_MeV)', 'Peak_Flare_Flux_(0.2-5_MeV)']
COSI_BAND_BAT_weekly_df
#COSI_BAND_BAT_weekly_df.to_csv(sourcename+'Analysis.csv')
# Remove blank space in sourcename

In [ ]:
initialtime,finaltime = 59500,60500
sourcename = '4FGL J1129.8-1447'
sourcearray = cadence_df[cadence_df['source_name'] == sourcename].reset_index(drop=True)
titlestring = sourcename
sourcearray = sourcearray[sourcearray['photon_flux2']!=-3333].reset_index(drop=True)
average_flux = np.nanmean(sourcearray['photon_flux2'])
sourcearray = sourcearray[sourcearray['photon_flux2']<=100*average_flux].reset_index(drop=True)
time = sourcearray['tmin']/SecsInDay + MJDREFI
photon_flux = sourcearray['photon_flux2']
errors = sourcearray['photon_flux_error2']
# Creates the lightcurve object of the source.
sourcelightcurve = LC.LightCurve(time,photon_flux,errors,name=titlestring)
i = COSI_LAT_Sources[COSI_LAT_Sources['Name']==sourcename].index[0]

selected1 = sourcelightcurve.select_by_time(initialtime,finaltime)
originalLC = sourcelightcurve
sourcelightcurve.get_bblocks()
photon_flux = selected1.flux

# Finding the initial flux threshold for detection.
maxflux = np.max(photon_flux)
minflux = np.min(photon_flux)
delta_flux = maxflux - minflux
delta_flux_percent = delta_flux * 0.3
thresholdflux = minflux + delta_flux_percent


# Initial flare detection.
selected1.get_bblocks(gamma_value=0.05)
selected1.find_hop(method='baseline', lc_edges ='add', baseline = thresholdflux)

quiescent_background,qui_err = quiescent_background_finder(sourcelightcurve=selected1,method='forward')


# Re-detecting flares with quiescent background.
sourcelightcurve = originalLC.select_by_time(initialtime,finaltime)
sourcelightcurve.get_bblocks(gamma_value=0.05)
sourcelightcurve.find_hop(method = 'baseline', lc_edges ='add',baseline = 0.5e-6)

hops_bl1 = sourcelightcurve.hops
plotit = originalLC.select_by_time(initialtime,finaltime)
plotit.plot_lc()
sourcelightcurve.plot_hop()

COSI_LAT_Sources = pd.read_csv(table, sep=",", na_filter=True)
# Effective Area of Fermi LAT for the source.
LAT_Aeff = COSI_LAT_Sources['Aeff_mean_LAT(cm2)'][i]

# Effective Area of COSI for the source.
COSI_Aeff = COSI_LAT_Sources['Aeff_mean_COSI(cm2)'][i]

# Name of Source
# Conversion factor of counts between Fermi and COSI Bands. (0.1-100 GeV to 0.2-5 MeV)
ph_ratio = COSI_LAT_Sources['ph/s_ratio'][i]

#ph_ratio = 1

# Conversion factor of flux between Fermi and COSI Bands. (0.1-100 GeV to 0.2-5 MeV)
flux_ratio = COSI_LAT_Sources['Int_flux_ratio'][i]


fsrq_names=select_fsrq()
blarray = [hops_bl1]
COSI_BAND_ALL = np.array([0,0,0,0,0,0,0,0,0,0,0,0,0])
for bl in blarray:
    characterized_flares = hop_characterizer(bl, COSI_bkg_rate=COSI_bkg_rate, COSI_Aeff=COSI_Aeff, LAT_Aeff=LAT_Aeff, ph_ratio=ph_ratio, int_flux_ratio=flux_ratio)
    COSI_BAND = np.array(characterized_flares).T
    COSI_BAND_ALL = np.vstack((COSI_BAND_ALL, COSI_BAND))

COSI_BAND_BAT_weekly_df = pd.DataFrame(COSI_BAND_ALL)
COSI_BAND_BAT_weekly_df.columns = ['Name','Duration_(s)', 'Source_Counts(ph)_(0.2-5_MeV)', 'Asymmetry', 'Coverage', 'Background_Counts', 'MDP99_(%)', 'Class', 'Photon_Fluence_(ph/cm2)_(0.2-5_MeV)', 'Average_Photon_Flux_(ph/cm2/s)_(0.2-5_MeV)','Start_Time_(MJD)','Average_Flux_of_Entire_Source_(0.2-5_MeV)', 'Peak_Flare_Flux_(0.2-5_MeV)']
COSI_BAND_BAT_weekly_df = COSI_BAND_BAT_weekly_df[COSI_BAND_BAT_weekly_df[:]['Source_Counts(ph)_(0.2-5_MeV)']!=0].reset_index(drop=True)
COSI_BAND_BAT_weekly_df = COSI_BAND_BAT_weekly_df[COSI_BAND_BAT_weekly_df[:]['Source_Counts(ph)_(0.2-5_MeV)']!='0.0'].reset_index(drop=True)
COSI_BAND_BAT_weekly_df['Source_Counts(ph)_(0.2-5_MeV)'] = COSI_BAND_BAT_weekly_df['Source_Counts(ph)_(0.2-5_MeV)'].astype(float)
COSI_BAND_BAT_weekly_df['Duration_(s)'] = COSI_BAND_BAT_weekly_df['Duration_(s)'].astype(float)
COSI_BAND_BAT_weekly_df['MDP99_(%)'] = ComputeMDP99(COSI_BAND_BAT_weekly_df['Source_Counts(ph)_(0.2-5_MeV)'].astype(float),
                                                COSI_BAND_BAT_weekly_df['Background_Counts'].astype(float))
COSI_BAND_BAT_weekly_df['Coverage'] = COSI_BAND_BAT_weekly_df['Coverage'].astype(float)
COSI_BAND_BAT_weekly_df = COSI_BAND_BAT_weekly_df[COSI_BAND_BAT_weekly_df[:]['Duration_(s)']<=2e8].reset_index(drop=True)


neworder = ['Name','Class','Average_Photon_Flux_(ph/cm2/s)_(0.2-5_MeV)','Photon_Fluence_(ph/cm2)_(0.2-5_MeV)','Source_Counts(ph)_(0.2-5_MeV)','Background_Counts','Duration_(s)','Start_Time_(MJD)','Coverage', 'MDP99_(%)','Asymmetry','Average_Flux_of_Entire_Source_(0.2-5_MeV)', 'Peak_Flare_Flux_(0.2-5_MeV)']
COSI_BAND_BAT_weekly_df
#COSI_BAND_BAT_weekly_df.to_csv(sourcename+'Analysis.csv')
# Remove blank space in sourcename

In [ ]:
initialtime,finaltime = 54500,55500
sourcename = '4FGL J1229.0+0202'
sourcearray = cadence_df[cadence_df['source_name'] == sourcename].reset_index(drop=True)
titlestring = sourcename
sourcearray = sourcearray[sourcearray['photon_flux2']!=-3333].reset_index(drop=True)
average_flux = np.nanmean(sourcearray['photon_flux2'])
sourcearray = sourcearray[sourcearray['photon_flux2']<=100*average_flux].reset_index(drop=True)
time = sourcearray['tmin']/SecsInDay + MJDREFI
photon_flux = sourcearray['photon_flux2']
errors = sourcearray['photon_flux_error2']
# Creates the lightcurve object of the source.
sourcelightcurve = LC.LightCurve(time,photon_flux,errors,name=titlestring)
i = COSI_LAT_Sources[COSI_LAT_Sources['Name']==sourcename].index[0]

selected1 = sourcelightcurve.select_by_time(initialtime,finaltime)
originalLC = sourcelightcurve
sourcelightcurve.get_bblocks()
photon_flux = selected1.flux

# Finding the initial flux threshold for detection.
maxflux = np.max(photon_flux)
minflux = np.min(photon_flux)
delta_flux = maxflux - minflux
delta_flux_percent = delta_flux * 0.3
thresholdflux = minflux + delta_flux_percent


# Initial flare detection.
selected1.get_bblocks(gamma_value=0.05)
selected1.find_hop(method='baseline', lc_edges ='add', baseline = thresholdflux)

quiescent_background,qui_err = quiescent_background_finder(sourcelightcurve=selected1,method='forward')


# Re-detecting flares with quiescent background.
sourcelightcurve = originalLC.select_by_time(initialtime,finaltime)
sourcelightcurve.get_bblocks(gamma_value=0.05)
sourcelightcurve.find_hop(method = 'baseline', lc_edges ='add',baseline = 1e-6)

hops_bl1 = sourcelightcurve.hops
plotit = originalLC.select_by_time(initialtime,finaltime)
plotit.plot_lc()
sourcelightcurve.plot_hop()

COSI_LAT_Sources = pd.read_csv(table, sep=",", na_filter=True)
# Effective Area of Fermi LAT for the source.
LAT_Aeff = COSI_LAT_Sources['Aeff_mean_LAT(cm2)'][i]

# Effective Area of COSI for the source.
COSI_Aeff = COSI_LAT_Sources['Aeff_mean_COSI(cm2)'][i]

# Name of Source
# Conversion factor of counts between Fermi and COSI Bands. (0.1-100 GeV to 0.2-5 MeV)
ph_ratio = COSI_LAT_Sources['ph/s_ratio'][i]

#ph_ratio = 1

# Conversion factor of flux between Fermi and COSI Bands. (0.1-100 GeV to 0.2-5 MeV)
flux_ratio = COSI_LAT_Sources['Int_flux_ratio'][i]


fsrq_names=select_fsrq()
blarray = [hops_bl1]
COSI_BAND_ALL = np.array([0,0,0,0,0,0,0,0,0,0,0,0,0])
for bl in blarray:
    characterized_flares = hop_characterizer(bl, COSI_bkg_rate=COSI_bkg_rate, COSI_Aeff=COSI_Aeff, LAT_Aeff=LAT_Aeff, ph_ratio=ph_ratio, int_flux_ratio=flux_ratio)
    COSI_BAND = np.array(characterized_flares).T
    COSI_BAND_ALL = np.vstack((COSI_BAND_ALL, COSI_BAND))

COSI_BAND_BAT_weekly_df = pd.DataFrame(COSI_BAND_ALL)
COSI_BAND_BAT_weekly_df.columns = ['Name','Duration_(s)', 'Source_Counts(ph)_(0.2-5_MeV)', 'Asymmetry', 'Coverage', 'Background_Counts', 'MDP99_(%)', 'Class', 'Photon_Fluence_(ph/cm2)_(0.2-5_MeV)', 'Average_Photon_Flux_(ph/cm2/s)_(0.2-5_MeV)','Start_Time_(MJD)','Average_Flux_of_Entire_Source_(0.2-5_MeV)', 'Peak_Flare_Flux_(0.2-5_MeV)']
COSI_BAND_BAT_weekly_df = COSI_BAND_BAT_weekly_df[COSI_BAND_BAT_weekly_df[:]['Source_Counts(ph)_(0.2-5_MeV)']!=0].reset_index(drop=True)
COSI_BAND_BAT_weekly_df = COSI_BAND_BAT_weekly_df[COSI_BAND_BAT_weekly_df[:]['Source_Counts(ph)_(0.2-5_MeV)']!='0.0'].reset_index(drop=True)
COSI_BAND_BAT_weekly_df['Source_Counts(ph)_(0.2-5_MeV)'] = COSI_BAND_BAT_weekly_df['Source_Counts(ph)_(0.2-5_MeV)'].astype(float)
COSI_BAND_BAT_weekly_df['Duration_(s)'] = COSI_BAND_BAT_weekly_df['Duration_(s)'].astype(float)
COSI_BAND_BAT_weekly_df['MDP99_(%)'] = ComputeMDP99(COSI_BAND_BAT_weekly_df['Source_Counts(ph)_(0.2-5_MeV)'].astype(float),
                                                COSI_BAND_BAT_weekly_df['Background_Counts'].astype(float))
COSI_BAND_BAT_weekly_df['Coverage'] = COSI_BAND_BAT_weekly_df['Coverage'].astype(float)
COSI_BAND_BAT_weekly_df = COSI_BAND_BAT_weekly_df[COSI_BAND_BAT_weekly_df[:]['Duration_(s)']<=2e8].reset_index(drop=True)


neworder = ['Name','Class','Average_Photon_Flux_(ph/cm2/s)_(0.2-5_MeV)','Photon_Fluence_(ph/cm2)_(0.2-5_MeV)','Source_Counts(ph)_(0.2-5_MeV)','Background_Counts','Duration_(s)','Start_Time_(MJD)','Coverage', 'MDP99_(%)','Asymmetry','Average_Flux_of_Entire_Source_(0.2-5_MeV)', 'Peak_Flare_Flux_(0.2-5_MeV)']
COSI_BAND_BAT_weekly_df
#COSI_BAND_BAT_weekly_df.to_csv(sourcename+'Analysis.csv')
# Remove blank space in sourcename

In [ ]:
initialtime,finaltime = 57700,58300
sourcename = '4FGL J1256.1-0547'
sourcearray = cadence_df[cadence_df['source_name'] == sourcename].reset_index(drop=True)
titlestring = sourcename
sourcearray = sourcearray[sourcearray['photon_flux2']!=-3333].reset_index(drop=True)
average_flux = np.nanmean(sourcearray['photon_flux2'])
sourcearray = sourcearray[sourcearray['photon_flux2']<=100*average_flux].reset_index(drop=True)
time = sourcearray['tmin']/SecsInDay + MJDREFI
photon_flux = sourcearray['photon_flux2']
errors = sourcearray['photon_flux_error2']
# Creates the lightcurve object of the source.
sourcelightcurve = LC.LightCurve(time,photon_flux,errors,name=titlestring)
i = COSI_LAT_Sources[COSI_LAT_Sources['Name']==sourcename].index[0]

selected1 = sourcelightcurve.select_by_time(initialtime,finaltime)
originalLC = sourcelightcurve
sourcelightcurve.get_bblocks()
photon_flux = selected1.flux

# Finding the initial flux threshold for detection.
maxflux = np.max(photon_flux)
minflux = np.min(photon_flux)
delta_flux = maxflux - minflux
delta_flux_percent = delta_flux * 0.3
thresholdflux = minflux + delta_flux_percent


# Initial flare detection.
selected1.get_bblocks(gamma_value=0.05)
selected1.find_hop(method='baseline', lc_edges ='add', baseline = thresholdflux)

quiescent_background,qui_err = quiescent_background_finder(sourcelightcurve=selected1,method='forward')


# Re-detecting flares with quiescent background.
sourcelightcurve = originalLC.select_by_time(initialtime,finaltime)
sourcelightcurve.get_bblocks(gamma_value=0.05)
sourcelightcurve.find_hop(method = 'baseline', lc_edges ='add',baseline = 2e-6)

hops_bl1 = sourcelightcurve.hops
plotit = originalLC.select_by_time(initialtime,finaltime)
plotit.plot_lc()
sourcelightcurve.plot_hop()

COSI_LAT_Sources = pd.read_csv(table, sep=",", na_filter=True)
# Effective Area of Fermi LAT for the source.
LAT_Aeff = COSI_LAT_Sources['Aeff_mean_LAT(cm2)'][i]

# Effective Area of COSI for the source.
COSI_Aeff = COSI_LAT_Sources['Aeff_mean_COSI(cm2)'][i]

# Name of Source
# Conversion factor of counts between Fermi and COSI Bands. (0.1-100 GeV to 0.2-5 MeV)
ph_ratio = COSI_LAT_Sources['ph/s_ratio'][i]
print(ph_ratio, sourcename)
#ph_ratio = 1

# Conversion factor of flux between Fermi and COSI Bands. (0.1-100 GeV to 0.2-5 MeV)
flux_ratio = COSI_LAT_Sources['Int_flux_ratio'][i]


fsrq_names=select_fsrq()
blarray = [hops_bl1]
COSI_BAND_ALL = np.array([0,0,0,0,0,0,0,0,0,0,0,0,0])
for bl in blarray:
    characterized_flares = hop_characterizer(bl, COSI_bkg_rate=COSI_bkg_rate, COSI_Aeff=COSI_Aeff, LAT_Aeff=LAT_Aeff, ph_ratio=ph_ratio, int_flux_ratio=flux_ratio)
    COSI_BAND = np.array(characterized_flares).T
    COSI_BAND_ALL = np.vstack((COSI_BAND_ALL, COSI_BAND))

COSI_BAND_BAT_weekly_df = pd.DataFrame(COSI_BAND_ALL)
COSI_BAND_BAT_weekly_df.columns = ['Name','Duration_(s)', 'Source_Counts(ph)_(0.2-5_MeV)', 'Asymmetry', 'Coverage', 'Background_Counts', 'MDP99_(%)', 'Class', 'Photon_Fluence_(ph/cm2)_(0.2-5_MeV)', 'Average_Photon_Flux_(ph/cm2/s)_(0.2-5_MeV)','Start_Time_(MJD)','Average_Flux_of_Entire_Source_(0.2-5_MeV)', 'Peak_Flare_Flux_(0.2-5_MeV)']
COSI_BAND_BAT_weekly_df = COSI_BAND_BAT_weekly_df[COSI_BAND_BAT_weekly_df[:]['Source_Counts(ph)_(0.2-5_MeV)']!=0].reset_index(drop=True)
COSI_BAND_BAT_weekly_df = COSI_BAND_BAT_weekly_df[COSI_BAND_BAT_weekly_df[:]['Source_Counts(ph)_(0.2-5_MeV)']!='0.0'].reset_index(drop=True)
COSI_BAND_BAT_weekly_df['Source_Counts(ph)_(0.2-5_MeV)'] = COSI_BAND_BAT_weekly_df['Source_Counts(ph)_(0.2-5_MeV)'].astype(float)
COSI_BAND_BAT_weekly_df['Duration_(s)'] = COSI_BAND_BAT_weekly_df['Duration_(s)'].astype(float)
COSI_BAND_BAT_weekly_df['MDP99_(%)'] = ComputeMDP99(COSI_BAND_BAT_weekly_df['Source_Counts(ph)_(0.2-5_MeV)'].astype(float),
                                                COSI_BAND_BAT_weekly_df['Background_Counts'].astype(float))
COSI_BAND_BAT_weekly_df['Coverage'] = COSI_BAND_BAT_weekly_df['Coverage'].astype(float)
COSI_BAND_BAT_weekly_df = COSI_BAND_BAT_weekly_df[COSI_BAND_BAT_weekly_df[:]['Duration_(s)']<=2e8].reset_index(drop=True)


neworder = ['Name','Class','Average_Photon_Flux_(ph/cm2/s)_(0.2-5_MeV)','Photon_Fluence_(ph/cm2)_(0.2-5_MeV)','Source_Counts(ph)_(0.2-5_MeV)','Background_Counts','Duration_(s)','Start_Time_(MJD)','Coverage', 'MDP99_(%)','Asymmetry','Average_Flux_of_Entire_Source_(0.2-5_MeV)', 'Peak_Flare_Flux_(0.2-5_MeV)']
COSI_BAND_BAT_weekly_df
#COSI_BAND_BAT_weekly_df.to_csv(sourcename+'Analysis.csv')
# Remove blank space in sourcename

In [ ]:
initialtime,finaltime = 55400,57200
sourcename = '4FGL J2151.8-3027'
sourcearray = cadence_df[cadence_df['source_name'] == sourcename].reset_index(drop=True)
titlestring = sourcename
sourcearray = sourcearray[sourcearray['photon_flux2']!=-3333].reset_index(drop=True)
average_flux = np.nanmean(sourcearray['photon_flux2'])
sourcearray = sourcearray[sourcearray['photon_flux2']<=100*average_flux].reset_index(drop=True)
time = sourcearray['tmin']/SecsInDay + MJDREFI
photon_flux = sourcearray['photon_flux2']
errors = sourcearray['photon_flux_error2']
# Creates the lightcurve object of the source.
sourcelightcurve = LC.LightCurve(time,photon_flux,errors,name=titlestring)
i = COSI_LAT_Sources[COSI_LAT_Sources['Name']==sourcename].index[0]

selected1 = sourcelightcurve.select_by_time(initialtime,finaltime)
originalLC = sourcelightcurve
sourcelightcurve.get_bblocks()
photon_flux = selected1.flux

# Finding the initial flux threshold for detection.
maxflux = np.max(photon_flux)
minflux = np.min(photon_flux)
delta_flux = maxflux - minflux
delta_flux_percent = delta_flux * 0.3
thresholdflux = minflux + delta_flux_percent


# Initial flare detection.
selected1.get_bblocks(gamma_value=0.05)
selected1.find_hop(method='baseline', lc_edges ='add', baseline = thresholdflux)

quiescent_background,qui_err = quiescent_background_finder(sourcelightcurve=selected1,method='forward')


# Re-detecting flares with quiescent background.
sourcelightcurve = originalLC.select_by_time(initialtime,finaltime)
sourcelightcurve.get_bblocks(gamma_value=0.05)
sourcelightcurve.find_hop(method = 'baseline', lc_edges ='add',baseline = 0.2e-6)

hops_bl1 = sourcelightcurve.hops
plotit = originalLC.select_by_time(initialtime,finaltime)
plotit.plot_lc()
sourcelightcurve.plot_hop()

COSI_LAT_Sources = pd.read_csv(table, sep=",", na_filter=True)
# Effective Area of Fermi LAT for the source.
LAT_Aeff = COSI_LAT_Sources['Aeff_mean_LAT(cm2)'][i]

# Effective Area of COSI for the source.
COSI_Aeff = COSI_LAT_Sources['Aeff_mean_COSI(cm2)'][i]

# Name of Source
# Conversion factor of counts between Fermi and COSI Bands. (0.1-100 GeV to 0.2-5 MeV)
ph_ratio = COSI_LAT_Sources['ph/s_ratio'][i]

#ph_ratio = 1

# Conversion factor of flux between Fermi and COSI Bands. (0.1-100 GeV to 0.2-5 MeV)
flux_ratio = COSI_LAT_Sources['Int_flux_ratio'][i]


fsrq_names=select_fsrq()
blarray = [hops_bl1]
COSI_BAND_ALL = np.array([0,0,0,0,0,0,0,0,0,0,0,0,0])
for bl in blarray:
    characterized_flares = hop_characterizer(bl, COSI_bkg_rate=COSI_bkg_rate, COSI_Aeff=COSI_Aeff, LAT_Aeff=LAT_Aeff, ph_ratio=ph_ratio, int_flux_ratio=flux_ratio)
    COSI_BAND = np.array(characterized_flares).T
    COSI_BAND_ALL = np.vstack((COSI_BAND_ALL, COSI_BAND))

COSI_BAND_BAT_weekly_df = pd.DataFrame(COSI_BAND_ALL)
COSI_BAND_BAT_weekly_df.columns = ['Name','Duration_(s)', 'Source_Counts(ph)_(0.2-5_MeV)', 'Asymmetry', 'Coverage', 'Background_Counts', 'MDP99_(%)', 'Class', 'Photon_Fluence_(ph/cm2)_(0.2-5_MeV)', 'Average_Photon_Flux_(ph/cm2/s)_(0.2-5_MeV)','Start_Time_(MJD)','Average_Flux_of_Entire_Source_(0.2-5_MeV)', 'Peak_Flare_Flux_(0.2-5_MeV)']
COSI_BAND_BAT_weekly_df = COSI_BAND_BAT_weekly_df[COSI_BAND_BAT_weekly_df[:]['Source_Counts(ph)_(0.2-5_MeV)']!=0].reset_index(drop=True)
COSI_BAND_BAT_weekly_df = COSI_BAND_BAT_weekly_df[COSI_BAND_BAT_weekly_df[:]['Source_Counts(ph)_(0.2-5_MeV)']!='0.0'].reset_index(drop=True)
COSI_BAND_BAT_weekly_df['Source_Counts(ph)_(0.2-5_MeV)'] = COSI_BAND_BAT_weekly_df['Source_Counts(ph)_(0.2-5_MeV)'].astype(float)
COSI_BAND_BAT_weekly_df['Duration_(s)'] = COSI_BAND_BAT_weekly_df['Duration_(s)'].astype(float)
COSI_BAND_BAT_weekly_df['MDP99_(%)'] = ComputeMDP99(COSI_BAND_BAT_weekly_df['Source_Counts(ph)_(0.2-5_MeV)'].astype(float),
                                                COSI_BAND_BAT_weekly_df['Background_Counts'].astype(float))
COSI_BAND_BAT_weekly_df['Coverage'] = COSI_BAND_BAT_weekly_df['Coverage'].astype(float)
COSI_BAND_BAT_weekly_df = COSI_BAND_BAT_weekly_df[COSI_BAND_BAT_weekly_df[:]['Duration_(s)']<=2e8].reset_index(drop=True)


neworder = ['Name','Class','Average_Photon_Flux_(ph/cm2/s)_(0.2-5_MeV)','Photon_Fluence_(ph/cm2)_(0.2-5_MeV)','Source_Counts(ph)_(0.2-5_MeV)','Background_Counts','Duration_(s)','Start_Time_(MJD)','Coverage', 'MDP99_(%)','Asymmetry','Average_Flux_of_Entire_Source_(0.2-5_MeV)', 'Peak_Flare_Flux_(0.2-5_MeV)']
COSI_BAND_BAT_weekly_df
#COSI_BAND_BAT_weekly_df.to_csv(sourcename+'Analysis.csv')
# Remove blank space in sourcename

In [ ]:
initialtime,finaltime = 54500,57000
sourcename = '4FGL J2253.9+1609'
sourcearray = cadence_df[cadence_df['source_name'] == sourcename].reset_index(drop=True)
titlestring = sourcename
sourcearray = sourcearray[sourcearray['photon_flux2']!=-3333].reset_index(drop=True)
average_flux = np.nanmean(sourcearray['photon_flux2'])
sourcearray = sourcearray[sourcearray['photon_flux2']<=100*average_flux].reset_index(drop=True)
time = sourcearray['tmin']/SecsInDay + MJDREFI
photon_flux = sourcearray['photon_flux2']
errors = sourcearray['photon_flux_error2']
# Creates the lightcurve object of the source.
sourcelightcurve = LC.LightCurve(time,photon_flux,errors,name=titlestring)
i = COSI_LAT_Sources[COSI_LAT_Sources['Name']==sourcename].index[0]

selected1 = sourcelightcurve.select_by_time(initialtime,finaltime)
originalLC = sourcelightcurve
sourcelightcurve.get_bblocks()
photon_flux = selected1.flux

# Finding the initial flux threshold for detection.
maxflux = np.max(photon_flux)
minflux = np.min(photon_flux)
delta_flux = maxflux - minflux
delta_flux_percent = delta_flux * 0.3
thresholdflux = minflux + delta_flux_percent


# Initial flare detection.
selected1.get_bblocks(gamma_value=0.05)
selected1.find_hop(method='baseline', lc_edges ='add', baseline = thresholdflux)

quiescent_background,qui_err = quiescent_background_finder(sourcelightcurve=selected1,method='forward')


# Re-detecting flares with quiescent background.
sourcelightcurve = originalLC.select_by_time(initialtime,finaltime)
sourcelightcurve.get_bblocks(gamma_value=0.05)
sourcelightcurve.find_hop(method = 'baseline', lc_edges ='add',baseline = 8e-6)

hops_bl1 = sourcelightcurve.hops
plotit = originalLC.select_by_time(initialtime,finaltime)
plotit.plot_lc()
sourcelightcurve.plot_hop()

COSI_LAT_Sources = pd.read_csv(table, sep=",", na_filter=True)
# Effective Area of Fermi LAT for the source.
LAT_Aeff = COSI_LAT_Sources['Aeff_mean_LAT(cm2)'][i]

# Effective Area of COSI for the source.
COSI_Aeff = COSI_LAT_Sources['Aeff_mean_COSI(cm2)'][i]

# Name of Source
# Conversion factor of counts between Fermi and COSI Bands. (0.1-100 GeV to 0.2-5 MeV)
ph_ratio = COSI_LAT_Sources['ph/s_ratio'][i]

#ph_ratio = 1

# Conversion factor of flux between Fermi and COSI Bands. (0.1-100 GeV to 0.2-5 MeV)
flux_ratio = COSI_LAT_Sources['Int_flux_ratio'][i]


fsrq_names=select_fsrq()
blarray = [hops_bl1]
COSI_BAND_ALL = np.array([0,0,0,0,0,0,0,0,0,0,0,0,0])
for bl in blarray:
    characterized_flares = hop_characterizer(bl, COSI_bkg_rate=COSI_bkg_rate, COSI_Aeff=COSI_Aeff, LAT_Aeff=LAT_Aeff, ph_ratio=ph_ratio, int_flux_ratio=flux_ratio)
    COSI_BAND = np.array(characterized_flares).T
    COSI_BAND_ALL = np.vstack((COSI_BAND_ALL, COSI_BAND))

COSI_BAND_BAT_weekly_df = pd.DataFrame(COSI_BAND_ALL)
COSI_BAND_BAT_weekly_df.columns = ['Name','Duration_(s)', 'Source_Counts(ph)_(0.2-5_MeV)', 'Asymmetry', 'Coverage', 'Background_Counts', 'MDP99_(%)', 'Class', 'Photon_Fluence_(ph/cm2)_(0.2-5_MeV)', 'Average_Photon_Flux_(ph/cm2/s)_(0.2-5_MeV)','Start_Time_(MJD)','Average_Flux_of_Entire_Source_(0.2-5_MeV)', 'Peak_Flare_Flux_(0.2-5_MeV)']
COSI_BAND_BAT_weekly_df = COSI_BAND_BAT_weekly_df[COSI_BAND_BAT_weekly_df[:]['Source_Counts(ph)_(0.2-5_MeV)']!=0].reset_index(drop=True)
COSI_BAND_BAT_weekly_df = COSI_BAND_BAT_weekly_df[COSI_BAND_BAT_weekly_df[:]['Source_Counts(ph)_(0.2-5_MeV)']!='0.0'].reset_index(drop=True)
COSI_BAND_BAT_weekly_df['Source_Counts(ph)_(0.2-5_MeV)'] = COSI_BAND_BAT_weekly_df['Source_Counts(ph)_(0.2-5_MeV)'].astype(float)
COSI_BAND_BAT_weekly_df['Duration_(s)'] = COSI_BAND_BAT_weekly_df['Duration_(s)'].astype(float)
COSI_BAND_BAT_weekly_df['MDP99_(%)'] = ComputeMDP99(COSI_BAND_BAT_weekly_df['Source_Counts(ph)_(0.2-5_MeV)'].astype(float),
                                                COSI_BAND_BAT_weekly_df['Background_Counts'].astype(float))
COSI_BAND_BAT_weekly_df['Coverage'] = COSI_BAND_BAT_weekly_df['Coverage'].astype(float)
COSI_BAND_BAT_weekly_df = COSI_BAND_BAT_weekly_df[COSI_BAND_BAT_weekly_df[:]['Duration_(s)']<=2e8].reset_index(drop=True)


neworder = ['Name','Class','Average_Photon_Flux_(ph/cm2/s)_(0.2-5_MeV)','Photon_Fluence_(ph/cm2)_(0.2-5_MeV)','Source_Counts(ph)_(0.2-5_MeV)','Background_Counts','Duration_(s)','Start_Time_(MJD)','Coverage', 'MDP99_(%)','Asymmetry','Average_Flux_of_Entire_Source_(0.2-5_MeV)', 'Peak_Flare_Flux_(0.2-5_MeV)']
COSI_BAND_BAT_weekly_df
#COSI_BAND_BAT_weekly_df.to_csv(sourcename+'Analysis.csv')
# Remove blank space in sourcename

In [ ]:
initialtime,finaltime = 56500,57000
sourcename = '4FGL J2253.9+1609'
sourcearray = cadence_df[cadence_df['source_name'] == sourcename].reset_index(drop=True)
titlestring = sourcename
sourcearray = sourcearray[sourcearray['photon_flux2']!=-3333].reset_index(drop=True)
average_flux = np.nanmean(sourcearray['photon_flux2'])
sourcearray = sourcearray[sourcearray['photon_flux2']<=100*average_flux].reset_index(drop=True)
time = sourcearray['tmin']/SecsInDay + MJDREFI
photon_flux = sourcearray['photon_flux2']
errors = sourcearray['photon_flux_error2']
# Creates the lightcurve object of the source.
sourcelightcurve = LC.LightCurve(time,photon_flux,errors,name=titlestring)
i = COSI_LAT_Sources[COSI_LAT_Sources['Name']==sourcename].index[0]

selected1 = sourcelightcurve.select_by_time(initialtime,finaltime)
originalLC = sourcelightcurve
sourcelightcurve.get_bblocks()
photon_flux = selected1.flux

# Finding the initial flux threshold for detection.
maxflux = np.max(photon_flux)
minflux = np.min(photon_flux)
delta_flux = maxflux - minflux
delta_flux_percent = delta_flux * 0.3
thresholdflux = minflux + delta_flux_percent


# Initial flare detection.
selected1.get_bblocks(gamma_value=0.05)
selected1.find_hop(method='baseline', lc_edges ='add', baseline = thresholdflux)

quiescent_background,qui_err = quiescent_background_finder(sourcelightcurve=selected1,method='forward')


# Re-detecting flares with quiescent background.
sourcelightcurve = originalLC.select_by_time(initialtime,finaltime)
sourcelightcurve.get_bblocks(gamma_value=0.05)
sourcelightcurve.find_hop(method = 'baseline', lc_edges ='add',baseline = 3e-6)

hops_bl1 = sourcelightcurve.hops
plotit = originalLC.select_by_time(initialtime,finaltime)
plotit.plot_lc()
sourcelightcurve.plot_hop()

COSI_LAT_Sources = pd.read_csv(table, sep=",", na_filter=True)
# Effective Area of Fermi LAT for the source.
LAT_Aeff = COSI_LAT_Sources['Aeff_mean_LAT(cm2)'][i]

# Effective Area of COSI for the source.
COSI_Aeff = COSI_LAT_Sources['Aeff_mean_COSI(cm2)'][i]

# Name of Source
# Conversion factor of counts between Fermi and COSI Bands. (0.1-100 GeV to 0.2-5 MeV)
ph_ratio = COSI_LAT_Sources['ph/s_ratio'][i]

#ph_ratio = 1

# Conversion factor of flux between Fermi and COSI Bands. (0.1-100 GeV to 0.2-5 MeV)
flux_ratio = COSI_LAT_Sources['Int_flux_ratio'][i]


fsrq_names=select_fsrq()
blarray = [hops_bl1]
COSI_BAND_ALL = np.array([0,0,0,0,0,0,0,0,0,0,0,0,0])
for bl in blarray:
    characterized_flares = hop_characterizer(bl, COSI_bkg_rate=COSI_bkg_rate, COSI_Aeff=COSI_Aeff, LAT_Aeff=LAT_Aeff, ph_ratio=ph_ratio, int_flux_ratio=flux_ratio)
    COSI_BAND = np.array(characterized_flares).T
    COSI_BAND_ALL = np.vstack((COSI_BAND_ALL, COSI_BAND))

COSI_BAND_BAT_weekly_df = pd.DataFrame(COSI_BAND_ALL)
COSI_BAND_BAT_weekly_df.columns = ['Name','Duration_(s)', 'Source_Counts(ph)_(0.2-5_MeV)', 'Asymmetry', 'Coverage', 'Background_Counts', 'MDP99_(%)', 'Class', 'Photon_Fluence_(ph/cm2)_(0.2-5_MeV)', 'Average_Photon_Flux_(ph/cm2/s)_(0.2-5_MeV)','Start_Time_(MJD)','Average_Flux_of_Entire_Source_(0.2-5_MeV)', 'Peak_Flare_Flux_(0.2-5_MeV)']
COSI_BAND_BAT_weekly_df = COSI_BAND_BAT_weekly_df[COSI_BAND_BAT_weekly_df[:]['Source_Counts(ph)_(0.2-5_MeV)']!=0].reset_index(drop=True)
COSI_BAND_BAT_weekly_df = COSI_BAND_BAT_weekly_df[COSI_BAND_BAT_weekly_df[:]['Source_Counts(ph)_(0.2-5_MeV)']!='0.0'].reset_index(drop=True)
COSI_BAND_BAT_weekly_df['Source_Counts(ph)_(0.2-5_MeV)'] = COSI_BAND_BAT_weekly_df['Source_Counts(ph)_(0.2-5_MeV)'].astype(float)
COSI_BAND_BAT_weekly_df['Duration_(s)'] = COSI_BAND_BAT_weekly_df['Duration_(s)'].astype(float)
COSI_BAND_BAT_weekly_df['MDP99_(%)'] = ComputeMDP99(COSI_BAND_BAT_weekly_df['Source_Counts(ph)_(0.2-5_MeV)'].astype(float),
                                                COSI_BAND_BAT_weekly_df['Background_Counts'].astype(float))
COSI_BAND_BAT_weekly_df['Coverage'] = COSI_BAND_BAT_weekly_df['Coverage'].astype(float)
COSI_BAND_BAT_weekly_df = COSI_BAND_BAT_weekly_df[COSI_BAND_BAT_weekly_df[:]['Duration_(s)']<=2e8].reset_index(drop=True)


neworder = ['Name','Class','Average_Photon_Flux_(ph/cm2/s)_(0.2-5_MeV)','Photon_Fluence_(ph/cm2)_(0.2-5_MeV)','Source_Counts(ph)_(0.2-5_MeV)','Background_Counts','Duration_(s)','Start_Time_(MJD)','Coverage', 'MDP99_(%)','Asymmetry','Average_Flux_of_Entire_Source_(0.2-5_MeV)', 'Peak_Flare_Flux_(0.2-5_MeV)']
COSI_BAND_BAT_weekly_df
#COSI_BAND_BAT_weekly_df.to_csv(sourcename+'Analysis.csv')
# Remove blank space in sourcename

### Work In Progress (Turning this into a repeatable function.)

In [ ]:
initialtime,finaltime = 56500,57000
sourcename = '4FGL J2253.9+1609'
def flare_isolator(sourcename, initialtime,finaltime, threshold,COSI_LAT_Sources=COSI_LAT_Sources):
    sourcearray = cadence_df[cadence_df['source_name'] == sourcename].reset_index(drop=True)
    titlestring = sourcename
    sourcearray = sourcearray[sourcearray['photon_flux2']!=-3333].reset_index(drop=True)
    average_flux = np.nanmean(sourcearray['photon_flux2'])
    sourcearray = sourcearray[sourcearray['photon_flux2']<=100*average_flux].reset_index(drop=True)
    time = sourcearray['tmin']/SecsInDay + MJDREFI
    photon_flux = sourcearray['photon_flux2']
    errors = sourcearray['photon_flux_error2']
    # Creates the lightcurve object of the source.
    sourcelightcurve = LC.LightCurve(time,photon_flux,errors,name=titlestring)
    i = COSI_LAT_Sources[COSI_LAT_Sources['Name']==sourcename].index[0]

    selected1 = sourcelightcurve.select_by_time(initialtime,finaltime)
    originalLC = sourcelightcurve
    sourcelightcurve.get_bblocks()
    photon_flux = selected1.flux

    # Finding the initial flux threshold for detection.
    maxflux = np.max(photon_flux)
    minflux = np.min(photon_flux)
    delta_flux = maxflux - minflux
    delta_flux_percent = delta_flux * 0.3
    thresholdflux = minflux + delta_flux_percent


    # Initial flare detection.
    selected1.get_bblocks(gamma_value=0.05)
    selected1.find_hop(method='baseline', lc_edges ='add', baseline = thresholdflux)

    quiescent_background,qui_err = quiescent_background_finder(sourcelightcurve=selected1,method='forward')


    # Re-detecting flares with quiescent background.
    sourcelightcurve = originalLC.select_by_time(initialtime,finaltime)
    sourcelightcurve.get_bblocks(gamma_value=0.05)
    sourcelightcurve.find_hop(method = 'baseline', lc_edges ='add',baseline = threshold)

    hops_bl1 = sourcelightcurve.hops
    plotit = originalLC.select_by_time(initialtime,finaltime)
    plotit.plot_lc()
    sourcelightcurve.plot_hop()

    COSI_LAT_Sources = pd.read_csv(table, sep=",", na_filter=True)
    # Effective Area of Fermi LAT for the source.
    LAT_Aeff = COSI_LAT_Sources['Aeff_mean_LAT(cm2)'][i]

    # Effective Area of COSI for the source.
    COSI_Aeff = COSI_LAT_Sources['Aeff_mean_COSI(cm2)'][i]

    # Name of Source
    # Conversion factor of counts between Fermi and COSI Bands. (0.1-100 GeV to 0.2-5 MeV)
    ph_ratio = COSI_LAT_Sources['ph/s_ratio'][i]

    #ph_ratio = 1

    # Conversion factor of flux between Fermi and COSI Bands. (0.1-100 GeV to 0.2-5 MeV)
    flux_ratio = COSI_LAT_Sources['Int_flux_ratio'][i]


    fsrq_names=select_fsrq()
    blarray = [hops_bl1]
    COSI_BAND_ALL = np.array([0,0,0,0,0,0,0,0,0,0,0,0,0])
    for bl in blarray:
        characterized_flares = hop_characterizer(bl, COSI_bkg_rate=COSI_bkg_rate, COSI_Aeff=COSI_Aeff, LAT_Aeff=LAT_Aeff, ph_ratio=ph_ratio, int_flux_ratio=flux_ratio)
        COSI_BAND = np.array(characterized_flares).T
        COSI_BAND_ALL = np.vstack((COSI_BAND_ALL, COSI_BAND))

    COSI_BAND_BAT_weekly_df = pd.DataFrame(COSI_BAND_ALL)
    COSI_BAND_BAT_weekly_df.columns = ['Name','Duration_(s)', 'Source_Counts(ph)_(0.2-5_MeV)', 'Asymmetry', 'Coverage', 'Background_Counts', 'MDP99_(%)', 'Class', 'Photon_Fluence_(ph/cm2)_(0.2-5_MeV)', 'Average_Photon_Flux_(ph/cm2/s)_(0.2-5_MeV)','Start_Time_(MJD)','Average_Flux_of_Entire_Source_(0.2-5_MeV)', 'Peak_Flare_Flux_(0.2-5_MeV)']
    COSI_BAND_BAT_weekly_df = COSI_BAND_BAT_weekly_df[COSI_BAND_BAT_weekly_df[:]['Source_Counts(ph)_(0.2-5_MeV)']!=0].reset_index(drop=True)
    COSI_BAND_BAT_weekly_df = COSI_BAND_BAT_weekly_df[COSI_BAND_BAT_weekly_df[:]['Source_Counts(ph)_(0.2-5_MeV)']!='0.0'].reset_index(drop=True)
    COSI_BAND_BAT_weekly_df['Source_Counts(ph)_(0.2-5_MeV)'] = COSI_BAND_BAT_weekly_df['Source_Counts(ph)_(0.2-5_MeV)'].astype(float)
    COSI_BAND_BAT_weekly_df['Duration_(s)'] = COSI_BAND_BAT_weekly_df['Duration_(s)'].astype(float)
    COSI_BAND_BAT_weekly_df['MDP99_(%)'] = ComputeMDP99(COSI_BAND_BAT_weekly_df['Source_Counts(ph)_(0.2-5_MeV)'].astype(float),
                                                    COSI_BAND_BAT_weekly_df['Background_Counts'].astype(float))
    COSI_BAND_BAT_weekly_df['Coverage'] = COSI_BAND_BAT_weekly_df['Coverage'].astype(float)
    COSI_BAND_BAT_weekly_df = COSI_BAND_BAT_weekly_df[COSI_BAND_BAT_weekly_df[:]['Duration_(s)']<=2e8].reset_index(drop=True)


    neworder = ['Name','Class','Average_Photon_Flux_(ph/cm2/s)_(0.2-5_MeV)','Photon_Fluence_(ph/cm2)_(0.2-5_MeV)','Source_Counts(ph)_(0.2-5_MeV)','Background_Counts','Duration_(s)','Start_Time_(MJD)','Coverage', 'MDP99_(%)','Asymmetry','Average_Flux_of_Entire_Source_(0.2-5_MeV)', 'Peak_Flare_Flux_(0.2-5_MeV)']
    return COSI_BAND_BAT_weekly_df
#COSI_BAND_BAT_weekly_df.to_csv(sourcename+'Analysis.csv')
# Remove blank space in sourcename

In [ ]:
flare_isolator(sourcename = '4FGL J2253.9+1609',initialtime=56500,finaltime=57000,threshold = 3e-6)

### Plotting

In [ ]:
f = ['RemadeNov2025/November2025_COSI_Eta0.3_bkg1.00softer0.5.csv']
backgrounds=[1.00]
percents=[0.3]

# Load the per-flare softer CSV for overlay
softer_flares_df = pd.read_csv('FlareCounting/source_flares_softer_bkg_1_rate.csv', sep=",", na_filter=True)
softer_flares_df = softer_flares_df[pd.to_numeric(softer_flares_df['Source_Counts(ph)_(0.2-5_MeV)'], errors='coerce') > 0].reset_index(drop=True)

for i in range(len(f)):
    COSI_BAND_BAT_weekly_df = pd.read_csv(f[i], sep=",", na_filter=True)
    COSI_BAND_BAT_weekly_df = COSI_BAND_BAT_weekly_df[COSI_BAND_BAT_weekly_df['Duration_(s)']!='0.0']
    COSI_BAND_BAT_weekly_df = COSI_BAND_BAT_weekly_df[~COSI_BAND_BAT_weekly_df['Name'].isin(ignorelist)].reset_index(drop=True)    # COSI_BAND_BAT_weekly_df = pd.read_csv('RemadeNov2025/November2025_COSI_0.5_Eta0.10.csv', sep=",", na_filter=True)
    COSI_BAND_BAT_weekly_df = COSI_BAND_BAT_weekly_df[COSI_BAND_BAT_weekly_df[:]['Source_Counts(ph)_(0.2-5_MeV)']!=-3333].reset_index(drop=True)
    # Filter out flares with duration >= 8 weeks.
    COSI_BAND_BAT_weekly_df = COSI_BAND_BAT_weekly_df[COSI_BAND_BAT_weekly_df[:]['Duration_(s)'] < 4838400].reset_index(drop=True)
    #COSI_BAND_BAT_weekly_df = COSI_BAND_BAT_weekly_df[COSI_BAND_BAT_weekly_df[:]['MDP99_(%)']<=100].reset_index(drop=True)
    labelrate = backgrounds[i]
    labelpercent = percents[i]

    # Round to a reasonable precision to identify duplicate (MDP, Fluence) pairs
    _key_cols = ['MDP99_(%)', 'Photon_Fluence_(ph/cm2)_(0.2-5_MeV)']
    _f_keys = COSI_BAND_BAT_weekly_df[_key_cols].apply(pd.to_numeric, errors='coerce').round(6)
    _s_keys = softer_flares_df[_key_cols].apply(pd.to_numeric, errors='coerce').round(6)

    # Rows in softer_flares_df that are NOT duplicates of any row in f
    _overlap_mask = _s_keys.apply(
        lambda row: ((_f_keys['MDP99_(%)'] == row['MDP99_(%)']) &
                     (_f_keys['Photon_Fluence_(ph/cm2)_(0.2-5_MeV)'] == row['Photon_Fluence_(ph/cm2)_(0.2-5_MeV)'])).any(),
        axis=1
    )
    softer_unique_df = softer_flares_df[~_overlap_mask].reset_index(drop=True)
    softer_unique_df = softer_unique_df[
        pd.to_numeric(softer_unique_df['Duration_(s)'], errors='coerce') <= 4838400
    ].reset_index(drop=True)

    plt.figure(figsize = (7,5))
    fig,ax = plt.subplots(figsize = (7,5))
    ax.yaxis.set_major_formatter(FormatStrFormatter('%.1f'))
    cm = plt.get_cmap('bwr_r')
    cm = plt.get_cmap('cubehelix_r')
    z = COSI_BAND_BAT_weekly_df['Duration_(s)']/604800
    #print(np.max(z))
    #sizes = COSI_BAND_BAT_weekly_df['Duration (s)']/604800
    sc = plt.scatter(COSI_BAND_BAT_weekly_df['Photon_Fluence_(ph/cm2)_(0.2-5_MeV)'],
                        COSI_BAND_BAT_weekly_df['MDP99_(%)'],
                        c=z, sizes=z*30,
                        cmap=cm, edgecolors='k', vmax=8, vmin=0, zorder=2, label='BB-Selected')
    cbar = plt.colorbar(sc)
    cbar.set_label('Duration (weeks)', size=15)
    cbar.ax.tick_params(labelsize=15)

    # Overlay softer per-flare CSV (non-overlapping rows only) with diamond markers
    z2 = pd.to_numeric(softer_unique_df['Duration_(s)'], errors='coerce') / 604800
    sc2 = ax.scatter(pd.to_numeric(softer_unique_df['Photon_Fluence_(ph/cm2)_(0.2-5_MeV)'], errors='coerce'),
                     pd.to_numeric(softer_unique_df['MDP99_(%)'], errors='coerce'),
                     c=z2, s=z2*30,
                     cmap=cm, edgecolors='dodgerblue', vmax=8, vmin=0,
                     marker='D', linewidths=1.2, zorder=1, label='Manually-Selected')

    plt.ylim(0,100)
    #plt.xlim(np.min(COSI_BAND_BAT_weekly_df[:]['Photon Fluence (ph/cm2) (0.2-5 MeV)'])*0.1,np.max(COSI_BAND_BAT_weekly_df[:]['Photon Fluence (ph/cm2) (0.2-5 MeV)']))
    plt.xlim(1000,40000)
    #plt.xlim(100,200000)
    plt.hlines(20,0,50000,colors='0.5', linestyles='dashed',label=r'20% MDP$_{99\%}$')
    plt.hlines(50,0,50000,colors='r', lw=1, label=r'50% MDP$_{99\%}$')
    plt.xscale('log')
    #plt.yscale('log')
    plt.xlabel(r'Photon Fluence (0.2-5 MeV)' + ' (ph$\u22c5cm^{-2}$)', size=15)
    plt.ylabel(r'MDP$_{99\%}$ (%)', size=15)
    plt.title(r'$\eta$=0.3, $\mathrm{R_{bkg}}=1.0$ ct/s, Softer spectrum ($\gamma_{1}+0.5$)', size=15)
    #plt.title(r'$\eta$=0.3, $\mathrm{R_{bkg}}=1.0$ ct/s, Nominal spectrum ($\gamma_{1}$)', size=15)
    #plt.title(r'$\eta$ = %.1f'%labelpercent + r', R$_{bkg}$ = %.2f ct/s'%(labelrate), size=17)
    plt.legend(loc=0, fontsize=13)
    #plt.locator_params(axis='x', nbins=6)
    # make ticks and labels bigger
    plt.xticks(fontsize=15)
    plt.yticks(fontsize=15)
    print('Eta = ' + str(percents[i])+ ', Rate = ' + str(backgrounds[i])+ ', ' + str(len(COSI_BAND_BAT_weekly_df[COSI_BAND_BAT_weekly_df['MDP99_(%)']<=50]['Name']))+' Flares from '+str(len(COSI_BAND_BAT_weekly_df[COSI_BAND_BAT_weekly_df['MDP99_(%)']<=50]['Name'].drop_duplicates())) + ' sources below 50.')
    print('Eta = ' + str(percents[i])+ ', Rate = ' + str(backgrounds[i])+ ', ' + str(len(COSI_BAND_BAT_weekly_df[COSI_BAND_BAT_weekly_df['MDP99_(%)']<=20]['Name']))+' Flares from '+str(len(COSI_BAND_BAT_weekly_df[COSI_BAND_BAT_weekly_df['MDP99_(%)']<=20]['Name'].drop_duplicates())) + ' sources below 20.')
    #Save Figure in FlareCounting.
    #plt.savefig('FlareCounting/COSI_Duration_vs_PhotonFluence_Eta%.1f'%percents[i]+'_bkg%.2f'%(backgrounds[i])+'.png', dpi=200, bbox_inches='tight')
    #plt.savefig('FlareCounting/COSI_Duration_vs_PhotonFluence_Eta%.1f'%percents[i]+'_bkg%.2f'%(backgrounds[i])+'_softer0.5.png', dpi=200, bbox_inches='tight')
    #plt.savefig('RemadeNov2025/COSI_Duration_vs_PhotonFluence_Eta%.1f'%labelpercent+'_bkg%.2f'%(labelrate)+'.png', dpi=200, bbox_inches='tight')
    #plt.savefig('RemadeNov2025/COSI_Duration_vs_PhotonFluence_Eta%.1f'%labelpercent+'_bkg%.2f'%(labelrate)+'_softer0.5.png', dpi=200, bbox_inches='tight')


In [ ]:
# f-only comparison plot (duration <= 8 weeks)
f_only_files = ['RemadeNov2025/November2025_COSI_Eta0.3_bkg1.00softer0.5.csv']
f_only_backgrounds = [1.00]
f_only_percents = [0.3]

for i in range(len(f_only_files)):
    df_f_only = pd.read_csv(f_only_files[i], sep=",", na_filter=True)
    df_f_only = df_f_only[df_f_only['Duration_(s)'] != '0.0']
    df_f_only = df_f_only[~df_f_only['Name'].isin(ignorelist)].reset_index(drop=True)
    df_f_only = df_f_only[df_f_only['Source_Counts(ph)_(0.2-5_MeV)'] != -3333].reset_index(drop=True)

    # Ensure numeric fields are usable for plotting.
    df_f_only['Duration_(s)'] = pd.to_numeric(df_f_only['Duration_(s)'], errors='coerce')
    df_f_only['Photon_Fluence_(ph/cm2)_(0.2-5_MeV)'] = pd.to_numeric(
        df_f_only['Photon_Fluence_(ph/cm2)_(0.2-5_MeV)'], errors='coerce'
    )
    df_f_only['MDP99_(%)'] = pd.to_numeric(df_f_only['MDP99_(%)'], errors='coerce')
    df_f_only = df_f_only.dropna(subset=['Duration_(s)', 'Photon_Fluence_(ph/cm2)_(0.2-5_MeV)', 'MDP99_(%)']).reset_index(drop=True)

    # Keep flares with duration <= 8 weeks for this f-only view.
    df_f_only = df_f_only[df_f_only['Duration_(s)'] <= 4838400].reset_index(drop=True)

    labelrate = f_only_backgrounds[i]
    labelpercent = f_only_percents[i]

    fig, ax = plt.subplots(figsize=(7, 5))
    ax.yaxis.set_major_formatter(FormatStrFormatter('%.1f'))
    cm = plt.get_cmap('cubehelix_r')

    z = df_f_only['Duration_(s)'] / 604800
    sc = ax.scatter(
        df_f_only['Photon_Fluence_(ph/cm2)_(0.2-5_MeV)'],
        df_f_only['MDP99_(%)'],
        c=z,
        s=z * 10,
        cmap=cm,
        edgecolors='k',
        vmax=8,
        vmin=0,
        zorder=2,
        label='Original Flares'
    )

    cbar = plt.colorbar(sc)
    cbar.set_label('Duration (weeks)', size=15)
    cbar.ax.tick_params(labelsize=15)

    plt.ylim(0, 100)
    plt.xlim(1000, 40000)
    plt.hlines(20, 0, 50000, colors='0.5', linestyles='dashed', label=r'20% MDP$_{99\%}$')
    plt.hlines(50, 0, 50000, colors='r', lw=1, label=r'50% MDP$_{99\%}$')
    plt.xscale('log')
    plt.xlabel(r'Photon Fluence (0.2-5 MeV)' + ' (ph$\u22c5cm^{-2}$)', size=15)
    plt.ylabel(r'MDP$_{99\%}$ (%)', size=15)
    plt.title(r'$\eta$=0.3, $\mathrm{R_{bkg}}=1.0$ ct/s, Softer spectrum ($\gamma_{1}+0.5$)', size=15)
    plt.legend(loc=0, fontsize=12)
    plt.xticks(fontsize=15)
    plt.yticks(fontsize=15)

    print(
        '<=8wk plot: Eta = ' + str(labelpercent) + ', Rate = ' + str(labelrate) +
        ', ' + str(len(df_f_only[df_f_only['MDP99_(%)'] <= 50]['Name'])) + ' Flares from ' +
        str(len(df_f_only[df_f_only['MDP99_(%)'] <= 50]['Name'].drop_duplicates())) + ' sources below 50.'
    )
    print(
        '<=8wk plot: Eta = ' + str(labelpercent) + ', Rate = ' + str(labelrate) +
        ', ' + str(len(df_f_only[df_f_only['MDP99_(%)'] <= 20]['Name'])) + ' Flares from ' +
        str(len(df_f_only[df_f_only['MDP99_(%)'] <= 20]['Name'].drop_duplicates())) + ' sources below 20.'
    )